# Deney 01 — Video Analiz & Hot-Moment Tespiti

**Amaç:** Uzun bir kaynak videodan (a) transcript çıkarımı, (b) sahne tespiti, (c) ses enerjisi analizi ve (d) LLM tabanlı "vurucu an" (hot-moment) skorlaması için aday model/sağlayıcıları **kalite + maliyet + hız** üzerinden karşılaştırmak.

**Tek kaynak politikası:** Bu notebook algoritmaları kendi içinde tekrar tanımlamaz — transkripsiyon, sahne tespiti, ses tepe analizi, hot-moment skorlama, önizleme klibi üretimi ve `hot_moments.json` şeması hepsi `local_pipeline/video_analysis.py` içindeki **kanonik** fonksiyonlardan gelir. `00_youtube_automation_master.ipynb` ve `verify_local.py` de aynı fonksiyonları çağırır. Bu notebook'un kendine özgü katkısı **çok-aday karşılaştırma (benchmark) döngüsüdür**; master notebook ise tek varsayılan adayla uçtan uca çalışır.

**Nasıl çalışır:**
- Colab'da Runtime → Change runtime type → **GPU (T4)** seçip baştan sona çalıştırabilirsin.
- Bazı adaylar API anahtarı gerektirir (OpenAI / Anthropic / Google). Anahtarı olmayanlar otomatik atlanır, notebook hata vermez.
- API anahtarlarını Colab'da soldaki 🔑 **Secrets** panelinden ekle: `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, `GOOGLE_API_KEY`. Hiçbiri yoksa sadece ücretsiz/lokal adaylar (faster-whisper, PySceneDetect, librosa) çalışır.


## Google model router (bu notebook'taki tum Gemini cagrilari icin tek merkez)

Bu notebook artik `genai.Client(...)` kurmuyor ve sabit `model='gemini-...'` yazmiyor.
Her Google cagrisi `local_pipeline/google_ai.py` uzerinden gider:

- **Colab Secret adlari:** `GeminiKey` (eski ad `GOOGLE_API_KEY` hala calisir),
  `YoutubeKEY` (eski ad `YOUTUBE_DATA_API_KEY`). Deger hicbir zaman yazdirilmaz;
  yalnizca `configured (GeminiKey)` / `missing` durumu gosterilir.
- **Tek anahtar, cok model:** ayni `GeminiKey` projenin yetkili oldugu tum modellere erisir.
- **Kota PROJE bazlidir, anahtar bazli DEGILDIR** (ai.google.dev/gemini-api/docs/rate-limits);
  ikinci bir anahtar limit artirmaz. Hesap/proje dondurerek kota asmak yasaktir
  (`ALLOW_ACCOUNT_QUOTA_ROTATION = False`).
- **Rota:** ilk basarili cagri rotayi bitirir; hicbir zaman tum katalog denenmez.
- **Ucretsiz katman veri politikasi:** gonderilen icerik Google tarafindan urun
  gelistirmede kullanilabilir ve insan gozden gecirenler okuyabilir. Ham telifli
  video/ses varsayilan olarak YUKLENMEZ; yalnizca yerel olarak turetilmis
  transcript/skor ozetleri gonderilir.
- **Kota defteri** (`results/google_quota_usage.json`) YEREL TAHMINDIR; sunucunun
  yetkili kotasi degildir. Gunluk sayaclar Pasifik gece yarisinda sifirlanir.

Bu notebook'ta kullanilan gorev: **hot-moment incelemesi** -> Rota A (`gemini-3.6-flash` -> `gemini-3.5-flash` -> `gemini-3.5-flash-lite` -> `gemma-4-31b-it`).


## 0. Kurulum

In [ ]:
import os, subprocess, sys
if os.path.isdir('/content'):
    packages = ['yt-dlp==2026.7.4', 'faster-whisper==1.2.1', 'scenedetect-headless==0.7.1', 'librosa==0.11.0', 'openai==2.50.0', 'anthropic==0.120.2', 'google-genai==2.15.0', 'pandas==3.0.5', 'tqdm==4.70.0', 'moviepy==2.2.1', 'imageio-ffmpeg==0.6.0']
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])
    print('Colab bagimliliklari kuruldu.')
else:
    print('Yerel calisma: proje .venv paketleri bir sonraki hucrede otomatik etkinlestirilecek.')

In [ ]:
import os, time, json, subprocess, shutil, sys, site
from pathlib import Path
if 'google.colab' not in sys.modules:
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        venv_site_packages = candidate / '.venv' / 'Lib' / 'site-packages'
        if venv_site_packages.is_dir():
            site.addsitedir(str(venv_site_packages))
            print('Yerel .venv paketleri etkinlestirildi:', venv_site_packages)
            break
import pandas as pd
import imageio_ffmpeg

IS_COLAB = 'google.colab' in sys.modules or os.path.isdir('/content')
PROJECT_ROOT = Path.cwd()
RUNTIME_DIR = Path('/content') if IS_COLAB else PROJECT_ROOT / '.local_runtime'
RESULTS_DIR = PROJECT_ROOT / 'results'
RUNTIME_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

FFMPEG_BIN = shutil.which('ffmpeg') or imageio_ffmpeg.get_ffmpeg_exe()
FFPROBE_BIN = shutil.which('ffprobe')
print('Calisma ortami:', 'Colab' if IS_COLAB else 'Yerel Jupyter')
print('FFmpeg:', FFMPEG_BIN)
print('FFprobe:', FFPROBE_BIN or 'bulunamadi; on kontrol FFmpeg donusumunde yapilacak')

def get_secret(name):
    try:
        from google.colab import userdata
        val = userdata.get(name)
        if val:
            return val
    except Exception:
        pass
    return os.environ.get(name)

OPENAI_API_KEY = get_secret('OPENAI_API_KEY')
ANTHROPIC_API_KEY = get_secret('ANTHROPIC_API_KEY')
GOOGLE_API_KEY = get_secret('GeminiKey') or get_secret('GOOGLE_API_KEY')  # Colab Secret adi: GeminiKey (eski ad: GOOGLE_API_KEY)

print('OpenAI key:', 'var' if OPENAI_API_KEY else 'yok (bu adaylar atlanacak)')
print('Anthropic key:', 'var' if ANTHROPIC_API_KEY else 'yok (bu adaylar atlanacak)')
print('Google key:', 'var' if GOOGLE_API_KEY else 'yok (bu adaylar atlanacak)')

# Paylasilan durum sozlugu (Faz 6) - tum notebooklarda (01-06) ayni degerler kullanilir.
STATUS_VALUES = {
    'experimental', 'development_placeholder', 'review_required',
    'approved', 'insufficient_data', 'failed', 'production_ready',
}

results = []  # her satır: task, candidate, quality_score, cost_usd, latency_sec, notes, status

def log_result(task, candidate, quality_score, cost_usd, latency_sec, notes='', status='experimental'):
    assert status in STATUS_VALUES, f'Bilinmeyen status: {status}'
    results.append({
        'task': task,
        'candidate': candidate,
        'quality_score_1_5': quality_score,
        'cost_usd_estimate': round(cost_usd, 5),
        'latency_sec': round(latency_sec, 2),
        'notes': notes,
        'status': status,
    })
    print(f"[{task}] {candidate} -> latency={latency_sec:.2f}s cost=${cost_usd:.5f} quality={quality_score} status={status}")

## `local_pipeline.video_analysis` modülünü yükle (kanonik tek kaynak)

In [ ]:
# @title Yerel pipeline yukleyici (otomatik - acmaniz gerekmez)
import sys, os, tempfile, subprocess, base64, hashlib, zlib
from pathlib import Path

PROJECT_REPO_URL = ''  # opsiyonel: gercek bir git repo URL'si verirsen modul(ler) oradan klonlanir

_LOCAL_PIPELINE_MODULE_NAMES = ['video_analysis', 'render_utils', 'google_ai']

# Her modul icin: (base64 kaynak, SHA-256 hash). Hash, gomulu payload'in
# bozulmadigini/kesilmedigini dogrulamak icin kullanilir -- eslesmezse acik
# bir hata verilir, asla sessizce bozuk kod calistirilmaz.
_LOCAL_PIPELINE_PAYLOADS = {
    'video_analysis': {
        'sha256': '57349f8d512413efa78af89f6bfb81aa66a92b04d2b91f42800b68e7308efc93',
        'encoding': 'zlib+b85',
        'data': (
            'c-pmnYm?iylIVB+3LK^C4jD_b#)-3Ml_<A-?8%916FU_<$vt=O6hn#7h!I6<`IxahJ5~G3_RFpNzivMO5CBQdB%9-^WM)X9(P%XK'
            'g+_Bc9{(ttDqdw$+@y&trg0u;$11JFQ?V`T>Au*@ycUV9<+@IbTx7*{x)#+oE@dKC$I<*eqK=X77(qAgHA@f2d?n&M5%YO;ESkDm'
            '$tZ5>Vjm-{XdhR#EV<ciG<s9ia#a*NF+UeODG!xM>q->2x!5;Zole&PDNW*9ij~aQ+kIT_()?Qd^jZY#Vt<HB{Hu~TvW&CQFWa;_'
            '$WrW!L}sE&Kgwz%s<q6enoLo`1H`R3ATn_#zWeTbXi=21te`^z!yM^nI2o;)TGU%9;&t6X%Q(9(%Cz3@5ycjpN=&ED7yuW>Xp*#;'
            '9%Q^59nv+T%;O@wtXKm$o?ew+BNi6b5gDA*rYNCFjL@_6MvAR0rKpP0s;IXjS5&J6fXWD}`~Y~0D&9-6hUtqm2Rg?v(~a22X<oyB'
            '2sUXBS-J*#PDW{-$b*Fc<#l$PCS|&*vBkPLq_Tpgc@NY?QZ~RRIZ{|a-y%+6c}j6;R#{qYWd+kZ)?4WKC6cAcv!f^_Hj33miBm?K'
            '6rdmlZh?l-i{Bked6UXpvCh&1avx5llyy_)z-Vbc-OGJZ9tEs_EULO}U?oa98@+@{<AMc-4xs?vgs#=0$g31N8|X)4;go>95tFx?'
            '{Z;`(*M&&ZYBw8=$K%naEcPObHVp!gB9ZP7MOg!~0G)WY8jaMmQtIETW_2iwwX7=rq)zveVIAUno24ra=m+?dUmzc2GyO*YJ)zDY'
            'i`;<U#ub9le<IQbiM-h#jv}r^ei)73y?^okr+3lMFTVTf;+<HEyODr@WBCF0L5eeuv++cX6Ij2jIN*cmki~1cEwV(G_==YxD&@a7'
            'DeUj@gdxNSnBNWlEylo0&1RFXQ|yk5fe$w^EGfN)hDk$=AK}s;0mMCE_R|kPy#4Y0#jEJu+n;`Xc@h2m`qjnT=;D99zj*WR_1iZV'
            'wzK`=a|ATo7dQNK$3Ji7YEQq!Hz~jP{Dz@_^Y$f7`^SsFD@eh(s^hZ8M*B2LGKs$dbK{S7Q-SyyhojLbksIWaBD;~T@G9f{S_Yqj'
            's3@9pEu*9<SyokYJpo37{xGp?c(Qy0Ys3T^_=lX_0@$aji(nigLb84QAS7-Ncr51=4>mr0_W3*%)2~GhJ5ye6V0YGlF#fn&FlkZN'
            '({y1!Zkv6aPd6njV*=YWEAs0p?M}h7!)*cbw#ZZ5n*hJa6BGmp=;A|K(O%M9jwBCmLX_ILann+DnXcjrwr^ZsH@xM<d)U@5fi*z>'
            '60n!BrwZ7kFqn~36(A*k{ne)SufK{z)_@F3^9E!F!=c3zAerLqHa=D(CRMyYz>t8yC9rul;fEM!pmfCfTtgb;$S@HYRPJEA!pn=d'
            '-^?_{=x4fdRvz#a0Walv94^el(E<Vg5zzy8%}`FxL(^2RoZH%7Pwaq8YPtXtJiu3CjxP;}ua;uIaJZ+8(@Kh;<E)VvWm%NL*hA)P'
            '=puj*x}$!Lz_WdvrQ>#J7Q;OJ#0=a^8l)tU$JvrCavj7r<5ila`CcAn?f_bdJNwVSl=ljsx;$DGcQ>h_X3yreXY=ok1V&4u2UY<%'
            'Bov>DFXpp3b(SC2@=%N5{qex)PpqkgD21WLiOG(6gH@nfsuanOQYZbWA5`%%t$<fG`330-0I3P>&(>Y|?r_b=CTEo^iX^^H1xW2&'
            '?BZh{?}UR3FG6}ON=4;#(?_O<26X3>#JhAC!*t>j#+#odu)|l)vDvu{W-S5`++Vs$4<2DJ=Pj2)$^Gt*G%J|(^-@Zr+s#KtObxan'
            'SSZg<LPdqs1vv~+fvRNN$>H@L3FGMzhgtss!}c#~@e%B25aSsLTd@3jRUc$mUt;~yQF3_=)Y%{x?c<M@Owyzvo?*eXk3R%3{mBU;'
            'hYUaow6bXOBxt)Km~bLKga7zd%Y~n5iy<Q1qjVF2-dIb}dr1IB&o!`oMfw*N1J8E7mFpdt%1d!A!FsGq(-2Lq4j3>`u&?26)xD*h'
            'dJwdzG<l>f&xl_p!Fr!egh8m){2&TqfRJX1P+i3B77g)jk+YY9>T=E6_zawhq5!iIW)E@-*z5p27hlh`Mnn+E&BpSb{Q%8#9=)m)'
            '@b#AV%hx|#&~sUq-RHIZP%qz?jqH)|j3c(69f~XqLR9xHiS@setKzgo>v~pYV5q>NpWFO|NNXfvZZszNmHQ?qCn@Oa!*(Js!T-#_'
            '<x76x2WA=8Gj5E2*u)zow6)6Lz)-rNUs3y+p$G&x$-w3=4jtJ*EpOwr4seRN0H1}vB(N}k&y6lV9MID50kun!+%6&m<Urbc2DRFP'
            'q-6s8k0?n?vqSj_`bbN#q;U73gVZrsku%f$06sWtty^%t0e8WE3i56&(5cMP<Awc>4l1~wO|A^q7x3r<`nuqi{`zZYY_KhF!Sn=P'
            'lenk{XepDKcniapw<xG^{9N9G79wz96;I{<P#>u=xM5&!GGd@!J@_zN01rG{$xN<mg!|rwR-#W04KQMW%Z#@5S_FIL3?fc?ZXpK_'
            'w6%>U^8uz$p$SW_0XjG>v6A5UvXz0pO~<C~5Mp8Riz3&ORwU>&b?j0!`gD&OI8lgi5cY}uZ{W?FqW%V@tF{3D%{E9@P1e9v<0J(O'
            'P{X_*JHwfwKw#HF2tOi>?>xp3_D3tKfnn4Z*RPn5v|dTeUb?_@BfQC@FwK!D@WZCbBk8dNccmyBmo6<}M45Gsj*V4-+GV?Ikq<EZ'
            'LB9Zd+BgBf5mGj7v0b}_<_)qPy&~uT!z?RqQ4t};2Yve<Anv+>&p~ajY~r;OvWzIR-N3%W5WsH*S&aB7co2^;e31AK^K8a8p~-ko'
            'q-#F6t1=U>&`90EMuiRAe2nut1;ZB?OCx<hHsIBwDJDY$q@C^ryy)W+%$-#d6S)>5@Tf+zA<CgCy+F~RR|F<G`b4}#K`8eH>XPYg'
            'Q6^$5o0269SbZqZK<8hl7;nt2sH+Y*q6VFfF1~;J9taox^7hA9R)nV77V907pMHVn`1?xMWqfRZ<4_ao5dD+mLkW2PTlI^HFWtrz'
            'ecGhgTZ-C%y9T7e-|DjA<`i_qU*c6kx)5fjyJy7>g?P3=J{^q$7lgu5NEI3mLk0BPDk};`rHro;2#m5V!J^RM(rujc=v6`UEV5?L'
            'KfxZP=kZNk;V;+e$Bcqe0DCQ|F`BR%QZ2Vxx+6q(F^wdD1I53km{}d;WHEk4iW6+0GG5maxuC!d5rvI|&1Qd)*U>5kkGQHqbC&U~'
            'C59)=h!JQ*^*42E|2`E}5+)$95Gr0|d(7scHl!#{<=MEPPF^;3B`yuDNelLg<A4iP*~P}20f>;g5V2CMCHfK{VS|4=d-lcWPhY%v'
            'Nihng2NjJlo_zkr?%R)I54NlL<;BnJd(^-H)SQ5!)Pv@EwJuXKs-AKn-l#+IBT9ivfN6~Ti6j7-7SOH9(Z)$e$}&WY5bO$G459eT'
            '7{l%oXw~38W!d8{j&A8b4SQhD4snSGD*GJ@Z2qGnlX(I}fqfknJIjK00sfB8BNJ={`_)k-Egu5wPHCf0w}4}`iu0T|)-)@wGmI@0'
            'qa!uPE&Gk98+0IO*F3qF`Jtv?r?`87uLGAULc3#CsWyhMLhM<qrw)(V{r2$zhnO#J7IPZQJct+b|LNd%v!QJbhLv2>;s@#N(`VtQ'
            '&lc*hE&b*6SuZFqou#V}z^XLB0d@&X4m3bFaaHr<^h31S)9=UT=JRLsg9nJ@K^(8Cb$nB#NyL6H7}89qeY}&=M_Cp;k>aK$CJv+G'
            '-Ig;``usPsRBS2f@L*|nHfetIdqoNu_c+BPMD(V~vLDoIWfUj}u-^q$3^Y*#8KcPahs0lc;z{;y1Y<~BLFp<LAcrgH+hp1fB6z$#'
            'FQ=c)=kvvtF^G5%{P}FG8P+E+a#7x4Gp8$Sx$`doQC?|U@N@cIF~MQIkE-)8=*R*`@kE2Yh^JP-O#V+>0u0`vuw19RxJ(~h0a9sN'
            '`qH5<Dl<ll^%R}UlKn@D4K2C~gg;U}kvHkOeY`$2;~_nU=e|u1yX1IiJAg*{>v*E9t0>Jk#R4q-B7=9|#K6!)z+63N^xJzgjY_mL'
            'U`?^16YS|NA2x$Zp)o5FY+%bo7}$Vyl-G3j!E10!hqp3bZ}kh@aCc`$xSdVJnFRf>enlAgGrqZwfHMKd)frFVEH5G$Rj#)YKNV+a'
            '>Q!e8ap}No!BGSFf$d`q`qsy@`>T7`GbCNg6N}Wu^dY4>RR12=j<*>fuP>c9LA$nkF_-*f;w;$G{)4HbX%(S!G7-3T7~?~DvzDC#'
            'N6@uv0=p37hfRDf)lu6@#(P3!`5a88I6=P#AhY%}Q1AulMKGG-UA4R$gM|sbz?QcU5%$%1A)e3Y_Y;9+jBt+tmG_6r1BVA+DRn7u'
            'Yr@QnrXproGXL>`f0iPunbkS`v!t^oBg(!jV+|R8%UwFR`@b)YW(yD-_AEYN#z3I{R<S#nByK_Do5|}LJ_8v6Rv0*p3J)6x3PpJB'
            'oM}X5aoe4W4Z?~S;>x(~bszGB@aPT)qn(1sXqQ)BpaQVLmopA9t7xN8{bzVG*&BgW3ty*sBb^rpBTcZoPQS6)tiWL$dFBXziwK9M'
            'yFJy`?&}AyhI-+9=t9lZ7)17-rk1Ga8^n4|N1U@cjQP0-=)S`}CkQe~3|zJ$S`5$uyY!P`jNAsTea7=zSyHzLwDq=U)?-&b_w;ug'
            '^noBYRVzg_Yv12nCv~l&^8;9p3StPtqqi2BV^W}jKC3`*6YDJ<cH<hu53`-&&u;R9=@a!aPlv^r9s|QVAST3L{d{&mIyq})l5kp9'
            'L^)bjVEfTJDXcX+U^7E+4ohd?CSCSqR$SQ!ES31-_zoezB3L56jxnP_w8PA-Vb0V5XE>j;IEfEv@1L<Q)^<cpBcs<GSPr&g_K=a_'
            'jE`#O=x>@K52lqjq4qtvzt^Z*@wkRPuHH^FlO(6Rzfc&yQZFW$x`7kW57;2k{i1b-GWgEWapAW|pGU!jmp=IW8E{0Pp&4Fxz;^`k'
            '>HLB!-cn{>B6k3x0rOQu`%8H*f|ZKiErddOA~b%X_A1yNSz6Q!T4#Vv5T&`0lF^=t4ef2(2R%(@agszDo4}rA&<AnW!^d_rVw$Db'
            '2k05KuY*vp*G5DtzlTna&@Bj0oI$HC#=BN`VBiW$*g1FVIoRre(gHw(<TxM#RQuiynqCHGJgpSW0~Gnn)Zmk0(%QL2oGT>{v<0ec'
            'NEz=c$4jY;Lj(+gXDmx`KA+%mbCdu8G>@zx8NQu8KM`%)0wCb7YYgWywCCr~NVFvBx;9eeJIblFybLhEsu<4}DWi#^ULgh2nGBrs'
            'bazI!C>Z|vVU{OxS;j~32FiWs5zUYt1d1WESAY${s-4YdSNH0mgQD!Kqu^60*fTm;G{GdQyb$$mAqW~}TcR@r1oo+8WRj90;Kr9l'
            '74t#@J{9f_^>Bvyq(R4^ugWEP%*r<2S5}XWj425eo(9Jyz-y_zY<~?e=WGZPk?ihh@q=MrkSQMkp<+(|gMI<P40-sf90JVqKq}zg'
            'jnmAtQq$xEvK;}wMw$}T%f(`Peq|l;Vt+~nU0S=Ux`f_W&e;rj$3eWRg0{m{fc?I|9P70lUxk?4<u3yx11%978(q)zwO<+ItOc$$'
            'xEY^QCB`LrMz(PSPZ<4)_#(V_CV{kdP{3pv-ewLis}d8YJ6I7FtpgIA)uKxUTT<{05JmdgBD}}niHG4YdlVg$$D|rOoVh`TqKU<S'
            'z*^z5R<yBH1K20z))EaEg{-&*=1H%&W^K9Y4AWMSzubo~R%lcmb(<mx@s&82)AMH?>kd%BKm#$XLK0==b7%hQ4THkGrJ#izn$w2^'
            'o;h%0bhN^yjLTG3LHCG286W!ZlLri<xRGTBoCL0JWSlj&rcmx(J<y#6c8iYc%C^_UoNOB({c%VEnUx|0Gg0Q1d)@&n^~G%d5Ms(~'
            '2er*pvwF{GpFMw|`v$i1Dqin~0Q}4IfmGfJsf*U?yr6XhGm-8sB$&sL0Ke!23Ch$5Q+>bZZ2p}5bF%f)S|vxkqp9%WiCQi`Aylxy'
            '$V&5d)_@KWo3yNI&R}5ocpo46Xl4&x2^?kp<J@lJl5|!qav7I+m4YrtsKo?f&8a<fUje3okPrv~&=M43%)~D^KuvKoihgK-Ehv~j'
            '#tsc#1<)1SHkC}+pr#897>l8Z108^72?3H-J=KuJN^WBe*OW}8byL?mH405W_?;(Nigi|0bl+rK)cRyA&P8F8My`t@!P^ExU50UD'
            'Y<AM4PCsK&(BWw%5cGL%WD?(WKNKa(r?d`8hPorIxw8w=hNJxp2g=Ced+On48hF!CoBh)A=0P`2_1n(&v1~>!RMjU!oD@TTM(%<&'
            'T4+bgtgyKn8m8#s53_I6{k%!&=mJdP$v=AvdaR)ds9hY{36PimT7K#*l+uT~XUhVYHe0m4kT_Rr#arP^*{hUlaYqZ?QtQ*8%LXks'
            '>CNS^6OqVlJ7b%-qj5m=UY|CpAgY7SrRnbjvzh^U&6}^5F7>pfmVqY3wfXNs|9WXt_^KE2qau9IghCGUb|*?vUY*w4LH4Lvpl=o-'
            'CrS5Ak6@b2W0P^}u-3E(`>(VaGoV8)gEG$2>s%&=PdmUyUK&CT*vM;~&)BqexZ*hO$&vPlF%0NWhVkyxykU3(6~DuH)MlS|mSXDc'
            'u|Gg8-kM#=twW8{!9p*mDL$G?YU?|)&PMsk<H+P=F+ikG#4EYM`QBju<_)l3O!gn)`;>(ZqehYvO#m_<*cv6R%XW=|#umv;yoR<-'
            'RjZqbWHr>rBvEDq>S%3SXmf;haED`n8qTwcr)<Mf4+(+ItS1vk!E%Q^I1_YRacM<d$A<#^XpDxAdHz#H{%Mkyd}B1urbM9W&AEMi'
            '?Be7&WWMybgWY2rvHo6!ztczQ0cQ(2eVsR_JNv_0&;5?h=A&@5{lIEz^B=Uc|KwKp-86r1iQ6Q9XBqLaUO43&5er7vz0sgeE9!S@'
            'w4+HL=4l^h$WIt_ZM?gH=CKp08umYU3%UNrXX^GtiHQ(yOly9^6Z&ttQGeKhdf0V3iLf%oA|k5Y@7aldX&#Z|5W#HmRJ=a`_}TN>'
            'yge*%{j`^ui2b<B-sWz(2vWR{O&2ivHX|SV6PPK1VJOp^XJ^Q?TW~n+Y)<Q5x;~1q@4BmaG2M3$XX0hCU!^(N21+g`xTN(_G4K>z'
            'q4p6)=wRE)Qg1jX+0?XaHwtQ*4bBIX08qGR@&nB*0p(C)YdaC}R$?kBIGpq8`Fzeu3d}>OvJ@9xv9U@Eye_3oQqUWmV-IfM0SLw+'
            'Kim~+3>X+Q_bp>w&rL}H1yx^j?ViEd1rbslD1iCxCay+7rg*A^-o#ED@4n@UVWvu4p(bw0#fcJN_~)qIp%JVQ!FpA|3cY#zUR)#f'
            'u@-`RZypN}NNL_i>Mf;}Rzc``p~Oj%nspnx)G|Uld2L{nEqhdbzIX!Drxh*gobCz!n4jCC#RRu7V&#lOaX>2-sTT>1wOs;G?VOkp'
            'KSepgG<SL}Tzf^XwW;hn%YbK#de{?HYVDprN6zsg)e*OyqtK%){Cop=ZuywvYimxpRp_<lM~koYgbbiI)1fB=<B|`%uWYP(XQENz'
            '=H`^wNv^>8mU6ZMieWsu9ACbe{_lAD@&Eqk^#A?W+0~OV-{&&<=x$~Pp5}HbhOaK2OoJ;o6aO^cowhpD1JBsWr{-hy2_}2==6X;g'
            'd;G5R?NXVKDE4u+!)u~^umJvF8-Nl2Lq9f)75co*hZz_vZ^2!4u83>Kg<YrFcq**J;PaVvN!;1dzc;_aiXJ^OzoHEkb@M9>+hgDz'
            '&*R<qeG3UK`yTh?M1XbK(3vvH5nNq=d}UxLRphi3PaWd=9m%9=`&G@}k`-TJOt4*3VmEx!6Tx_39A0zQp<`u5;2G2buM*(x5gD|6'
            'G7<ke4?|CI;$f1B4*2WE`q-O?-Wcx94tvWO3L(E)#1-em3mmn&-JZx^#@wBEJ)>_)fdR~wVGtR^0ALa(m8=yu{rKVaXB6+wLO&yw'
            'g?q%woEholQC6`#d32?cyE+q}D<&QOvYiu;^phc4P5LHlr}$d;A;5<OBp2pL>hksy{$`PWiwo!M{40*HK?{84&hy@~uqxL#^;i<@'
            '@cE>>A;?zl*<kb8`STNGPz{C|2VT!7{p$~l_}q5v=sYt|<cEnlrO07RHhVI90*4k>l*PBH3{`EJ))Jj4d`aV6A#;H#IuXwvByB6<'
            'FUS6fpoEtlG&;TLF*RkxXzTKV@v48?fzfh<!c4Uf5#u(~!*PFLr)<E|EU}?Qr4)UfN|pQa&Ka<WWvZyoPJn!>w&unS{+p)ufq)fw'
            '+hS<B((xE?96YQX1ZcPF8N)v_b*9I1L=Qf}W7dDQ$fX1=yL|zkGKZZr*nkh8ZTswrI*_)g!qp;1oC;zh32wSTTi=W>YEr1(VP)rP'
            'pndYK7zRBF@^tt*kF}(g+D@2bfM(mM=a5!0g~0-RkZn=PJOp*u#xb47t~Es`k*g|jsb=7Xt&gBzG5oOOcz<mor>9)+uw(c!KqQ|<'
            'Wb4+YW%sotaq|wmFp(1+(h*i2+X7+^9-McRgEbvIF1DVx`)auN>`)wne(Q<s+it+A3#{YeOooCv%Yo-znB9QRviqpptuLIj?0e{T'
            'b(h{EAv)wFInfHRm;k-e^Px^1|J{<k>QvTlLFVp5uv)3q4@svb55e(+1{NT%n9~!{32i<Aw2RvyNEH+v0_g7u0_i^-Tq_EDP?w&p'
            'V75k<cWs9=U+JA)U7p!C%h}aE7!W<6h=U&l<rq4{u!yT42IiPYgV2<f^dLCTngV#^6C1)Va;TiZRt)>uSk5kaC>cvBJzKnTlzbrK'
            ')VaTG=UiJ*-1xbMfwe+TWzsz6W^zn-nZ&<5n;)E58O0OZ{0t61+5?MygEb#j$_L%pvVwn<N{7`G`6Aev?uj!y69MLQ>qm5n@GLaR'
            'kblbcOM|DrMQlClw0Z2s;ku~ftVeiSXwS&=4(1bXJP}Qv*30n@C_2WYld8hoSDW$t=^Zt=7e$_a%;a9G>Kdr06S|yb2ydD0)~M<t'
            '`SLup%i*+C)@RC!3{PP<wY{S|{&{(Z8sY3}F@KiaQ*FsMRe>9fZI~pCoS2Up!phiD9lN4_+IeTm@74|RE~Wk%7gN2f(kIXeC!JfY'
            '-j!>HrSDh!1n4xhFHcXOEnuUm8?7hqM>iRGfqX&QgXZ0a9=hu6pyuI!<rFO%yrt~ljvcrmyX;hz8DyLePHg}>^X*I@2ruZBE<C00'
            ')SlCwTGR6r@d71nljc~f3pOGiOKh-c&{ihx5dh_@K@%2Gu)$1KXTE+KTL3|{XF>&?VC%A2P{o2e+nzV>>79Rvjqdxl|7f(%Vi0kT'
            'L^IO*O))JFqF7b3ypdERiVH8<MMkOMNri&p2*cM!1FM;kU^PBlg|i4MnQh2jYB5t4rbt6zuDj{MQ{x_MUC7JK@>4z0@gqPXvI>%F'
            'OKNYRb<b@}?CZ5wxyaQve)jweF3dyBg{=TK(=SP{*?U&if@j=ZsS8q3RG^I&@3>rzss11Im+^iva-FCq-yzfB-Ozx&&4p??!=k>y'
            '`TW`E6c2g!IY^V0Qv|g|X4V|g-UtYulaZ|6Z{-L6%~Z6rq<^F?;i!4rih;@wFGvf!L;yZ`S5qx>q=Eopns9M2s&;F%w_K-{DshXO'
            '#Ow{=MvzNJO-tJUD5|mK@+@ig2g|MV@xjDsEPNTo)jCb}MPKydDKz9=5w0ZOH^%E#%V0b~;kFoKew^IIfWi_jB~E~wF&4qNsW;QV'
            'jvX?3ORew?d+54q(#$xid`b!g2u6JqVZy4DOI5T?cST)FIR+Dj2Z}Cuy*jdWr=I@S=~P)Yj+MZ}tx}oWc4PgHDo<-8(%jo`sb|KF'
            'Ywr9XY$`3)Ma1r%8mcxdIq8?n9%~rlSDz$Q68u*IDkj~;Yn>vE`Of+g+Gph8Ap1}Oc7%&r_}1}o1^#rNZg5BHL?)`q?hw^jqpKrT'
            'G=72Nn$io4Oj0#uJbDJyORC4j8PHVGjVl&&RuY%y#?<2`ucXW|@2N^boWLx#O`i1TAXgwe_c1FTj6~glv49)aPIZa6QD^s=b6f|='
            '$%}ZM!s``utWr>CYmD=L#8O!{uzSPov{<FmuNKdFmr?ENVY<_)YPH|L{r=+3`{>7upI=}6a`7tq+dr@#oT@bXrIr?8Go||!OH;>K'
            'JGYF>6s*nOV#Ip0Aj1Q;DJzxEUI_o3V#4n(zPosd<<9gxUcC7?aVS2fST1!125}wlsYvTUg;u?stD{Ev4l&kt2?tWbg{A{)hSp&B'
            '`Qg%D-m$txM#srEO;mAcZ{!n`^y;zoO?i3ejB#((Wc$+dxHsdyyurgdY^aK}ag^1=D0E7-*)q+_fkt9cV<HXB4~q-*N<UR-KxH$7'
            'I?Rh(ERb)2UQ&M>ka>w0yoZTrYUcSsAA2dblu3fwGka8khAuv|`?fB$yQCJTr<fIYQ6`NFj*oeYk}-CMH#L4;F}d{TeCeQYHR^E;'
            'O|?yRvTqscu5-+~({}pIoAe`iUq`N}Xx{1^Z9No<cgOuE)2y!uW-r{Hu5NKe*E8urcvQ(tkN8^cBt>;Z0n9}(Z6xeMm>oj7hG@sz'
            'p<>v+8h`W-_fVyR<>)@-5?QuvRi<4I(?;Si<-Nb{$KS*s#knfB4zAKu4Fs&UI>m~s0yY93L!yg;MH0)QN{!Z8snS(+Mp@+Jg*dm)'
            'KDDtxm2EKB*S9mqZrW?zVD5wUoelqOq++EDp_y4}B=Ck~%Edu@0n7vmCNGwjx3pK5qz=Fyl|+RwtUWc{99nwmB&Yi`3_q5=@HS|t'
            'LD&zW%|hwzZ?4YxsTQ3mh!3FKVhF|%3jX`f4CsiigE&3)+nU988>`1BGi}@+)+@i4BNG?h&_zemQ7p2j*oP}1@FuPEV5qf`VRjGm'
            '&`Q>@c(GTvVoH*F?ld)KA~j%hu&+T}w+-+w`nHDp7|#rpV4%OpGi&NKO7;Rj$AmJv1C4zeMNRwBgJo03!=5jI`yFm^H~>?Lkr;?Z'
            'czIB<tGPjk(#Xlnr`<JX$1oV^#%*Y7clQn!=yJ8A-u3f?UI1v}ddij%!t)Qx7%BHd4fV&x)|RMU_*LKTbt(xv$HwpJ_m9n=wqtzC'
            'X+G`4wi)8U4EC3KIN6x@IG$P@P_tB2a6w~$z9$$;6{jxOaVGKJ2|9l9rY%Qm9J))VpmJJa1~>+q0t1k*6L)hOE;|ZK<g&`?D<#Fp'
            'tl5b~UZ>gk%ARoIoa6Zla^h>niP3zM0i|?83r*Y|K5_T+<M!3W<4wB;8(QL<RAvbqxIDjwZH)0-Wt6u(;;<vHgg$}Wbcw2JN}t^k'
            'wEHt5cU8J;ZS9U*t*VmKW7i<Z;Du5t4&6g6@px{fLOVEhQF6P(I;*PI-q&HgLEWXLF}pU;AQDb*(LsrMZD?pMuxLzUK|L^GYfo{1'
            '9Y{(1(WsC+7N2!TwYkr)iZwX49lHRlTMc3}p>bWzKn{y4@oFkWSnpg<$B0vt$c)lw5HY_Swe~5|9j#9+KW@ODo{t=}Pr{oqNzf0o'
            'S+^_qFr)s{<6;UU%B4vHr%`xfm*Xe=Sp2E5VnhM%<?pf#2KLVh2}ygZf9ydbcfx!~Tkxs;fSPu>l-IJC=7UiG{J_*B!*;ER>C;Z('
            'W8!$j!#Ib`y}QF5&%{ZGA~6pI*B&#5ijTEf5tUX^$6_}QAESq?z))9a_L=WtrwV5WQ;rD`SRF5tc_vG;qrfNu1pTN{R5klO-;O`M'
            'n7&N)0VvdI#c?}0vwSRdXM}q%$*~5CtTI{I3U(lx<>J&6)%xSnDd_oNEUG8kG_X`XA*Qi?7(Pg6&zU5O4+ot75H2imK;d9Tz#=bW'
            'mQY|<j~^4kdhCo%fXY3yQ(MM9ma|2cma)NLYLuNluT#2IYE|cM?*mncxPG#G>35<N$CXFZ`QC)d9i^1Phrcn={d}o=lezDU6AuH_'
            'Gmq5Q0U2BOEU<eL_&21tItcXD6skV_xCHf>iFa%jy5AN(^S+OJ#<><Lw`;MQs)IK+FXdsvl?qf{`ns5EWAmy1x!ksh=a5!&&+3hj'
            '(ig&-bfZ?2o?zUh6JGN+7S#_kVR+i8G;ZfEo#$p!zljL&yqH%u47){>`;Q9s<WWX0@2rtIpDX0=@mQXd{v08hatm=sgSVdzwJ;2Z'
            '^S8b}mA@4}7-)Yy-ulL|`c$SqwzI9@K8!p0yDh@qeBnP6U;X{uest{;k41%Nmr=nxrj;I?Rz%ssMdp9*gA^6?nNg8kZtAxgIhpGu'
            'J6xDu$<z<Gr=6;0ug~nAOHq8W9wYC47w`K6`TgI~?vLklRe<*UOzkZYONwAwlrJ&Si$2DzofieaqA%BjG~hHBeMG{!0_x-JHVS<z'
            '1TSg!JhpnZ+6C)Tw|BkmscB`xAyD;)4TE(H0^r~JqKuvM1)B|YaMk0VwDVb^p>Va&kr@2-9{DlF4!<ZQ`WOz&baoyOBGbpt0KD(9'
            '*V9$0(8qny*V@*!y2FY4spj(rQG<DGCbk{)l&|*-9Jq3dD#XUCs>mAP>SD90BsvzDZ|yGHmpv)+T}k}ZliLJ-uD&$xK<`<Jo&kc}'
            'ZMxpFswXAP6%)DQ?xjapkIqjzRJ`gNqS%np0kh@&_wY;CPDj(03+ZYU&<#_cFxO}1Io2zb7;~g=mCw8j0R2ZMw)g9V{O#+MSo*7D'
            'QCTJ3SS((SJB=t`l}-gZuR`|=cMJmT-d7jjy!h$6_t7sGumAq-d+W1uf7a#Gf5OF6eNupF#LTUlvmX9FTge~O;Tr|LhA@Ahdf4b4'
            'GLxa-v(M>frus@tH2h7SLEyS;-#2&6%M)MO8J#ZOI#^Y|Q?O{bGSTSK&Wo;CWZw%=AA)giiVTC)1>ai@sPA67pAhQ$7T<zCn^JoI'
            '1oQ8%GdWe+oYPjLbNP6aaC%4&;B11!669R2hHC@yH+4+)9sm|D+Hc>f&opn+>!wsW-xKRoOCb7`1I4tjcqJ<H=^a&GoQrp>5;T<^'
            '@|l>sJe3`c;Oz2=U3~m8p46x<znWGmS2AbwWj>FNMN>EQeJBpi;%9c~%R{sDaLiYf^IZupjn<Zh6+iy;Mt$D%#hVx3{o~#1cVCM6'
            'xwkjbu~r7H4GdAB;v`e9frCqCa0242Fr6~!6#bgG-;M}XP3=k)x4E@;?ONOG2Ll!dOlYTsQJv8aOS46pPf5tS>XVzJ`HA>db1r>F'
            '7%R-`5>7f(p0m4WocA6+Rhjys`p~c*OMyFJ{yjY0QergL#>0xYDm^tjqV3Lx<CpgLz^NN@4l){j)4VDaKy3BXYx)*3xL|^HDdB?n'
            'g7m|pxtOd{gI>ei3RK|_Y{fLouo?t?Ky>zJ+>26QQ#RsOr&n2zLch&pFGg90<<c4Bsj+@=d$bpftP=hwJKV-M<B@&6ikA@sVU!`R'
            'j!xA#f$^?|{sfy9E_~>-O%Hxsbm+69_9sOj4E6L{N{)fqN?{v2cj7b{PWc`3+YUS}v5h}c#%)h)rJ<nxX6$~8ZF<+$iuau0Wl_%k'
            'LYos^Q{Nuze8kSH0px$pZrL5SrJDK}nyYlZkD~P{-H8`@C*%~jdNj39u|J_NpVODhtx9Yu7ZBUTb)12(fu%Il^^~^dQ`lltRsW53'
            '74Oq+tmOyz0(ifIKc&jX8rP^YMdH{sR*J$ZM|W148kq`~u6Ren5xRi63$lh2)){M6+=B{AVb#yq?|=Z%3$JAqxhaYFN$a}XwHf1^'
            'IL#2<rPA3IRwD<1IbUWG+|G*DX(0*ol`PVv<jUwM>4DOK6(B_vxk2FyswvKj>qernj?#>BtY_+!sf&q@Rdl~~rb0b^o$MS^Je@lG'
            '=^rnCe61Yz9jz%Qt|vvFG$Pm)c>`7-5Uo0-wV=ZiaAs0a2;W-O26sh?>9(-F&b&>1R>g_LU=P*_2Fm<~d+>?+dd^N(dx-|lkv>A7'
            '9$`+$_yCBm<K35>Qkg;Tog!`*I#cogoZ%gDAm0(`ht&&nPE<c4pxIgPZfDg?!a*aG)`$g%+ndKrM23XU%Yz?;TDq3(?||>slg7fa'
            'Xf#Cb_h_n1{2a&?9~u)@q}?vBx|eRL3KE+i6XE>9H`Hi}`fPIRa-P-gUact+FKNaDG^b6fC9+`%E60>rsZ1Y9dKQ~Xx}N_2PS}37'
            'WuiX)()PYopY^z!I1lmL9#^B@Di==qzm9e?ALHP!ufUp^dRz3QEwvTu>c+N#)!KW&mQsB}0Yo{lOu7a~rh0lluZ+mk71|Jl0!G)A'
            'fIVI+lS`XS^24FPt=M|sd|`2nwpxszIH0;Gq)3cahT=-sb)-_s<k;wl0@wl#Y+sPvMlO*tBHo&>7sTS(T+96S@+f^A8~wKBa}tkW'
            'vSS7V)hGpUYr1L079N^uFl$}2*(8c#K2g%TSCy(TbUvYjcLJ^?qVdDsZ`p96RV_1z9X2jU)8T=y`-)7O(vVyVx7+J!FYZpDzISq8'
            '+o@Ns&x+Y{r^{&4w_19^PR04DyT#5RNL{-$=ZW%`pNMa82OxtU3UthqGTi|0)a*2283j`dw5@wzZHq>w!cnlty<DuY7aM@IbxIro'
            'Gj6~Vrab^0r@m6d1_#HuEkIu7R>R!L8ep6?I7GZIi%L7le2tg+dSJb|u+?+D?=8aH1>rqX+C8kgrfrWwwpG2IdZ^-45Ga2Q`wByK'
            '%T9Mb!FsyT^rWxO&rq0{cQyUr4#g~O&S}wJ_C@zk)@oYzlf|}Hl?Bv$*4gu6>M?klc+Jf!$_v8aXdk@`R_MSDAe7T82Bme3(XIij'
            '9B&)ZoszI$@2RLqMXtr2Ul60c)~>!y&W6wL@W~#mmoiu?<$;PQZAA=FCV}UAS_Lx91jEObSkcJ)M`b6H82T<t_h&FHALYJTrp@nS'
            'QGMF9^n%ZpTjjXF%ZHO5me%W0zX!a0&>>!)ID_c~)R+Fbm=>*ceW9x)Iu|c6(|}t0%<Q++UnmI1fcv$qr(|zTwW`JM5}a1OL;OSr'
            ')OeGAkja#rPK{VVsNvA@n0IvL)YIyyvz}tK&e^5&4P(3TbElhgpy`&%HG;pVn^rzOq{eC@rx)wq7K!cAOP^mzr!RdkhWnbY`FdT^'
            'cIFH$(;|Erit};a2O&DGe}~iCqTrsLUx_D6G5%8g^5VxguiyOLKEM{Ycz>Eg2mv0K>5BF>rJRN1fwL*+Hb_sluUB_s_!fs%GF}xN'
            'fX#wtHUrXEGy+C3Rimov?C&rrOdsEj^P?TE;IKrXrr1U;LJm>j_cJTskl>I!n!@A;lyU<OgP=-R#aeP0B?cN~c&UlZv~2h$iB(=S'
            '(0LKgH4gi<(M>MIZYo2SsCe?81Y%l&b~<8Q9NY}L6|oR!SgiH_%uN$UAKu(Ta>gIH{9^PFE)gYv;wH5>=zK8S@5~zDjDjv?QBWGH'
            'lf2{F<T?*Tt6s`kBD!*7=|ns}dy6g%PRt!a=U8*>e(*j`KJ*va(vw|u3TSrjytpWLv$y=4Fx-66s{#CFwupIo2gu%EbuYx=`lrsd'
            '7&npf{{t)>gmM'
        ),
    },
    'render_utils': {
        'sha256': '53d5053906b6586f80fbd0890c7680173e054e2fc6b45aa2b6210ef250a06833',
        'encoding': 'zlib+b85',
        'data': (
            'c-rlK+m_q7mFRnY1-g38P-?SO7hl?uqt0^M)oGu&?OwJ!Jz3>CG9^(ZMii;=Qf0YDYt0{=m-(FYHS+`K7tX!_AOM1@@+Fg*Oq{e$'
            'k-)}nV`Ja3zrX(~uFGhW#^Q$`R_pje6md3-i-UEtj?*NI#Uf7EaZ!qL5f$-FOg7@^!I%q;RiQz$-eeQeyILgE#a@!(yHZ4%i2qV2'
            'mr)vLRVbe1X*4-VpyC>9XO)=d>y60gqFN-SNb`$i+J{OraTyoMd^1k-X_OAun>~grlGQpds*;PZ@>va?o+U**t<sH{=LLL-1yz!R'
            '5alXg#-fVLsz2D<dyzxU>;gtq#G+oOc{D2pR9n}RG$|L<;|&!Vh-a0^GXPt}Scrknla$KL#8r_1&^_~YO$|l@-lQ;nWxZaf3Cves'
            'SL?bG>!?}`_V)Mp_vS^u665i_uIeHlkL9QUY?fD1mE>8uw<q5%qH>WY6ZP}YWuB?$B34i3qOOuu|EeeJBA>=(soqoz90LrBTb@N#'
            'TqUbmwX1%I0{{MyXEB$<4ny0j*w65DZ}0V+$8UalJ^th4=f9l379(-JC*aS1{2nj_YG+lH?uTN37GK6`zQz~hbsA0MMV`X!;HM(K'
            'OyVoRA3zL1r~Qy2L~H2vCH@xsNmkbLc`{9~I&?3>m-7f_nLb0oSv|#x8soq=0AdS0`{nyL&)z&g9sl~{v)4bLz8e4Z^6BYw>SItP'
            '*(L}@P_AHY@L?6tl6r+tX;fUq2bXsbU@<0n0Dzu6{`t+bmoLU|emwo@bo}$Hmp?pvZj3dU#q+36t6(T!&`anOV!;>?w}g(t$K$a1'
            'I?rJsS+t7beeg8?bM(hrypFOmaLbg*2Q2gLiSZ$Vp<c%0X`bfAI4Vno@y|aV!Ji-Cj|pNDm7a1x{NR>bMeoN~c`?Hx_nh}r*yQ;B'
            '?veGOxJa_`CA>d5F+Tu~$Hf$yg)cuuX&D>e(zvRCLX_)h3b+RaWW_E26I{T2jPv;z&=YX3ylFm9y5{qzsF=oST9@MWlg<e}`kPPa'
            '$-|GG&^JZBdp-vy$f^K@nik2LHU`Y!YK=JB(<P|X(<ol$nRo;3N5v-2Lh))5#o4Ba%y)lG%DC8+=BuAJWmF^)>=+y1k9l3jX2D;9'
            '*2YC%hT>@vT>vqwV0A<B2k7(LpcK!6dctE1&a(o$MW_JnN9hI{!=A*jK{tQ^x^&-NV#JdcF1&tn@G8Ejq4J=r=54fCL?`zjV2z`@'
            'NB8bUlLz-tP99E<?%%&R{Y`X#dK^EvJDZ>UW_B;W|1kRY_~HF;<CEi)heyXzH2?O&!`XaxeDq*4n;*46*F}nO7ge<`hlhtnbTzn0'
            'szp74c`Bw*0*KmRny(Ho^86x=53y2txQdeOFrTOJAV*M64{tG=G?~U3AgMVCFMoJGsNPq;VqdUxo>nGs1Okl)s+>l&#qbEh_F(hG'
            '07pTSKn`v<L<Loi;Ul2ieUyafYw?G84aY6BC-32qo+ikNtk!_waKIiGd74h5=~Cphmj>c-UF9oib1HuS@{K6qKu0Db%daFnn7oo$'
            'k5vwQ1suRtQl;?$9!|tpnG{M$$|j--9CrgRDICKF{}892e8N28>K(N3!Nb2jE&TRN)57Dy$u7FUGji}pBw?^3H_(Lnz1hRNv%B|Z'
            '$MXl%Z|@x+|K@IdFPhHpO@JuOe{=utaeO~|@a^P56cOE+9UVpA&L-cUMEByO`Q3Y8nkFzh9P%Lk>Hhau7mMnj?mzw8(gNA3PbLG_'
            'pgLFP<2JvyPg`OSfBzVmxtBixSNH1Wi__PVyZgiGpMHJ$>M3$}y=E)2M8aGN98?QB;^-ZS8+54Q>q?z@_~S~Rda}(R!Q+f!;OAL$'
            '6w*&Rh4HD7r!T=sV9twqj4bM2e{XLOI89*l%RI#$8u7sneOSdk<ess`8UG%NvMOK<K?IZvK+)>Gug3V7{ucDb!FM1oO{*cZ+$i}G'
            'a1dH%F*LnOvZO>ZCmYkWisS)JDyc~nn-r3uP=gS4ub??-)a@l!;U4w}@Htuc`hzsTii=*KswZ;;!dGfELyZj9caW7tk_kVvKD2m?'
            'A_8^>d5StdEsDIbSt9&1-~T=ViFFmjYU;tunZ4F+epPHAiG5GyReU)`36EwAWIw1>g2)uldj9V9w>Ff%{v}s?7NXb*XAw-KZXPg1'
            'EdkFb^F%Rl-OYHjyZ{*o2>om<kwC76i4wFBfj#;72hO<Zf>tksQuZ|UmhlG3I*Hlp85ZD%eFbY~6mIv<;O6kp3uOQH+VcWnWapfv'
            'ZO>Ort_a0EkbM|lODNlhpS0(%3#bzMU=gyMr++MgjtOQUcL0cynR#qGdjLpQv!G@vZqg_u`{0zRmDpPn3&0Oyi%Wn{xY0ZC?@U2I'
            'm(#zM#73Thbb^fMb(;2&%o5!I9O7A;B^S_pm=md+7;qhlMptlx0N)g6R1*o@1n3U%T8jt?c{9<7M$<)|EfFsgAU?e`T1{rrP?j7Z'
            '_tblE|Nh+v{ZLE*UF}8UZVc)*sxNv3sc$ZU{Jx0a^HWa|R5(UZ?PAf>F&-a1euTM!h35GKN*V#WjAu#FCZ0b<OQf8dMgj;CT}DYt'
            'JRfO>(v6|bf>D-Yon#s8u-896K7dIZ>=Cf<`H&NQmcbkX&-pA9C9pQYMiO*VouSGrrc)?>4~!ceNvI@=Qt^8_p$XOxco33CN+hM2'
            '<yRSMxlj$2P2p8m<pkhq0f*=Ta=GwH>dI)W!0j^So5|UvPpEuBC0IQ}jTo$KjJmc6`-rMNcv-^APf583Ex>t_%;#}|Dn6W-ftoY='
            'SIyW+m@|H+arc}`DJ}+z$q$2zxav8)ojy76JWy|k20i8d$h9QTKtzT)vl5rKc~#~$aO%iblpENgE5ZxZ2_Y<v3eY1|hV>fUu)hyh'
            'OZXc$0FJ*L5$g&o21n~SU(!#mFaxwqGSc_~$OinO9taco_v?6y1cWzQGaqLb-+Jzzr3UyxQqq+5Si9QZ=0;h4EQry_ZTp6a#dF4l'
            '<V-RhLwe4v>82f$;IV+;)r`Y9{|wEKi#)H?q>s7R<NiPrF<>HirsY3WQ?ZTMF`lpn&|x!weopmV&7cY8#wj!l-);_BZ#zI8_=#0L'
            'G}o><1RY+ub6mVnc5}`lG63fB>J)E=Ym;4XpJkoU!eOg3Ezu-wM3a8g4@L+)Zxv{b7(^R=9FJ5_gEWJOkvv@Y^k>+`@E!PccZ&ss'
            'Yt@_WK+(yDEvSl(J#sqP>{bTwlg~P9L!w}H&f4_6u15Fn(*X$Rx&{%z;Wex-4#=u7o@}bP?5Xm65E_#t7p#qe8cu_LU+q9Yn>~J2'
            'XV_t8kb@+HS;E=jV@)1caGYEeHJsB?nTSb*3c)h@5ZGO7MpMM3`tNy>**nL#o@Y~rHKzLH=!U5#PWBVA^!BS0zn`Lp>6HFK>JEKU'
            'u;E*~Tk8T2=H5K`<xh`aJsa|zr%4%QrC7pF!|7Hlf|d)AFAQ*7X91c`r1*oeLT|mD4EER4*H#|^-I`^kldN<JgT3@M8UUBQ{@x9U'
            '<`Jge#H)yApaxFxLsFy>EJZX+MDN<(X0$@*+dj+&&ceVWn(hA(70WasaFHnE63|s##8cpmHqkms5$?c|#gwo+-P`NjY2M><*Ix4s'
            'eI6JiMJYpVvL(DJfYV0{DRx#ZxzVUv@-KS|^TXCrJj~~46dq3iT#>|j-AQ|Zt(T(L6aiUl4y!bY@kf7P{sIARBPKDiTAE?YVU*%O'
            '{_Wp+s#noPoaC}aw!W5&WKD`%`I5S^3Ar`{5251%5f@rO4m2hbvd7Y}U`9?=PqMNCw%qC$Aj33Yt*ANd+(0}b3vMY^H3&L&87pQ4'
            'D&Sf~u-MWdJ%^H|AZ0PIy}*DV&o17{)((<)eT!R$L#$q*A(l|x?gQK4wJt>-YxbP%$%<;aQMTs;nYshaF?|Aw3#2Q%DpEMLmht;o'
            'ZcVcfZCLcpMzxNmR<czIVij{3z&aPWV^Fed3z0C&Qz{FqM%Hs)C1Tf5GK^P~c!qW&T(Ja%o(2OL=0y>$7YI+yW)EoiK<nPi(Qgsf'
            'cYuOV@(ie6#j8}N5t;d6v7ni&sF(@$3b{J&e@Or%@hi~W{0f99l9gEHEKmcUiQbbTj_1kSx5K_(GDF+KOrSadgST(-C^>AT6*ymd'
            'r2Y`!iDPOSf1MqlBf}UB4W*X^N}rF;#T@~_h5`IbZ%&5i=IY=(xfGK+Nr~Q92?(NNW7c{oMG_qdVW-Th#Sq29P%Pr)Vo}K-le8|z'
            'Wpc5K=mqou&sJv%6#?)s&b){&&;da7U>m0FfpAy>B}{Q$Ck4u$uq>=;Bak_C7kB4$5drnUErV_j0A~g3b9OYmJZAwm5^7xH*Y`gv'
            '2MZ98W0a!LsKV@=g-#x90H-Ay45mfC4oLll11x}2BUW4#NQVMjN=20uircs#X6=>?WFKPled;J6QW}r0X@0lEYo6Qf5Rh&fkMknA'
            '06LZeQ@bYi9SQX)$)>2ziNk#dAU4|qrAE2$Xv@rQd&YRn-mlXHxT&+r#kee{=Vx>N_mSH(Rp75a+VBF$G9rW=T{p<Ng&ugiHTQMv'
            ';ACfqW-)gNDtiZtvv7_Wnns&Z@7IH?{^7~6_v66={_R?fmgafBKWP-rQn$hwdX4{lRcu9X5-))~#Mw|7XvaI!Ud;Y}-`ea^JwtMD'
            '?DR!c8l%O}Lovzo6a>tAx`b^w20j-~otb>?<(C_k8`R;+0u~k6NhA|!6ubmx1vTaJ9URb9D6>C&D)m?;?~z^rUPkAlr$!^DX|fhH'
            'B<{8J)gq<S6sQ2~gZAkAEM1%(6D-})K9iNR2q?+<z#ss~Wq9V$-9vyM2$6?7x^^dE%YZdhckXYoB^%(zj3e^HlLuRO_XghhtH}?M'
            '+Wr~xGqd;{A7B~0+X5@-g7_B^Z?vWZ16hvH_pC(cuhG%q=>Cwm+I53T+hJ5yfYBrP2Gkn{H@bUtbcO|BKeZc_A5@Pb_zOr0kpFUY'
            'GMv>s(h(fdsPi4pVCTTpkI6SCp-(3O-B=D|uey`}Q4{|OlWt8P&HLvLM%xVQOtN|gYc5axsJJMHC|p)&@Ec_3Z$e~kFKD%F^NZp@'
            '@(M*r*`0A2Pvwpl|H{n#i2mXIt(^}CPXGmgXNqS(^NVKR*Rbj3j;HN-un`9fz#JfLSq230AVqEVWt@r_i6-zm*y(|Ih19gPq^QQ`'
            'fJ$xD>DTC@T2??<3(>1!CZ-Yep^t1vmS+ba;vz>~Y$a9izb=3ffB*8$kK&!0g?FL{Bq^ViaRJnxp%11&a4V4WiC25qo|1RaM`mI!'
            'fLSEwm^g9ZZ>}=&?w|n<vmQq)<oHo_z-%Y%msvbJ062LzD-X*2U>*SjUrNMWlQB_57ZJ)YG}L8=a3$xXIx-M%kYxhU5eSZ1jk3@R'
            '_7uzslG0GhL=`!{fPlDysTr!J0Kuq&*<IvUAoqZ1BVi{*0m<LB!f_F=3LT&+V9G1muQ<DanS#c-Z<BbA-Y~o_X{zQMW)mvFaf513'
            '{HQ6qG<N{CU}sqtieTKZh9Ep$pg|5KJwS|Nqec?Lv9%=lP`!$w(M5C_muw$F>q{y7(c<mYf?z9z#eiryHSMLvFP^u*9f_lcbWc}k'
            'Ha#=Nqu_viPEFLN-wM0&2O&5-e1jN%zSro3@+nFZ&29oj?m-)&!tpWBx->2z-)|TenjuaHOx6&;P(g7tyh$T750s>>^m`y$!G00f'
            '4meW0&^z93X9qyIkA~-_I!e-1EG2CypEXTp-vhm+umHND509x{mLn#Y_N3DKLCZw9dhONZF0WC^XwUo4q<5s5hhd_Q(fs)A=hILU'
            'o7Vf?yp&!s;Gya{dS;D|`_i3EQZF>$!2)GuyCdTvZYy@?9!kpTP|AchKU>+;jRY63*3|}^sZiLK;uqi(pc0UQ71W8dyuMhFP@KR{'
            '!d;0IN_%|<t))5YH>8mEwVVhdUqBy0=*q7K(gMcxcMQET&KO+r^F3+unu{@w%G4jW^uQ*dLA1tbfgUZ6#5JjPm3|DJy38T=Hp~A4'
            'GVJ%yPmhj{twz>SS;}6~G#6x$l&<oFH}up)C!e?)ai4*0CdVDqhR0Q*Ur|v>rHVBi)Mii=MUn|U1eiW&z>6688tehI1{_h*Q`n6r'
            'blj!mWIl!kLqEcJ2FOiEuB)V#VA`5l`N2`hE@QxpaVRcPK!LU?kVARqtYJXJW%Pnk&}s>)Di@qyKwJq>ka10$QU6?(fk#V>o*zk8'
            '$2Xe-RtbncrQWvGO|PcCk+8a(Yu<dVjyy#EQUos!9|z`fHBSmSWl5y7pnWag8AE%ATaA0j$2QJ|1U!q&H6BL@4b{|-=)7})!4#H&'
            '=|!=jawXcj>ny3HJ~PIiOJ`4b#3!<s4{SFGX24dkp@^!5YSIqH@fhea{x>+XCrzQ2WNJv3?9Ipme%ucQ|CZ431$Jr_lv%KormX;C'
            'bP<o{fea(MMv%k7(R_P&P0;9Bl$luOvlMn&>!6@hY9zPGRg|a=C0F@-;J}-`-5i!woZZ*Z7o^`!R}9F?WOTPhJ`6(sh6|m(Ut_3>'
            '57rV0rn9SB-Pb3r^lQkFIJ^C4sxX+QfKEm|MPL4$;|j|@-&ZRtPl}O6Fi}m2w7_^pCQ~4ljhkHuQ}9s?fx`><*#IxH8OZgK*w)6f'
            '2DNshndC)wNGN>FH={xuxjNpJXr*-C%*B^ILkBJeEUoNzJS|z-fhGbt(BH{=0@>Xw1x@~%<M~~^6aVY=%NKNhaOfC9@`ymVo{lMD'
            '9;M|*$Z$Afj@f_bKmP6CWpr9e%2k%0BJ^9*9`0z-Q1%R7_yd%DXWF;{U98g?a^|*4gmDYx6O?N5IW<J?Kp3m1g7jSFiaC9|kWd*T'
            'fQw0Mwg(Ox1oQC98rYf91hR*hCfglY)8D+{6`<ew9RI<<JHiF2+COiQb2mdC?pdnY-7I;>?vo=|+s|dYPpthW(HuKGp!mj~DhwS|'
            'Sj;nWnMBGWq*lGHFYu%D_aHtV#Pd0tsA&C#?BvE{1QTqFc_+70NH{HRbu8kNrXM9Yb%3%p2Ri6jvHHZuqFFD7c7WOoJw0R?lv1Xr'
            'ZQ_@qqLfyn^eWnvqzNb+#Q<)QN{Z0RmP-IBU=-P5lonsU5r{A(l9#l#VEqT;C2AnG;0@1prXxVe->sMB>>U17N*=?2SwjSjV`9yK'
            '44uV3rhUeVQjSZjOldVRAa@O&NE32I1iozrYoTX^$cOC;aUCdBABwZ<08wx{t~P7-!2$s+XFv2A8IqH%`d8@~$7*=v-8K}1!9Z<;'
            'KyDa7%|MA&Y-;AhTPpSjDFMd$qS(DI&<r25T@?DHZL)IhsS}6XuA-WV4@G<tzvqJ#?evR!6=hAN4cpWZN1&Z1h*ZZ0FQW2*#(9WO'
            'q|H6t(>}lCTzjYnz)Z8$K>%U{`<bA#sSJGv7TMJ)Fh$cF=mvtuhpMJ$j~dd;^Dv9pNg-M#o4mL^xidv;SEx=tn}}(a!=OPdhe=sV'
            'N!8>PP5$eHGLDMrqE|GC|F-<5_jY!tKWzT){ciYnfWLm%H%?syNekwM_nLRi*HCQ;|DFBsx7qnOzxyMWAMy_e6q`tH59%@L`&*zI'
            'O};f)rAi{7U!&9Jd$R}f{q5K<7d!HD6KxWaFY68AkHLW*-&ZGsTr%b^NDP-fH@auqA4>1s^nUl-uino3uu9h_;r(s@y?*Nb4nM%m'
            '81M#Yom%&f45ujC%m5U<oOC0y+;P9pEU9XncvYMiU0qTEfBlo2>&ppHnuHq(AN^}mJ!ys1$GEYG!`rq(b%Jf**(MyEqr2Ness=vr'
            'T}6rVT~=2yJSeeq<kaAKDNP~5ZVU+k&Ljcsi*Ll;2S-QZ4$ZEFj&C2}JC?j<Q+?Ta?RM0hCh{rlItt7#Y_}F0!u(^4@z}vxS2ghM'
            'XY8Pl3MS2#{ERh9@eWJA1Hz4bh&ooTP}8sd^FxuNTh0~kH_M5KiEP`EXP^&Xn7esC=GMu9KAZIsdJ(IiN^`GWXox^MtXAMPXs{g<'
            'uz+k5O6R7=5LM`6SE2WjIhelI%4;HlTnVu=5g<%iIzC5|TLKOP$r8CaWvEw|1PI3xqB4=b0AF!&5f>JGlaKfE@Ga3>`ww->PjZ37'
            'X4<+Hdo4vCX>}q}UKUE3NgXyt7(F|rv)r>a)HgE3X8-J!l<4d-2)9J(uyGqS-Es6w(=W%bfPqcxwa<ZYF}M#DBYNNQ88`mm$Rn-R'
            '#AKE+#+ZmQ0+D>9*db+wilqMn&MSJ*y38pt+!5~sT%mXtNZ*4<1Pm+SlF$Ttw*nci(Xfx7ut%~mV0JKzD->WEAz=Sa)0&*a*a=U`'
            'N4)p0B4*i!H*CMY<f(}(W$f3%ENpT^4swbt7Sj%vb8z#EDT%<y?%_!+Lno!EfSV5e>2q{tQ~OqJPv<D7T?&{=mOxfLB57(&T)a&V'
            'JGoOSs8CQK^qW#}@RX@3hXVB0W>$IrImih!n{1_ao~F5Z9Y}oTsg-duTS+46M5mYx+828U;In{yW!Y0WKm&jsLeVY?A{f~zJcVgI'
            '^a>3-J&YZtP<HX1INCE=Z*SX%iH<(2h_P!s{RDb>Fb~61uJf7jCqD3<Pc{T2Ja3v8sO~Ue7~3x^lSvvI-*}Fd2-AkX?YE<D<mxs+'
            '<c1o`QHIT)8LDNKK{Xa-Mr_un7&G;r6WO9%k_<Z2dgcxT*AZohh6&+boH;m!<>d3;t^Cn<v`3TCD%e`u-9@dX?k?(C?(V|v<!_gD'
            '=7&{O=aOB68ThhT(8vUc0v!I(sKL(8P07oUA4-y>$<(3V!saWvzQONO;FDym&kf1zlR+`}e7<+%!lGxb)$Kmfi5piL>sfv5TlB>O'
            '*&yRpD3A(TU(b%tc@Qj4^uvzNQo883F`&UM>e@WGzS_ROUKl~kH(+7!$_hMV@e9f5!g|j==zD^yZiVHRAY%$K^HpUbmR6@;HzE|2'
            ';g4EK*kuqlT49L<syEvzIBywodT#)Ch@%b->tMeVs_wg91e0T;L8}*gH}7l>n5^+tBmKLa8hVlF!tm^}YBCOz_Bp_fFwPLzqTXen'
            'cF!g59(k^jb+lBWNNpOy54R=~dtZx#FZTzay_T-|s348yh<mq4DX4g-RfHKf_P*9`KLJQizq}Z~I(_l<^p#k{u8x7H`?B3;n-8Jy'
            'xGdQA3tY%*twdt4b891+!+Q3B!wy?=v?0(JG3Hd6k@X+A;8i|T<w9c6xv(krmrY4sIRtrKl}yv5RN4|Uul1<X0lmS;KFasUM@L6C'
            'wg91N^a8`9li7B7Z7KzfR|JKAhu7TT@aQh4RWKcrr+|=SRMP?~lQVDGQxDQT1R2e9mQW}+kP0@qcS1ATl+mI|1K<qx4^aVhB|t_Z'
            '3{YPgNX4}1%?V!)Ktw3l$lF&N5R?bvHTt&72#;(dBsf7%mNH65_lDhqS9Mw?Yg8`7;-l9Q$_9YFD67B#wJ3}~8XX%QZWM7ObeG+r'
            '4)}1!y*^iN7pDIfxuanQm%W|t6IM8hq#-<+c0KH?)=lxu4Y>>a;~WiR#O2dVV$Lwl4fbzH9ohDG_k_|u$zYWMIT0HJ0-$4q;Ns3`'
            '+5z0Odjf*1$zs;oGUR{8DY<B!VHvx?m@2Djm5}yqFIHYwZe33r+MV}|L6!9iBhh8iBg>UXWip|mx;{M62{ps`z2E*!y}!fU1{D!^'
            '#IZZ6X8SwKlE;sN`d#PBv#}&Xvw;Ks(3}>J(-6XD!N8nT=!dnK-w^b6buYphOQrEmH`Jh8AOiMz;LIKl=C#*cgYcKzg*<8&Z?j!|'
            'z4z@FfA=MsMbMH?a!4_pEIoW_6Gg4=Y-|K<8nkFkGJ78)L`2FsgS9OvA4*L}1@?1vY;ZjUKn;uZz_a&;p5FRA-@o>rd;nXcLVpX#'
            '?RR2FL7JHUcE6($D#IkOql9wsYcvfq5F2atg14ERX{rQ!nVhMU>|zwu)%@ULplseMKjtK+&_o?6*o1*dp@4OAN=M`5&@#@d@fu@u'
            '>bN<xZt(dO@+=x-s832d3=LRircI;lqK+=)2g||!Cg@3)R2b~dsXvXhU;6BPqo<%0zEQKz&N(=V_8jO{<&!f_E;970hvSisdL4Ml'
            '`$oLX=ya8~anX}HXrVGcPd7Xo6e?;=#z=uYBCe)`z7teS=LjWSw3dX<W@Qo8C-Q-z9)pcniU^(IO6iCvM;w%_=a@CDH?Vr-QQj!Q'
            'Vf#oQfkNs1<))G&9do#}5&mhpeWG!iXL77&Qik;Y%0GWXBxv$f5Kh2)nO}Qtgu&Dxm&r691=Dp+nci0G8aD%oJEH*R<Y8bPxx{#k'
            'F~3qM#-zI(aZ`h4KiKg`NELj@0b7f*U1>~EnQbnknG8A`*;yLpaW*3_(lI5ag}&!?NeeiVN(+VYBlQr93CPJb+R=R*nvLS4p8Xru'
            'yyO<^%OIxa0;#9L{tO&iKCn)qazJd4p{X!u^xS3E+KgVO_21rOle}d5w8Z?q=+p_AIjh^A^f&U=9{NNU3#+{Y+P_#nxSh_;%}-(X'
            'evy2L+@v6)M@bp_kL>NKHQ&oRh4oI-Xc{dC#yC5aj%1MT$VbqvHE7V7^_=66RCIc`K}QQM4Tar!TM^=FKM;G5EnnT;RxfYU2%yD|'
            '_iVx`lmAKn9Dkk#J<c30R&kXjVv{fX2J0-D9#pxH!~+ZbSdv-TRMgs8GE9IyY-*e{lY)Dvz!;t$$ij-D#nZYe**C&p5OFCt+{ib~'
            'L(RS$U2wQ~DT`S%)@=I(cvjtr-uyl6`f=a+*1^%)fr&WAu*qlFFk)s8b(M2>Tl2{hc(+R6=U--<ELtT~mB+5^-G+I&frC(4AU~PG'
            'xh<P>D=?qI3f-0=Vv{8cb<MdZ)|jdp-m#IKa0ypfPPtAH%K9n{7EzYOm_Z>I&sigo0l}nO2lRfCTr5&lxzYs_FRcb5V3Qadq#ZFT'
            'TWMJ}^(1>Ffz*_oqUAE}5@4JA{*G9aze|SXyhXE(S5lM2i|nu<PI5N+sV%oE^&@>}o~N(rp8XNsS#H>0nmv+LJh~xMSE51l;PAjX'
            '_hygM=~J*EmDY!qqaE{T$%Jf?M`wn0(%V=%?^^zLH-;rK))9xud;`x_(@sA$U*X*O`iOOZJ7P%w)}@w(x_(E%DgLmX7!7AnFRd4f'
            'UT;?W%AVhCmYUxak9)Ucl}&}8v`$Buw4&PR`o@Tj&p6e{zl7qR2lTf8pU5rC<NuSmWw)CiO`klqQH>-|Zil~?@QpLvsY*b*YLmeI'
            'nJhXUEvp3h>^Q}%es;6yv-*R0{pyXlkvo?Xuu3r3sZz0KtCKPD%)X5`?K5WNId=AZOoKtZ9U5)_^ik&4^74RHj^wB;Z)pdT%2imz'
            '#4!nlhH=I=4Y6USf1`=04V$e+Eqjrtu4Ob*70{v+3a7K}3dgE^V>6;oZim;hNogfQc=R~9Hi6Qsz?@4?y4BOHsy&A)Yzs~WQr5dk'
            'wf^7#{Xc>6@*3au%r{$n(>LF|rOf+6YX-?*Rwsm}t;zC9+ejr}OMbZfm~0ue?jG&(!#go=@^Sx6ia1)1FQT>U+V|joV<c-j+L+3I'
            'kK<ChI>}j(MXOxo1-Z_$C+vCwhhh^$q1fALgy)T7A=#-IcSJ#Homq=+D55(oBdA^HDsG53@L}XFDbIZ^V4VSDvr<Z77OgNh99p^='
            '5E>vz+*F0m`@5Y5xGZa;#esY&ft?+lH`;6q49vI#K^1GkI5<96@SR9o$zQ^^5?ur;OiqO63bUfL02V6p8VjQ)w$}>=h^b%UU%$h@'
            'hWOW`z)3gC5qV?66Md0|MHDfG0k_zfcB0-flOT=M{Wjx*k?UB~S}Amk!TJxI`h@8I`g$q8lcw<dieGX!gTFqm!FS4xeyr+RJHlVl'
            'Kl2^8E=W8z=dsPD@djw@7CfCEt5>r>_MDe(<DhSM>}W_5LoQ*NkmNl+BoF~n_0carfic}r8f`G?`uM$~PRB<Nk9P6=y2eJ=_{|A='
            'HK;tSTDbOKZrA>!FC<a`zdb<$$u|M%l!GTbd*va>Uc$QrgnRvw9*6f1BoA%2E=96xw{Avi|Irs5ar!moShcH)eescb8hG5ee?`;~'
            'k6*t=k75)xIZA;-3@9|1aSPL|uJff*DPVHkIHjZ>1n4xvpc({#e$jD6IU8d>M6APtTy;2A4_jRL8c<4^Sr;OoOHP(k^B@coA=m>u'
            'y3CUqSKC*cSDslwb~lagB%bYycs`G(9KM{-8%@nJDk8e>GrHh>o)@fw5l^D^le#*fK*?2piN+*edG3LwE8)$BnK-)G0)rJ8=_dpA'
            'Mh#Ik-IE!x8|-By2MnG{rX5!-f%R3~`v>6MGck)+7s$YG@}-LGT5t*$Q_&fmy=GPJvuvK9n^!6Ed-RJNipzV0qoX^<*Lm<W?D1D|'
            '`7c934n7T$=x=@H{^XY1SmmI#D2->|!_>sZ<81bN5zX?eAz+qu>E*9Qw0@1AIYV(`zBixG{&??>phBfJo*zh=J`^t^yt@?dy~y+v'
            '>!W@y5+Es?Cxm~YcugFr@#1A&A%d!(jg%IA4>F2+{e7N-xcm&*?PMB?Uyu#PS^y8u>tzf;;1NgrFY@0%!N0seL-81MTG^m!N@zT~'
            '2T+0L2g)n4FT`8pfEqqm53kh2A472}ZQRzv$a#EBi2W<H5?*WiybWowG}Z0H3!8Sq3zryf!@vN~6cPV~|NP^(59Q;*ad<@k1G#|y'
            'yXGU>rc?dl#TLti_u;=23oINw@PBqQljI*dcif2#r$ouTi10kxpoH?8wH~La<dD<MD6(rpiJ*kyO{}jC<I`IbPRr$j2?NSQ??e7!'
            '>->46_w5`&vbb?hdn&n?ElB4Uo>HS~2gjPsxfwKLh~E<P<^#)??)TN01~;}_p$cajOc!$uglTS0gxh5K(QTGz4!HAf*1UaZQwMx2'
            '3{=r#x0vCi+XnizctK79fY744%5CWgjy8%j87SSjl7ayDFX1TnK*KEPoJDpmN;=Te%_HOXoIhj8oso<Pd&^kp0S0n;ekl*zd4vp)'
            'Iyw(HvA}d%uj-Up@G^!I9<NNptAqx7ehnlh67ry%l$}^J8MdeJ06!dmqVChHdG#-`sf6wBprfqgD7CRiwQkC;YD3>{Yia|iqqWw~'
            'Oao%>dK8<i`8<wIZk27ToTIs<5hF`SqFeONbTbO{@}~#?gxgc1B#ln4a~@n@zkNGxpAY~1_U+_&yS>)k-3B)TaZVix!e6gFN<UU~'
            'k4g8;p&x|0uHwN4;i=I*RrCF)NtEUnXe@=(%+(Q+rC{57kLH4H=(|zSIK<`o8eTV@_Ff-&lYgA#74OKQ^eL4*YQowMbN-v`BtML4'
            'oPwcA>Cw7CeKWtwRywY8r630V0wJ_+boKHP_H}i}y&Rr91{NQEN8S83*VD~!x|wd#eJS0-ypQf|c>Lhpxoks*K(NBOx5%sUDrajS'
            'K#bI-bY+}!01toEuc?*{iK06g@SrJ@S?sfY|D0enue`<r)6<}0Fpih3B2)=eaJy2j?V)vBvH~3H*dY*Su8gKbei>y^x+#-#j3CSp'
            'd|NWvHz=>=o>rH|SA5@A6K_9V<Yk=6+D1guigI*epOr^0Ju9tJs4iJkm!kJ|#QhI2-WPfG1M1&(60MBPg>#uSOX|EzzPL@uu%B5{'
            'F2%tCC$(DfQ4R9tG>xNznY)~ldUHO`P&yPwGP2lAuEpGBG$|5FP;9s#XqBs6&sAJSh{_H!$e2C8yoKC&WE8Kx5O_8<aJZ6PLT>o>'
            'Qu(U3OEp<WU<oLuY2+fP+10TwC))*#zeO!sO7)d>+T$DBY!!3>+|amFOjf3po_N`W%A5%#q7lG70}Y$W0gBNYQ>f;^pq9R_hwX6_'
            'IJcLfJ3I;ZeCXOXlaemP$$-|-NJ0#*ucVr_-@oA+-T6L9)JdA4swEP^HO<L30Pbazlu-&>?;=Tu;+jBzRcr?)4i7Lx8Jq(j>ZL%2'
            '6yOwteM@1|Q>*ixF@HPsC-Z%KV$s0KPusp|ElLI4aPM0G-eTIQ4RnTj?e@gQm+w4Fv9o6`fZnz92HJ10IQOyjzTH>V+3q8N^kKv!'
            'beHM-ZvERueH#klfkv5Yuhf-&n4-rfOu9q%`%meFd-gaDdq|sZy#4J{I^a9!Y+7k_$4hF=voGiyt7R8HvkRVo(vs~4NlqX1!gt*w'
            '<96!+wOQieMVlYU{M<l>2{e$m$g|`FUT-Isz_2BB(pD<pBT$D~G7z?u5bQv9+!*CD&XOz<m@!#In+QbiL138sn`EfY1dSMfjLc45'
            'EQWH~!A+^DIL5N6mqu*2W<w~VJ1>$yQ`Y-sw8^3+hlmSzD**8{)^aP(r0a!2whlCnE6>fn<{6ZYkGC6Mi}th*7Z`4<P{**{i%qaY'
            '$p^XMJs);_h>!$cx7ZLWVjSw*GQivNmPvuP`XwdIoO5+Z;LM~li+iZ+L5nu4scob!dsFP1K??u<^f{Xc?Y+@6H>{c@;2m*;HFE$6'
            'xnz$1PcQ%AqEcXG^7>Q`+w7}x_lSL6j`j=Ek=;Ebr<q^E;df)ERFuh`-<##Bw}z9dT|*Cs1gcq5=x2TXnebnc)xhNB??zT8Whzx)'
            'jMPlZCc0<Y{$Y){IceWx2UR^Gb7`t*?pSN;>q4$o^fyPL`SBLc+Pd^dcxLN5wi`C3;;8b#P94dK-N^i)3K{ljwkCS2(DmOAT>a6p'
            'EqC~TTcw++yrHz;?0Z8cpRM<XDvpKPE^U=EY}PlzG|{1>@NHB9&b>droYmwuM#<?V%b<M!c(<`81vBWreD&=2&t5!!&dz?sy=ZSh'
            'GtX91D)Aaf8s6WBeq{%WE;RQ)YLo6vs?|sbQ4Yj2R)f#-n3eXF-5Mz5^XEUI2>j;FYaA7hyBLr|zT0!UHw>@*tSe6RAIc2GadXE8'
            '4<acI2S=>m1UVY90_?ZXS8L0pb9<&fk{aZ$=$mdBzD)PazI9TE-}-FZra3PehcC(7YwVkBbg~grvW#EUY5KGL+>;y+x&IH8$3J6;'
            'I(nrfuW2Z0F-o1_Mim?AOm{BT?RC=Qutn>UApfOKig+wZr7vN&+n1d})=!L;aF`~mgs#plIA694WvfIc%?iUpFDd@O?zPlG*bFlj'
            'mbw5npvgvoxK>;>l{z1)y+TQ^bLGYFPR3baya~>?_V`r?<&7IoqdVrrVYo<Mk0}&uIl8uBZ`;B-x*!1SxXG99-O4c4@g?pcteHjy'
            'PBGn4)pNSm-@4bVIdiG!Tvs`>x^=#1>?XRiTjrc=-t?UkgVhrL?yV#AZBa2Hq{SQOOXKPm-c0Dme-V@XMH$5tO7tH5J^>1p0`-$u'
            'o@bbYS?Pa1F-Gx|xhzS|ZN|{AGVbK8T3X{>`eJhv(jh%Y8=dQ?X=c7Pm9%}%G2QO9q2HncJ~>#C+;tVG`+aUv3vcdJL6o|NG^|<N'
            '2D+-$X=Tix^|@#L3hlw$;N)fG^70M00ZQYB-vga;c?TD!HumdZzJK%V&GXaouRlI}{qyOo@lP+Go<4tl>kS^@z29_wDBm!u4=iP#'
            'rY0#>ba!XIDz=XtQF|UVmy)WT-qF|oHfUX}_?H^HH5<2hN@+N4zsF@Heo%-0sB6|6seK1j#Z3;)+-i(9PSbv^dV_?zyv8kWhja@0'
            '%Z_2T+n2Fn-3<3w$5}Ya6)h<Hx=9(NTaEGkFKVrEZpK`QLXS<g!2mC~r##ZVA~=f<KAc~lpbPzu`R)&IHwQaF+-4R6Wqs;aOLc9G'
            '{dRX4LBwUeiA1tY1n}SUG!YXxVbo@U0|4)ksYMPG0^i&-u$LEEyc}X|6&@KCoDYvsuFs-ublqwZwv~GpHtQz|3dRgFUOH69Ma!mq'
            '3orMqb<k3jL8qE`XymjTIt%pI;M@aBMuDTdR!j8i28UmjpM!>H3Sma?T-nSE)h*=qwUN~f$hH9Ig=-gap=xo?nStiL52pUwnt>+K'
            'SQ{R{?}A5PJ@8mGY=M)n{m}Sh3s~c{GGcSR7bP2`e#p+teFuzfz}L<@USN(7KuBKA_NhzggqaL+qr@3rZEBz3RHElQl1*ihI244L'
            'Wzgt>Erm^!gn=&VK@kPo4AswRG1t#@%Q(^}XrqWQmJfQ4@jgd``yNX55dce4n~39w@Sn`<AcvqXRKUQ96OP0%l~v*!3szqo9*Prq'
            'd1CR(`dWoH6#1ze%wFUDZkUVh3D%zntM$FW<he10V=qwBh?E4DahaqBo9(^DI5b_I59snwW(pPa+&8Z|v6<4~pfZaq{@V0WIaoz&'
            'q~%A$%i$5-n4(`q!=vAsUZ)4sA>AdFCX@Fk5AIQ6UQ~R<4BiqIpkzUpNS!dmb@F~ZU(uIMeR=QXXzhTF@KY2`WoPYKR=lmDz2Qj>'
            'J(tvu?146mrt_hc@y8}bak|5<Hmy6-$tJexNhQXvoW302E-yvSn(ldW68CR?FS{p*SYewqxGZ4=6{?3y5iJvTtA8YvS>}M=r`0C<'
            'kRa1sqoOG(qgC`_ph$+DRb?b@EzdY7wJC*ar5U2YOJyS}k!#9yOt0*h+$L5iv6Gyt9i!Vy8?tAIvNWa_Csemv<J=&xCti1loYB<N'
            'LA%}TW4PCma>2(FIXWcL6JmP3o%rF-wpxA6tp}$}m7oEpxpq#k6eYd$(!WLn3PqUy+4=rL4-^Y4_|8NQ-=pMq#v|X8sBpm(Gt^v_'
            'c&)2#^{`4H2}Ob)5H`we`wG8-gjJ}c9bXjDdZEMV^=nSCK_7dDX(hDIsL(!a+_^)4$lY_NFm{0a80&s*6juIG<`p`YO@e7n+a5t$'
            'YA0H?1GYcLZm*S{A@mH*UYRDWW@H}NMo`{nWSzm`UMp-4ne$Pb!w;LFx>1{Hx8W$RZ`9%)&ETds<Lg3E5l?NF)-LJt>m#oeVArvE'
            '@sWqOx65l*bkycyog@JkpE>H_RlP}D>Itu^^vK3jlb`r4l33msy*4HoS<Z6OHzb8Cl@VvFa)rq<fv4U$@3N^6sSFeu1=k%t(@^@`'
            'V4rfa>+YEM#I{DxWI|Dc`TiMQkvu!c)d{X!Q?m`O-TrQecx55ZIZEGWmw8>C@3))aG-+fLRGZ%1oU{RTJfL~cgkzQel*O{?um&S3'
            'i96i^2AjRc9e3R_>TW|T{%q6)g@#t(*~o~ja@B4|u-NFXsx~?aW#E$0s@lKXk6b)Ga{m$nI<eav_CS`PAKTy-%csx1oy*tUgTbpY'
            'p3=p<?-9ZyrEIoOqtF@Bx#G!)8{}II{Raz|py1V(SR$D)W_^JJn+H!5yy$!x&5G#H;*LH%t9Xg#r34rSaxyd|Iyyh;7-9n(av11a'
            'PHz8=qbyQB96P-p;9tF4OFc=4U;v-&3B(~-FFcYCuVPEoxwWL@h_g!Ap-W4&Xxsl-xd%FvU(VC$f*9dDC`rj&?*zHE%%3e4NhC?p'
            'm1eq;KEdYVD6cMBxIGWnNSmm%A3aHy72!#ikbi3dB8SJ6B%5vOqOH}rM(n2Cm|+|1ck*C{&x~^$#prn8NmDZ7RPCl$(>LNI6vucC'
            'xyfp`26Lt+g(i(}#(*C*t66?U_nChHwrx;XQ*>m^nR4_x!Qll^HD4$dNNtU@A<qIf3bd-o`!CMr(S{j`;pgrK#?qX0ZV~)IuI3Wx'
            'dy(TU+DHSc<dJmVyNoq`NU}1*Si5PALIWmUikGa~Fg>tu>TT09YNbh2J&P6*NM;|>NZ}UWW{cvg>{o4Mc*@gR9qbwIjyB4!02k*m'
            'D=DP2hkAty@efI1V{5cV-wI4Sq>|{N=F$ieYB^zX4Z44%!?eyTDis2%bZAccbV05n#ypMnr6xP3P=yMysE(^{cl(Sq;HZ3H<~gTO'
            'AI}g6n0}_@NIn!L^X0j5*6;wn3_{Y$*kn{ooE)Y`+BNAi+@xmf_uEN$Ok6WO{W}0SzW+uHbOdKFT4J2;ZLHlPGtSkkUV9XMICyos'
            'R{IMki`#Q!80{&r?{nqSAiv=X%@fQeE_+6_siRM3uYZe*+50qI=4&}?PX8RS`b&<ZBg{_jh@pjFq5EBtsG(~d%ndaLkOW{SX0<_@'
            ';3Q&_0mS4YU)4Y$9{I^toY@5HV~Lj48%v)i^QI~aCXdBcG=ol{(+;Xx3@+9fJ8VCY>F`>6+WmHR&mhU*{KJ`Z@Ah+?M)|thp8MKK'
            'S{QgeN@ZQ|Hy41{ZD?q_^n_lQcE>q{2%YUN;;m-5T3|b^m&Tof#<VGt*PghB2EsQFWpsMcqOyp@?mK@UXfI#1d!Rf6)ieYhuElP$'
            'p}<f`HOGW!=1ijxUXo?}bV{1)8cdIAYHz$H_3hC+^f`gar!^h*LA7|erl=R{4y`L9#%*-6HXLn@*EW3WL8m?c7I0YSqEP$tlc31x'
            '$6=@2Ko<mCi9D{En;<i%PoeDYY)ik0*TR^;JIX3kWHWz$zoBMU>AhlPI^pSg>;fle_twyu%INcD2Ux4*ZphNki`wHdvEXV;v|FSL'
            'VB?@L^!PdKswDCMCgrGINmBh?YC0LTwX<o3sa<eXAa4hdmczqP+(LyK_+1~&%3a#jpyk|P*6L6Q2Q3c<w~|LU9Jqc8b{(mf_E#};'
            '3f!di)s<SQ25wXUD_+JtdPw6@TzVK0J_LZPsLk71?a!9l1j^WI6mMBntPD+m0q;rLoQSQ%v`sC&L~>fkRwf6wH`!(QS_7kyywAaD'
            'u&7L?$)jX-Dkx2PtAbXcn~s#tYvZzL&1JjH3ExnKOG1iYK0eLK;1W(XztR$K%JhD4$75KGXpbQaqs&j}sOMLSc8$Sjoie&&zIHp('
            'A;KYF*YU}L&?#{t0(5<AF^I~f+!)1&j-lC$)U4WSV>F9MMk<5hPW7O-o=hTg`~WK(2}RmK$W-FKy#?wSnn91o!q4Uzib$Pyz5OW^'
            'Cw>2nE$UUI)Gt2P-cZb_%cIq+@nj=!dN*H2m3nCdLRX&BQ=WXL@!+E=ZPm4yf8Zc0)x_--gmxYn)R{Y%$KISfh4HWT`Z7<ZG5W#F'
            'FWtE~Td{~PZ1=4O?Xe4cav6PG&|-J6Vs|^;QH;By7iaZ!38&*Yrc5U@`)hY&&hbfn$0hKP7~t1rF}-`=ymra`j{52D7O-PXcbXax'
            'cSl?9jI!)*=~m&Fw+q7SZsdmOydPoF4$}A%(~#iZdTD$B`#|CgunASM5$ha#qc2&esyi=QJ~|q2fVkC@n4H8{kz~y9egZBA9LReh'
            '-&1*qt`!`p6Z#mjcA>iOIWC)nn}g7>E^PgD&$`Y^-33_0WT7lEs7@LG#2qoAoInkvJ7Sf*N97X7>~WeNjDTZn30->)n?q@l(~Y*T'
            'aL65#@{mTaZu+xh_{K^n&QVcF*l_%`8Zx$#Qv|YRikx=W^(0No1@<f3RFv4SXDY8K3`t%v*q1;aOJc2C7;>7Sd7Y7tNj!P{;>F81'
            'h!=qAm_`m2fun|$V`8GDf?d6gd2zB_Yi$IecTHYt;xWvATrOZHbc`Y!Ffa=*UwPQlWfQm_y01xnHD5ubr&SmhEG&Tv(MddwfZc*#'
            'O!C<V{q@8Z#uY$&2}f~mK==_q11jWH5f<7Q%z|>Qm7J|Ongaq}QOd>3B%;di(j2V<8j00N<pUrv;LIGRvDbeW$~;$cJiPL+e|&rZ'
            'N0Z7+jIVeeI2#uHq(c;aS875WbkTv%KYsC4M1)tktiCBQ?t`k=(zQ`91W;DP4xuhyQ{xVWJ|&uW?2e`0A7D~5K_MD)Bn+cZv8ZL0'
            'q^XE8=MLqSV!WQBAHhu2xZXT_db*m<yc37U`<CYJ9a1#8;VD!uwv0m1a=b<qR2R1M?7@p^M9C+0=I5(LL`gSc>!C?vMmK1gVXj|`'
            'muq>$c0M5?tKd*$X}IbiO3%3d0N4t?4ZfrmK?4o#Hp>1SR}!3R8M~#EOHblOgipmIj!A|g=4nh|C}}kA!6n`t-URi2t;j3PIA+tt'
            'k&zjRW(E6(y7VfJW|Ws2F}GagSJWX<u9FNX;oyr!^8U9Eiex_t)Y@xKbqt$4gZVNNb(&GIjOmW)@GH}FXymx^18%N2cA*k^S5)<S'
            'XPetV^i-3ag{i;aZ}nDh*)QGU25?88J6iU4+dhBI`+!HCV`&NnmY=N`4BA-X8=<K8PEB7Dqpn0l7{y$sFd<0+8*i4E8@knq@&v7-'
            '83s4{+Q9r{t*CEln30#y`B&U7Yh~K=8F<?Hl?-ooBXGQ(2a2n=?kH<Tw)yc^25Y<h9Z-`L!@b`fv2FybbDa7og{Ygu_}oZ!?--r1'
            'b)&t%R`mMdGs4%6?#U>2y7Fk1lpJ0;Phw0gU%{bMpkaPQDU_QjihzaMg-ukyA9~Y?@i)^T9ho=cP5f8kNn>>5k7vA$CZ$RY-Q0gO'
            '>Em@UdIWn0g6VOx6T_VFYEIVy`Vu6z@;qXnjk)UUv!DfjcSb-D9tE>#nNXGp%3Ww)uK?JPcu)!0f_hVzs~XO`p~~=JbfF<F|NUb8'
            '{}KN7Hwu6ILlcN3rpXd8?;p``e`nEew+MW@Ip&Se9o8tKid@~bSXwb>7SGZYW`5;MN2Wc_FyYY(9d?vc4~Iw@2z<+cwC2)CAJ-Oo'
            '2fiyB?H|!uUm!Z`bHgS&LMHx-;S!$_Dq&;HUoAv}k=kA@erGOKo0hZ`?xCUR4tJ>4+luu5=LK1W<{18yBQEZcKjUY`T~tS3KHfro'
            '@=CSlx%MRjGtMG@^ItSF!^+b1w~5fGj{Xsu@jo~+Lr(o6FULW1owbhGNH}Ce^tka+g^JHdK#j$-7q1_`5HDXm{?jw@^z`@7o<Du|'
            '%DT6WG)=(kr65UvK=0_a$g^kzScA!dFi!J}$9CK>p1)$Ghil0ulSAt|m0NP3$XGJP>8UGb`t6HQSik;@vX|Iq5@eMl@%-_d(^uNy'
            'Fwo8t+UWr45};ctfnlz*3E3XegfuDXu9QgW@JV@}i!{$J8e<FQ0FBc*UR+Do9ZFW3rV?O<mJ4ZrLn{{L^~7sVC<&-EvN%Q($3W7P'
            '<`D^zjU;%ZjA~A;OJCbfl)Z#aJ1wipjZryrK@69V-TL&dP@Kkp!3U~K*?luK#Hz8>q>Qy&kgXBC1*REB6Fc@ClhJndQfi_aEax8k'
            '@ZTGe@Lw}L$<)a2=!0?HkrT^g5@Ywfq`_Fsjo@eq(jy{{z{jmIx5_MrEyg<gByGYwY|YS3@l-c-rswmc!mQ}gD#h)BK`^cU$^bql'
            '{?A@KIemWm)9LGH;``G-Jb(75m#^F_2YMVkQ_~SJ15N^yEaq7e@o=L}!71cmJA-r_icX_6=@bDC@!FHVFJv2i&0MswwY9!!xPuAf'
            '2jV4mCeNlA%L7s-olzU0Cd<66Q`UI5fIBJSKO-W^)3Lu*43EzvOF5T%K6O^NUAJ5a_ScUM1G4i4|BV8In#pe)AcJ9-Y4X2cKv8#G'
            '(3gq@`f~9<9~blEXbI4j($O32bZ0jd2T*l~R>5)qbW{vb*M0R-p6{RI?cVL+-sRMO8yEGzA3x;Ja4r5QKj#0De#p}l-H$9k9qp?5'
            'B0tF2+LcMWQDxF~i7zq0MU;2l1DfVFI?B;ifZBA-vK)#hp;*S7Q0tIEz(V&>GU$rIV4!QThMH@zrm2c&QsEMcZ<LxkxCJP(SurG8'
            'mT=rD{r>@s!cBh'
        ),
    },
    'google_ai': {
        'sha256': '6a2cc8b2916cf3485b8fb85ca1b4b4ad217c00dec1f962cecdc219a2c7bed49e',
        'encoding': 'zlib+b85',
        'data': (
            'c-ri}d4JnDmMHqaKLwg~-&m}YWM`>#x6|r+DqBfu;w59tRbB1!P?SX3Y)Pb=r0ghe-_QQe0)PM)SxF`L&D`JAZ>A%Y00<l$oDFAj'
            '9OuxF;wT*Z^QOrBbk_3k{Ui`?<M?JCh;Cmr-Uf>>3dNp)9}C|VNxaH}L`>sEWVc}|mP!1NV4Ssgc7FNkS26ghe<4oJjxLXTqW8D{'
            '#o%J6Rr%k}j{yu3@ED7DvVzYAcR{iiQJe*%IG%~oDx7B`%u*5GM`CyrM1I&loQFY_HQMdAI~0C2*}3(T$vB<_6ETEgf_Z26hGyJ)'
            '(SFsM&i(XucPPSW9!5c13~v2QEc{6T!-{aRjMFq6!I!gGG+=GBD8AphzYWK?g2qZ=dV!cGL7<i?#(pIHakld3^R-9<e|*~%jS+MW'
            'f~9WV%EBPoIe&k$H-P^?e}4pPncM^!yo?uO6ok<YJV_#$^k^+;#Bn%*b@_8K^`jPmxQUAsjj;<4!Z|EJ_&duan1tiZhZ)1`)g+{!'
            ';@)55%5pe{Jnys9o><QPF#_4o4t92gc-9tUm~<AT!WUx#rD&u9Jj+^PGzpeLgwtpW^>e(2=eY1-;zIxDZQKhUo*)DuxCzq?w(UDH'
            'j-zRKvw}(HKoI|2#hDMoP5sq8OJRr?Z7~VcaSVH%0yLNYD4d7TG!=KDFNQp+HiN%Chp`*H(EtL#JD6)mpI?akFuR5E;6DIh@L@R*'
            '$FQleFY!HggI6!xVi}HSxYkJkoe?e4EF3cqizQ4CaUNQJ-=+=3mhd!%Rs@CK#h`!E9rTXmGUfs7Vgl`7wPE2|g0tQeBOkvJPp9x3'
            'p3x}dFd464IwOGgjJo`>E$;kzI3eHz#8!Yr2r8J$a+Qg64ZA<i@FW6abx)5(8Z7)sK7jszYKwV%lgdK^=y(ShlL$Y{g2ggJ=)|-w'
            'ev*Y#7zw)kxh+yZ0^oiORKFle;NXt<?BN-}1B{S`i6naMlO6GFzl}hRQ$m$ASo#Ufyr}^VlZ^ZYbQ{M&7GQf2nv!hjU0v_&oXXP-'
            '$hk^0Jlp`e7972Kut4ndqcwnzXFdX?KuiwSbT*UBqz*t}e(s(3-}GTOx^KJv(~E&Pd-JA$*zX>Tw`XT>f#LwYIyyVNJn5Ydx`Y1N'
            'DdKh@&U@XXlb&$i`RmA^iAj8utmgj0cibA<^!fkow8RM=i~doHkO}z2-)7k|J=og=`gKDfYEOc@JxQnhWw<wq$LSv9JPc7d1VH!#'
            '1il9c4DqtTbPh;hQ^-fD{tHfEhlpbn5TGznuowjsJ&GowwTF(#a{%MO^j5fAu-!cW77=5St`b1C0FQNJ%Qc2KxD>YyAEn6-fgG;^'
            'A@StReIRp4;b1(#MxX}SdKskcZx3bDc!Z?rwZ$}+eu_AH)IS_th%`o|PBVX<N)$p0xg%(qi+DU^f(euZjvXQ0#nC%BJJ4eehr5ly'
            'hry#6P-t%sM`k`m+8iu_=s38Ng+F)1zk~xshW>>U5Uv*V^ESM>bwmTsL5l$C;tp&F74=E@{NdU8dGC;r_bzSI6cy_k0lk?Xzy#sI'
            'BSHZA7&hV0c<C+Gj~O#X+<N$zs10oRD#~~}PtOK${1$M|CXAUle)oNU55UkCXVWPhZQwySE9e;k;{Y-NV@VO<1u>QwsGxrbn32x{'
            'h!C-9N#%jd?Ar_pzLF5hfx%Kd`)hI9gVQW7dZ$MhOwLiDP#<c4Q2dt;@X$k1VB(nv0r7z#%<LNYa0uJQoQVLnn-a$W5~H>o5e0}N'
            '@LnI7>O2jQ%O%0s&%!urZwGt1bI;q~vux1<Di;h9FbLW?5l{FKFl(Wx;YHvlFvA;QI<O<~b!MDf9vFGvQW}I7%s4K;2nPnI$%6-z'
            'RuXVaW^tdkC?XFy!`Aa59QlC1kgk?^ju54gS$cRRhp&lo(!oJS2JlOPGN{?W_{fs7;6t`OWzb(wUE%@!1=ycy_yJZ61Ho3s2?~1h'
            'Gm6vEGy<Pa?;9<6n9^6Yg26%DP)A_78o^fG=0o8OUh7|C83XD7s>4TtrCeknYQ;oCfKGtmFb@}q?l_v`FM%Hh%>E=?o5bbt0h}z5'
            'E7lJ6MIj6ZbZ|l>PNV{NE5p;X2qzJck4(Jx#~~a-k%f!E*&cX!Dq1`(fL;$-J<^mLSRZa22;{wE@4WXG1oHD=hf+Y~BMFD$@a*(W'
            '|Lx^@_ebENN4+=Q%i{qo0wo3#PFHY#65!^*g6Y^Eh}Ci)`;)Y(A&Eaxj$mN{<;f=en*cE(;Uj+tr#y)fis{ZM$nJw6Qs)HdEE1}8'
            'PlAzX%_Ihi3%V1)+B=Tp>=51Zyy*%_kLM{-!H*&w7g2F%M?M?*Y4GZ$`s)_X>pUE(KmU=&k@_8{>Q@3=6)*H(f%?UWuKvR9;fWxH'
            '#lO@vCL|@Pr%h@J$364;=f^mb9hZKFvr+xt!=F1l--s_MjC>=00_6y>86>u~NP!NHVW;SHr2(NY%q$!cO$Cz7#JWXn4@C(;cGwQW'
            'W*BO^is+z2zZ((;;7uA6R*UI8TrLsaV0DOGKFIQ{Sf%kQ0X{Om^`jfsLrA3f2K5HO{uBpHK~N4zu>xY>{=zWW`Q|{J6JnswWn*0T'
            '@bK*Nba1iv{`~CUdni8u#>q2TQej|O^aBSZ>3(qsKtRqR`b3UF0br$V$c)B|Vo&0<M?JknA}9)1aQIdzw*ZlW0h3iE!YK;F2tfoV'
            'aIdT_4iRoRb2`BnNa;Z8Cos=J@dAuOdZg%i1GqH|Huq6M=k3P6uqaS~!s!%FJdib5?9ct9-nomDo`PD;+Eogily^s|_{aj<JKf{s'
            'vtK+3D(^oo&jwxZ{7lN%9q|SkDM2|Wz3FPS2x)IqCEdYVN!OK-hbaNxgAPNoK}e)?vcfY@f||d^0ihdcPt@&p1v&=&br%NrNXVZi'
            'i>U5tT8N`*Zea?K_dwPHR6`Te<>(cldk82C{YJ1`i}uc&^Ip#z^n2&tQ5O~u45WY9gQYh}`*ftxg0y~xG%gK4ikY9mgb#ox0#{DL'
            'Ip7?OnS?VKRsh+`=+PRLv$?PJCWi;iK-GeM0SOG6uA({g6E^|M7Gbgofa{KdYt2^kIdq?sqLtvVfE&=`CjJO&yBg{y08|Zd2w{vG'
            '&sLG_*LV_u1Nib(jd$zM<{=Q$g#fCy_Sa!?$-Z3jB3Oq}+u3pX0Q&c$F^-q(1eshgaTQ%a9X8(ciNB&mLqZ`l7seMUQoaV1pVKzZ'
            'W7d!ntSH51K(Ny>Xx%910qMR+jXVWGfmHKFkoiE1eIj-#>Npag&%3{PC%vP7*BhL9#s)L<%klaG5dHGP`?-63*}H(p4-5y;g5moM'
            've-CH;Y^Tb#!GzQk;&yYp2H0B6(g;e1pi#YR!p2Gj{tH5qW=V&z|O5m^v3QuU3`dUAAT**1{^03&%+vqc!Zf9{?t3^dH}EsK8eqE'
            'c6P=flco}#y8ZV=_<N}GH_m|!!G!4#_kic%IOl$d2lM_GP{tRxC2oP6au>!(Q6_C<zB@axBjRZSCt80Mr$DCkm)^i~QrNxK1U3&z'
            '9e3OV@kpp{;@)RFg#hhxMEV$YKP2T$Na_S?4q6O`!G~ZBbjoA&^@wNzWhJf@#knnFJ}2G+)C3KGG$2~j$h=5PN7MjS9f@kjMmaqp'
            ')C=SSxZ_l8qKCX%{4YVkR&IoDLgQ{4tushg0g%Bk!4p08KhAx%Dq!GnfLT&XLwP#&pm5StJn2A8{aZNqK*^&CP@M#b2S}fLQZ!~c'
            '7Rd{cFF#%QGciL>R~Rk4oul1~iOgul+y<D?+E4u%kfb*7SK=3Ie;$P&$G+(PdfMk^M5`u2aY*83aD;D&rUgWb+Ibrw8ed!zx*+)z'
            'grA@N`6^)H{XT(H2-{6!4B+9IWJtsmX%lOrR=PkF+XN+77zTyLmKJQ;d@F6YUqf=~eL#+N2nTQ^S3p)bR&#-;f_y$FngH@Cl01;Y'
            '=c|+~lJ{_!TVqhf0F%)Mf-G<zf<O$7Lgb}sJYO-A1AZm#or~V#d2ir#kNe#Vlh(=e1w6qy5Dm%h-hoyFq>1fr9`HWzUGG=NmBeig'
            '2(cOkM9Xx?U-==ngPy;hT@EgP>@iC%_6K$LpZu}&v;WX<o9gu#$q1<5ApR@*6Hci8;qlq!(VOG$c~76I{!w*DP-?s^o&`~<<K^Hd'
            ';0o`0r+G*JBkCIIG6=@E9%^*hr~CIyXnyg&*E{^lo}kqnE#z@j?t$vq8Gi%Fx#T<^?7>m*bkOe}dmyO2=^vw{;Yc><?5Ky{Y4@ZD'
            'L%_=<XyY~mw`e5Je_nN4fA?D-_gjCy&VPFC*7d`F^VN&T|K-5pIy^h>{^-g5{;7BTzHYPiiF6nR0cAjMi2~qlvrto)hCEv?+IDV$'
            '9w8wOXF#oh3G99+ehq-W+=L(JCZTd<?_F%vog9c=jaa*)aTlz85v2H$3B0keY}ajyqu?e;4$R?lfiNX?T}d?nnmt$zvKfDp;N*~)'
            '`twP6gIs_o(KbigRVHfz#~fA^?y6=a<{h4Ne+5Y|T7kF&=N;spC7i2u>eG2djBnor8E`;jNCps$`v%Q4#0U1&SGwad=t1-hWll{9'
            'H=U*8zQ_&Gpv;e236@po4k2pHRst6_z!)KjhsFRWeg!@-fDecxl&w(my?dd7^dvCHAO^?t1wbud>Z2B34nasl#mb+b2U9IRbeUkG'
            '#K)EdVB9fq*gK@=iodgxvC5R`LSL$Z$dTU0_mPw*hDoqYKvdWpf>uxeGMQ(+3}pZkH;Krq3lwSlX-ZR0AZ^hs6_C;({{7<YRPzSD'
            'q)Sot8tWA)4l@t@8Q<8Tc||HFO)?q^yyX#iTS>MOy4phfGZJbb{HafBGzdF4w@A3d$WPXaOM$YNi!1cQ^U`^orJjfCiZcV`N7_q#'
            'z(Z*1zk<}m>ntU2PJx=>Pmkb>Gy_LN(a-bX6l9*)NP{`(yEx<l8b_KUAC(@6e~|##6xR4n-m#`7572+|tK7FE>gl1IOIbLFHckOL'
            ';6L_@d_wR#|D&RjnjN&uK>G+9)>GH+AtwrL)qS3q@31G|HsrsBz6?+93je|eBa9fj|1Wd_ts+}%Smzh21Z@(k;VjJ!F_W<sA2y5+'
            'xOF4a+kCKmoAMz6chLHpfXc=vtu4;*Fbt&)o*4w+7~z3|gS7%VnlyFPpW8}SHs)tcEypA}Pu5&Y>vD1kI4FD5IL0+|fJS_WYz@(k'
            'lMkK69vLRb9^NM9-@w1Dm0qaR-PFghmHJaOjImNl_MO8RhzKZY;`B^WAGEvRL9bfvgDvdRsXNGV)y92?z@XkUOx!{fbs!Hc1S~W%'
            '4)sA1|Ern;c1zPf^`J|?@`*5}@@NtPq@a=r=-m_I8#u-O&B8xG1&prvMGLRQ?*LLoC^e?eUo2WrC%E&V;q=HDBA-aeJQh?5{BNoN'
            'B%8{=9{J46hdD4F|ElO2P{%AG+Nc&}E|H0<rgR|9jX!g##BK;wQh)ZOJm)bsc_1(VIyF$VC;BHzoZ>}kJBaSW1W*wr3v8tZ+JXY{'
            'fYu2Eoc&q5BKT~>AETWYRU1NYMdLuY2<FUJi6i1A-NKJnz{+4f0U0Iq5m2-ZnW`E9+B=R5y9bA<;VK;(d0<hJZBJKo5DK#KZG&l8'
            'QJE&itxAVB_^bNtv(~GZAekkgpu)zmuz~8UR#9*L1ojl(`-`U7ZSTVWc)Nl`rQOyp8S56oJT!GuV@D(<qb-nfi2}Ra(uTnp&5h81'
            'r3XZfDi~q@pM@PAB-6By)e_P<&m#qS0jQXgUU)JfnuBN+-k@y*gbuWH@GhcYMEi(0SUfQ)BhmG(w)wL_#aD}nGq8cTeBnjlMwy74'
            '$h#fkPZ99)-i8Jw&y%(os`cZ=|1y=xXCqvyCYk0cuM~zzSCu@RST&l0D%4#ib%0nnkJ*Ht$;*lhz_)y{rgcYy;uBcA#Ac9f>AqyF'
            'UxS*B8|>o5^}y6L%5gyNzJKSVEtkZKlxT7TkI~GfBZtDG&;#R$509=}e?F!TnpC1ium)YHd^`@sgPPGFlE>E|SKf_R89ddqcyxGl'
            'gV%QStglq(Yigq)-9ZztYTk4aG6qzPS4@zIceAvWzq4Sf2Bw`~Hws|(<v8$`8Y>z@0Q1Sj$Ne`ElBr`tLas1m3p=WbAJO_IZUP_&'
            '@K~awKS6tZ63l=;@Pzq<;?|ib7N=HC>8TUV6Q^j*HJ@shtPd>Ops1_l71SX|0ag-3Pw5KPeM#6<4?`&q+D#>evuF$vGbzY7b1M={'
            '@(L|F3Xm41tQyBs3)G+GQEc*BCaa%E9qH;UzCpJ-wVz6zqC0)sknt^udrDa>^JtfD(3M}LXs{AKH1&V<$FceLkSvIFU<6ReOiW@R'
            'eSyo?$NWl)=_ua{r(&q}ZurCyr-;Ig=n4aoh^i54r;JpQSVd`&?akxyELWKE(w}@k$w9@zLo-Elgd=@<{VYGu`u3lE852LfwJBwO'
            '=54DX2`IQnRR{TCOM<1gWtFUeCgEs!HX9>2a6uGANWGXs+TV5`-Tyg`pL6)sQPUF47_`QThrbcq8|p`eI8%}<yORldduXePV(fIo'
            '^#Q%L&}R=X*^@is%C~ft?T__cA9+T!e<x4QyKOQhloU?2RC<o{fX||5hK7L!U*H}z*E9<jk8RCK9_3S7@uRvkV~uiT&7PH-{K0{}'
            'WhLnD8vMAS?WOLn*b}=_3pSoNg>5*6DZufWgzl~+ZKrg`$4a}!tso>S-Ff9RG_>zB@Emh`4@M_L`Enf_f_mByD1*2E5r<Ku*i$Q`'
            'k9r9Tss=D`gZa`!pR=4v*Z@?gqG<irM5|e}r;@FkEym&e1|<V!<P=DZ+RoRwsp1F+_R~k%yHVkb{~EQ<+i-$2V^a)~t<egN?`g^$'
            '&i&#yQs(t?j;SZSg;iI~=$tEes>*y+i}_8tDi19(ak6IiuFt`bI2O_w%Ntq7Gmv)`#y%KmBy}2pX6Q#K2@I{T4$+=386O!$G#eDl'
            'IG)S!5q6$q;w0VpeT;+fQ#8>60SjC5R!Vvf>D%P|elhrfBvAUqAor)?qy)e|uTR~v4_zgo*r00o%Mh=q*bCeeRM6QGB#%_yki(Sc'
            '=>NPI1=)R^%!Hgq7~PSjy+j5I5_k^ZW`S-MIsIV0tkm2qZ~|sHBca>24H5HCL<29rm!Q4htv;^y5ESVe01LD=Bk5ofe01AIrCrYa'
            'DI#Xv*q3p8$2K@A2eSA5oFc5_SzA0*5+vi(Bpf|fOwry&b*Et4W+ery*>K}4hAcqGfkCW%4f(i;Z{**A8FT+*h_--OB1jaFvWnPY'
            'l1?iGtNydn$fZPgb=c{j4htX^$Gd6lka^xUY_@<&nPHJzk=<eYjdo50^=%*)#-MmnH=l&B<+Y}|2}Q@Mg^w;)=m@?ThhnzkSd)+K'
            '8dK6{vRDF?{pZ(UKPOgL7qAd`HcXa_;odN#|4o(?4koB_8@?bpnxX)uSK@BU0d(jYsC-FRQHbGR6zw)fr;H1b2GM#Hq{}#>kfR8Y'
            '5*iu)*V60y9yBx1f7pqhV!*x;Fs05uxfU-M@B{IL-H7Jl4OxSvTI8>?c;V{^urJOZyX0r@4SIh=7c!?9xZ|h>7lZT5!@(uUqu4AD'
            '2J<)=Ot*V`es+0!#H}QXuno@B(FD5fV1PawPL`$mQSanO?rR19=-1u1APwVd{|2#b?5W@ZC0wBg!?^Kn69XR6nX`%TZHf&42SDBw'
            'cBo9+X}C?;(FKr_j>E2mj%naDz2-XPU`GfMq#ehBk1WGZ8DzF8{K4=a;5j)5>dKD)HZCj-anyr}Lra`~<?D=IXLYY(vDDkRNO1~l'
            '*YhYmH|N8kftLKkcw_k!9`Jph`OF-JAm$t}<c#O+8vy@nJY$i8oh{*|BhRE~tWCL=a4)=vx1$BLZwQ?g(LE3t=+ye>%EtgdDW={9'
            'QUF3x{;6XkJyY!<U6>uCTW%8N4ed0HQ`Hou6Qm!h!(zY+$-(!8u$|}M?|b|E`-Hfi=lgk^5)SJ)uO^3iHFcC%QwDj-FKBNiwUqDw'
            'yDx>e_(!?LAO)@#ff=hxQsOq0uo5m5EA`cY^Q;Ow-|v5&oqqNuR`h><0ldG52l6JTrNYL94jWV9{?FUr8xd<nMqE+7zt8^*;{8To'
            ';D3_s)P?BL1%M0`juvd3#hAT{ki522i6hUy4AvKMWdCb{q9FJlUH-jyJUHncAZjeaQuHE4I+8P;`1R~v6a9kMD<kqi52R4cGa?f+'
            'bWirt2#L~iXo|)a=z4S@P_}6-*^?ekI(YXu4rf^?3f}du*oGdd(+jdvOOCq51`RHnE=fS07rY72j6HZ}?13fbpbvMAY~xYfgHu9E'
            'wNQfeYq?Nd@<%_9g3(KU6943IxZ3|~R>_zF*qKL-ex4+6!ORGNeHv0ETq{^T+m{FPg$XVCnGb-_u_f^*4xx^>1?}o$(0g~>J3Z<B'
            'T~Z+<UKZ5Q7~mx^cq~B&Tor*^IE#xQi_kfD7Y9BN?RH!I3;%Z)ZDF%;Jx2$Lk40zv3cyn;7}1EAaNcF0cFcjac`%eKwgddaZ@p~2'
            'cs6Q<*_KpN;im-ovsVVlUwSX`x1oeA{4(w-J=06ax52`<>_@N3k5fS~q;x8<Ssec0uM-McU>KDS9<Am1t5M5;`QLs$ywh($qX6CT'
            'Ch_l3+P9RFrihoAHile->bG0E#OI@EinRZI>-qlkS5Je9G*tzsR|-!17G*L{nu3<`O{ZBwu!>;B%Y}6c+|B=oFA+O@4M$=bqN#!@'
            ')Qz9uuyDqR@7?)H>aRnP62!ai`|feS5{iEhlDNgjU^*vZY9$pqkmeTjY|+6eXkp2dt;oUfzuw;tk1x?E&`_?(!H4mDMWM79d#myS'
            'xlQepv%>H#(W;l<U?N{ldXw8FH)_8Gbr2nN5TS}K+NVgQ*CJ+n&5VSGwJ*9C$6axQVUQnd_#Se1808cAdY$04dGZzP?=#l(<WiW0'
            'gN%O@d>S=*3hKZyR#CR%U{nzBzTB({{QsS1oQq2T)o1-kL1Ba2pd#N@yDp87OfUtcp?^B)onQ10F`*S(P7jC}y%xR{>)B_JY_Tf&'
            '(})t~_}cnT@sirYig&tgX)&T$#^e6QK)cb~mLzpCaO<aD6h|L{DDs^aCp7Xx<I*;FbH}ZK0qvpa%2xHVZh5eI>2iRBc}<6OXmwSz'
            '4_=GE!d6h4kMipYzm`h~Q%~EHIr7v~w&snCp-2`6H3r6GN~J2pG&|d(108`~?=i3k%nWub6U-oz5g0e5Uek^ga^-4vco|qQ^x?|u'
            '_vUNdH4HGt3#~X6jrZ{FK)i6<;tZWURTjG`tR5Xa+4U1qHC<82)O0l$(=bUhit)t|&%xRIi#&k=XOUU>$XCTOT;ff>a<7{DQHlrS'
            'i{AhM3mo*kF8Z2u2mPNZO+eY!*{m=cHtP&#&61ex%Fq1h{djr&&i+J2*9wzWvr=4D;~tt94~?>&r>Pnq$y<lUX5G5^gjTAC>**Vd'
            'MwLpYKl?^sT{JezmC#{l(7kx)Q8y!`^)}8HF$L>uDWSM=l0|S7h{`<0ahx?xF<y>D>+$-@VWst)bmOeP*=RMNdE|6~NrCL|^XAKj'
            'cVXhgAw_ZmK*%<<R2dPfmSi=|6Dz<B0zX<UtJ{?fJi1wqMrug1Kp3bapXaKqel1a<SyCR#pOM(0xa0dEX*E#6Z8EKMwM|TrO{^`t'
            'Bg_OLI)B9*ZrfmnM2FKv5Z(-a>Yu*rpT6~udp~3Pl}!lrP<5qYnB6LF3H00gAb?sUI@$FeH(=@4na`FJsyqu{X+~R4sd(hR!fdvi'
            'NcEw9sPAkT`RgtqbeqS`1mO#RrJ9(0;g3`jqA&c7DkAn!b-|%_QcB_|zkq;;GDn{Dl>fx=<VeYSnEF&_4VGbdlE$3({^PO_3a7`)'
            '#YP%q>L2wdzuc<wIF_3gk4U*iHSU(%)u3OwVKoMo+f^b#xmndd7OWE%FEXs&6|Bgz^ls!tmYvE>=Wx>fn<u?+rIIul^iJNB)bvs+'
            '6E+LNxWfh;Xvy8WtWFf8b}768g?kp54PC$<xk<X5QY}bMR>_-FOqgo=A)*mX;j2o33d%@>Ex)LM7`#({5)<@52Un{c%u+j|ikFfG'
            '$87rH(XH3#XpE9Vr#<Jl&<;6V%G7}*tDmYLtPfJJl3PbpG+Y_1#G%?>-rqMT94aw3X478yQ8>jo5vdqG|MKp<Ig+0C-}Ej99{R*%'
            'IHuuml#?}2&8Uq@aGY}Q!Qj~Y>Fn|xar^npw$hcoX&B5Wjd=8rCPnf!m7{;bsrM)xXDwFfsE3%x@ocpu-!072%Lx}!m4G2k%rKf%'
            'I>X{1;!XejVxa65<2VH(6OIq848uBa4x;jq9T(H#-mnb}VnIJ<!8#psuuGOD;RuyfO6V}WL4uYg4Q+L&Ku)pCgY3FPaZe%fDaH(B'
            'lqR0gj#c&%QYcJJb{kJr=2DdbTT>r#7odY*$paT)P~p(X!%L{DriP;~fu4d9OP%1A4l33g<Jvx-yxQ{qX-CfpBc(APlE;URqDB(`'
            'z7)SOESRPeNwzqr(L9_5Vwaojm6G^tszTAA9fel#8DRAI%;8W=+!*+@W$npi+*UhX0wy<{J;%LjJ-gOn$-EAnm8U}>l}{$m$M#<R'
            'c1s6VJLdo_;IMmocG^Ge9*e`X6F4|W2b^`0vhCK`@7NH>;j4MQg&!E-72$u_jb|7@!F_iR#8o&+ui*iO-R`O%9=adzKDObUC`#kc'
            '(Geaez%e119+wg`Dw`k^d$u#ttp#6Xv|8FMcz-Kp9?a90khCKm$z{fVvNm$gUS-M>jY2|+GM&*L?uT&7R)`_Mb8<O>=W!s4NJaTa'
            'b>csqlYNY=P%fQSrc>juT-<4MlNrfmtXGXgF^14Vi_*b+aN-#V4$=$K;Wfa6E7&xL_R1lS-+<1pdgRGm5=O|or?Ou>mKUek2`x{?'
            '%so_})&<ASE_7-z^#8X2`{>pzfiFMu735zvR>J4QUIjjT+^Gtes|A{UpIr+<yP1Zj3`$cdN6!(y)}B^%Rx_58@47UCohG9q-ldG)'
            'pG8qN>cPf;`5s96C5KB^5jr14oVme62Fl*1D&yt0A1Dmsp(KaiMrD1ZS*<m=+kPpE@3$wWBMuC*6E_)}nHFJ`w4kUUj9s8s!@{M;'
            'kiMV9OYt$k&6Um%t-s1>`Na~g_ml%YNLra;-X>a!ce+}vUZu!pbV{o7&d{~u5J$pPBuLdx`ltO9SZ}(BXEL>=e}l1mnK(N={#C~%'
            'EfdggleHw^AY5Rb8kMr0Dle2b&j^0P`QP8ha~=#f40yziN@kS<b9@_&XF8|}&nG2N_t(2ArkRvk+)+lOVb#nqRt;VCDVsa*706mP'
            'jfPDJjKu$imMWQc8~$#nB--s&Hr9$l3Ej&=QZ8C|wqf*71YT^5yw;J)k8+8Hz{#tgs!3z3znK%h?J0Z-UXOLhaSH;zQ(`s5F!U;}'
            ')C{j(89SceW_85XqH0>Wsf+Rz)wSXl^A#%jfOp>ZqsiPG8%NCVy$XYwpOKnr#Z{Th!|}=9JVJGE>E_X&8c4v2fW<uuX|m?S9aYX@'
            '1g5iiz!jO4%4nP|IU){1l{!H8z#BEr5h`TOAY}*G<x*QZYz8x-!>LCSMrytv$>Gv01lltSIIZ3u=v5KsmWiiR&Rho~`^^m}8dyab'
            'ECs^4j3?9!^VD+y6a_pO7z0RuZJ7bez4G%?3GU*R^k$GVhI%8+n1HA5QKtkj?@Lu;7SNZB-E_sj5Ks-ChHIv$t1?bpsS&SrJM-)L'
            'Yi$Y4Q-!H~U0xpi!g;&szFIxJF3K>aIGF0|@A_U@b+2r5FOHh2zDqCkD#WR9AM0VUM>+h)fJE_S2_iZYhuh0POM;^vmBVeVhs7Rs'
            'P65z+8<&CNG;<OL1DKd-$k=ud6&%EK<-e{=Ioza7Vj());@JHFzF^#&vQ4C684uViSK0OoHtZ`L>)OnGNb$!YLCiu*xZEK%=PGYf'
            'JZUtIVkQg<utq(Nfv23_Jbt5r*woW&O487*ARn$ZcsNz#mjs96D%zMcI+_%$mD(-L;^ew7uUmC}V2)_)DdmaB?_-!5{V>%n+ARiZ'
            'nJ6$caLMTOsZ;sU<03wBTLube3FM6_kQtsWHsD{yQ6zXJ6{0CifR|Gnt`HlMo?BTtIB{bX%ek-=;*{`IGc<G64m-KxPNw{MI~BeW'
            '{-&zr;S^ZPGrnP*F3-;>pzgjMrm@NeYsUIzbMF@D5U!=(>{&5%j&FuyoIkAWp(HMY9rB~=`sPX^EB4{%X2)Fg#6-eI>~<k%W+{JH'
            'ZLoixC`^0&pweHOnWdX~)=Os<VbRUVF(pt(b617U1#a$E?_!k+!i=%378{{`-K(nJx%gmq%%2*4Oyo8MbSWqp4Rs4s5-{-(0|A%}'
            ';Ax{DcBHDQ&nao$jFqj*Ac5cqNKvH)^IV{j&qEI~&N<EXP|(s+RR_iBmmhb#JVvpFr%7EQ-eDbNK@`mNu@A)>_Hb?(4ZkCA>^YXU'
            'OTsw@W>r-zkWSf)Ii~FPQ6q%`lh-J}&ck9vIh9u#`8RXuF~%|n<=6*FjgU!H*iUjXPEPx(v>`~gv8%EM*{^LVY^Q%#Ym+wpRrBen'
            'A6()$?NsXj&FcIFs;^G!ur2eI++iaRuDZ=cT}?>>l`2~~OR&ZdAYG}<*}0AEO<doat@>g?psq-Dq;Pc|&qU*)FpSD_RIe#2;i0^k'
            'N(A2*jiPffN6&rCzf^=8!@c|g%c{#!4O}f`ExB}shQ|DQ7Nq!EhDG`qyJz{9$_rA&6NNS&h`m5h(%Xi5BJ-YVyB;8=;;sjvH^zP@'
            '(Lp9iD|paKkZQ?5rhn9iJ-WoTIlH^T9~i!k-pXnQbTOX)k>?v>2-siArGn-Lsla{0F0ZXyn5ZQeKXPJXFXr0a*_>dmv_#83*3LeL'
            '(KN2NkMW%H_DMkXj2hgpQ#QYSim6Vee#X`<<14QCODC_C*C3#F2p1XvWLIC%#s0(tp@vQCRW^0NG%Pa>bRba_bhCkd=hKaMJRWS*'
            '7{zWv?$5KKZ()tkPsz4pZaz$(v0l$g%wIm5wn^R}V^z)ktS13gxf{xj*?YpPU?>v=hPto<+Tu|i9UMyobP`~(ym$&fb*1=mFrV`s'
            's;n<f!UA7OlJQz!9&eYtf2SS{h~?@^+?0upstFrP7wqKx)yytvpsg$RY7;Tm`fKQ<jF!#JLQ%5(QC>o%$hb&%Lw3tC?_-DR9kr>}'
            'XM;MU(*)<V3Zik0t_mGzl}%fJa$L7!1TTkoqrs1ub{cAkbkJcRAPq+M2~34PPr=zx?i<DFbj;k<m7gkeFdl=?B>ca@xH|fmY5uc8'
            '0a79UOR}JI8)oG^_9~MVHQzFoC??oLgX}m?lGQTf&Uh@7fT(0qjehc0a_7s+p;gyf70+^@8Yje%r*p{RbvJ{KYBzd_puiV_b&d%`'
            'o4VU3F?NG6{&lWpwJa`0p8D%jw@g(}*jrV{+w%1v-K-U_|2ACxsXI@ZWff@a*_$__F+9IKRW6E$XQ!t<<y3j0%ODO_nPf$9-8SbA'
            'HLvvKJ>F|7;J4;<@8nLe?qTg4J|=m#VJGpU+@ByLbCzd2Mwku<l4urg{K@8tI;c+dp9W*h+&Kb)I)Yye8LEhh01PIa$Sqqfu!JQt'
            '3r;eMzTRz}73sf{;|POE`90LbsYV#F0r$DiCYIfp@B>l$z&2VE**8UXP3BFDM(zQMPi=N!Su9;MS+cZLSD9<ox6S(?)1IBXo}zDO'
            '_(V9Ldu@J}1dJ-{G{W>s!}dD2_2QQ>I~aXXTYlwz>ZXmV3wKZ(<`inc@{va^?$NEB9Nwzw(xwG(jx3B;fqf%|x@3nJd$oVPQyk4k'
            'yz6?qcuBfYj+0cp&h|AbW)EqUph68hxxGO)#Ec?{nPo8Y)l=ueS@<7hf;}$$hbQ1qk9G%l%Uvy>s_Jbv4E!4|^78}^5un9lnXOe4'
            'I8VPJp-cVcx<%hAYs|;`Oyni9vK4}Sc|g<=r{CkO^?43k2EcT^9$e*DaP{UYf3_!-_)e`=rh8_0?>~NdckkZ$`PsR5-W~M3<Nis1'
            ';GMreL6M5i8`(nX9=NLAQK{WTwZn3mltgls-KuA&XP}#4xCKdU?1K^mC!}%%vdu;B9E&|;{!0otm-im>?Vx|s17b}XEyG0+V^U1n'
            'uluGq_?7zk(}2P-robQO>V5z8=k9U;$b;VRd4Mn?UC*n?v&mfy1>&TCanV10i`^>No@{@4`tJ1X7Yx<W*B4x^6;~i#y*L4M7Gqr+'
            ')<m&pe3*$5s8F*Y*-fdyB9<@??}C={&!BEtjdgI|J-z6|tUUhayZ-4Bh1#0XYYGcmt~m>?zys>^_|`1yzKHLr>O9MHKvVc+ws+oi'
            'kB@)s9=<Dnj}Ar^>NKmK9;v4q1)9bHm9tiV+8fuz(%~vg`*HtW&wG!dGv}xIq^!cH-Tp`aYQObou6gSH*1CRpw%>eIeuh@t#t)(U'
            '+XxdFUtz-^Ky3XNqHmnYL-?;z;3q-yuMaJx8(0p#1#O_uZ|S#J2c2v9=eLXRYR7I|{qS$sjUNtZ+-2zT2PkNrIgi_Mum1DjeoKFI'
            'cCWwNFvWD%GK#pGYjSnH%nOaxXN1(8XFIR0QCI9+Ghu1vq%%Uff||K0h3av!J0)GZQcX%3C6unC49(qKkDa??fYC+oqzi}X5b)_&'
            'P|{*d?oT`|cN-wr{U!$Bx8i9F<k=e;&4YE@o0yUc3(qIv%}tQx3EL^BJLtKntx>{Jj9ij=r&0_GU`*yR<q|kVBF>pYY$}9KsRCr?'
            '<EgB7CYZp;+;?ePmZ#6uywvToPVjD2NUvu+#h8c_tb%2g`E1C#pTsOML?OCy@x)?YCpf|w7m;GB2yTF!PO5gl#K}x&;*!Ne_xR3j'
            'Zw!kQ=_&zjgzJ(5DN>9GvwXzatxgD^*XxqeHb$Hc8RTNjCqlWyRS6I1RSH%yZjXs?mp7wqnKx;C?Ko8!UpJIZRqBSU(ZT8lEi;TA'
            '=>R;DHDSm;NJBgL(2<o^OzR=Pl~<x>BUUz%Gy{VX4%fuUy=kzzo}Xmw&WXRMIG82Q*q@M?poVW&&LDrQ^}zf>5J84tGK@3x_+DNX'
            ';7r@}XT#MOr349_*USB9P4RO71uflKBKoFuaR`tOsjO$D$HEq?2q`h;qsCH2*8AUf3Kl@jd5lJfoPd?alUeFLDFm4BYc>}J8fzJy'
            '4lLj$jh0I>#qZqvsW3_^ZJf2FxOw@_rzfS^X=!H9|4d_|B884}Uhv`8U!_?EXrbv76JxiAVn3Ao_^!NqqA(i|WA#!GBe=W?h%#$='
            '&nerwl|_xJ#D|@y+nkhe43p%F@AseM|GdEedCAyD9{mx}G!`v4uN@Fid4!iBfVWg=1g`)pd~u8IW5GvFT#K~D!RQK1RK1x5{sdiX'
            '@#Z}UCR^c*l&|zxnVpq}9&wql(sN*Mx8W$vjC_@wAqb2)l%6G9KqM#e)^jOaD|ds<cvY2RNd;R*>&C)QW|&AQpETDmRZopN>Oh##'
            'PlU=}V-+|0C7tS0Z%Mu|YfdS^1`72~wKg1*aGvtKV24#SU(IN^rE;^;H=00HQ@o-9Nya0A9Sf7iAZ##gt`3y_DUOIU%S>G@DThoj'
            '7%*A4x+M7Tv3ztC%>6Zsd{lHRQlT%$2w05RZiKR`ri((yPlLgGF*F7mid#TEEZyMaEgnvEf@5MK2>pRVoGjf0V~{O;4h^5=bRI?='
            'Opahx{m_W@;#um5Db~`Z5+9LvT9<6Atz#VN=8=|Sq$x{%_IS9pil5`ZJnz)t$2Q+n0&vdx9NDFwTNfHsh!e6!Dc+XKcmk(Fv~vvj'
            'GM?d|zWNh1p7UQ@-8Z1^I0ecH-;#sK@04$wO4hc*@mu&G8vAX`DN6V2a-1s|_S!99x=Hg<Ls#%=ijrzks^pGEpL9t*03I*eKpwql'
            '#saHe(0x&W!e1u5mHii5O6xBnn6U@;o`KK|C(;mb3Okd37TULEYqAmJAXVQi7om~Wdhz7qg}^utkQJ-5tmBc5p;YVI;jHrM)K*@K'
            'p+<&=(>7<H0A-OO`vZaYH|GaKNp*f`{5JUxRMFqs@UQ!WoBrS!5+07y*H}zLmtCC*P3T=4HKAqWnQK|gRNT3w*cEF(s5mDSw+*x>'
            'lZfb+W$gl{;z(}YNO{rh0+6_(bmIIcfTI8lc#<Np{+EN5pzp1nF2G_dc;pD6GXzj@x_n+F8XGa7!e(9t6FG)kviELvV2jB#%5Gd+'
            'Sz+a!CYsmhSe`S)<XRww?U!`j0nu?_&AGO_M~V(~LH#zp1$o#Ux<vu!tOK`I#QOwm28o96%H+53r<wFbmJj73o0u~Jxduq*C~nWu'
            '=~`d;H%7?w*2Fv5YPUdt)3rE0J9~G*`3QU@xO<j$NYdWDe46sQ{_^}7xE0WdT$k5LD-xA@FwJ6|C4BcDQ)#~Dge(-=uQPQP0@hg-'
            'gh8uR*SwhW)Oo;U;dHdy9!33m-s6FIfCrCNT6w)sH&N6l3Z}_vk@}~+#EP}R0Q<lF_FK@FyO?woemW}Pv3@4BAp85<ezfj$tSMum'
            ')(Ck~rCG7`4cVccu11ZyzZgw?u{aQog+BG-yYlCVc5sWdA@`-&XyZ*Ee!SWOKx^g%7B&IK7gq=J_+5X6Ym4{(7_Tc@sGgvd%sEWZ'
            '%K%f+uTXG9>GS;&)e*@MFdRb(2jd2-_VJa*t1o}0K^$kv)AQo#6u!=WI1QOPYn6=;v{#}l!@M;;y1Y<p6PJTS!2=wK)*r=Wg(OED'
            'T@0F{^(R5)?(f4?g==4;Qycm=k~1aCq#p$(oWS%GN|9{JD(5J!qLqaz`z2K45k{uce6oUbAf5xpUm#dXygQ*%8PE0)_V;OO(qR8A'
            '&5TB;pqDO20O0*U%XThPj{agf59GGwImBgZaX|`7i?)hQdoX3&vWVA&;+HU*#P_L8TJu30{)X8{T<DOJ*I>qv$%gzIRDDnbO4c9z'
            'JN(^;zjZ4HJRp0%Yi&PO{-X`mzTtF%K!;<0?>J7qZgdmOgVb?dbTH*18Wt0m!`6ye^85fsG`MLy1FW5E*~Jxmd*a0_U}R7D+-iw;'
            '4I-Jn416L5w}fO8M8*a_ZHgC7@yrw*yp*@LBO8Bb_{75Vw9|McTKukk9|SXah7Jt>kN6`r{iAZnl{jI2>De>=Dax>ve@VBm{+N1Q'
            '_#O{6!GSWgK;~}D4QnZF{=EHc5l85!Wj}=f-#{0!?hR(zu5&`hnFNvPPTerqDW_pQlC!X$ZJNZNH_t%g5|9q`x7L99i-`>4FM6Ux'
            '7g~SPW&oyO@O?B^RuZoY*IP@pf`h#)a|2Nh2Gr(}B+DB`F$(4}mPyItCZ?2NN}#DXqi8sZ!kgQyrHCTY2<<y-ETkihIc|3R&ee<j'
            'P}Di0TVF8RKk0;C*U=y%<xcZ>?9Z+C)*)ce&ZoGG&5CKrHXS_Ij3t!&jQ6d2KmzWzTdiK@tP7X|i%*%#9>%x1#XZz)`M<+HYDVxR'
            '*tg$79zjw^+yLt3DL|Pa7~^;`3ISKlZ7dVUXYMDZ|IniYdu<bGa-XInRTFL1E{(CP&LI}LH6f9eGO|lexYGg10gI&|f9TM-P@RNH'
            'z~zX)6LG4xPi2Bh{8TwA$5cjsyh;)@;-Q27*pH;bGY@>qCU)!3r!D+k7DQ4q+|$ng+CSg$ipSr%uY7yyCzKs+2D57LAE~Uo5q!Yl'
            'uXsj(O6eKa)8URNxB!KpPQwrE#LmBAO;e{0?H*z>_bkIn!+msg_cp_m{I19iezIDCP>;}rFGeY7L!O_G!%&qnr9$yp=Q$|`zhUO1'
            'l8-zCE6oqWV%ZeDPIYDNy@6dgj>og2`!4`IPBE!^JmwPb<C(Y%eR0NF!~b>WVP%*mXD{+m;G737Y9+55r0n`Oj)DI~@#^N59XMgP'
            'c{_GNNru@%H(Gkq!X!pI$YO=oSW?We45Ta-KH;0XIn`5FA|o|A4x?2rGy)J4yuX%}?8Cs&ahZdoB=5mWGtUDVVm@WKG}Yug&&cet'
            'jOX(X{QADh3uLQ#=M_9M6K_EOwuwr7X$oM*t!X1r3};emEkDzgec?0mr|}ul5xSXdk-xNu^QIF?zRl%Py$88IW4h{cM~*Lv1UiNp'
            'Gb=Cd6Hf&B%u{p?KC>m^N~5u{5I|Wh_C;{i-v=@V&n5Z+&!RsN2i5Oi&e1L1tv74Vr#ltKW;_5roPSD2t4}a-WAvT@z?K0k)W3iy'
            '15g&kV*gs&-Gf@9o{zGMV4r3U=?_+hZ-wuv?O0n_>e)rj5-qJ_!xI16(b!Wx$T#e%FF3UgMF^-;7*i}72d_xm!`H6`5tm?6GZL(x'
            't_nqiGK3+G5MhSUAxD483B;U7>Y>|iJZ&njDxC+xvZ0TlYaN~s25tBj0O%nLb*+17jk2{mA0Xy<9;ZP=HFbAta9*25KMe(}_{%)U'
            '?s4zvZSTA)!!8Hq6LpoDb<vCO&~_J8R`%5N4?XOkA71tc-j8qq-@)L6pU!(1Kb;+;+vSVW+ru+>bN0(ATEC#dv+u>X;?=(9JO6>x'
            'I3_-zInR4cv4MHRf#CHn2K|#RmJUL$z`jy!<&oJ1aue+UjhU2&^Y{$Y^Qo??AeoPzN8v*FBV-NIwwiGnX?d-ua*VR#O%WxW)C$L)'
            '`y-6tn1~@J30R<?2zyo!+oGQ_?MvC(fft-0{7}#!6{l8pb1EYT;Y5HSz$sbLWUP_|r%M6>vNg0B8}5`UB^G;)p_yjklhv4#^U?Q-'
            'zmlM;W~;}gM5_cUw2S~k<p}|g=oVQjZz+5ZhKmV!xs0!jS53aIP`+4(O^yZstvIc;;)C4pO{2P~rn<1u&Zc0z4Sbn-GI<xl!#vBj'
            'eH?X&G0Y4_O=*rjcv4%@LsZ0ZZjJygKPrq!w!V&>Ah|c%wx@ilR8UHjkBaC^Jtn%CC?6^`l;TN~dHaLUwVUQ!&XXk3!;UbkPq4am'
            'rXBXYSBJ4O&Vl0C@z_&3o^o7=^K0+CcPs|opHBLxZM1T(#_<1<SO?h*1Gp#=!eo`K7AsiL4Tf#$oF3(1O7wOTnD-;YEFf6x@)PoY'
            '7`Z=ew+>d?J%wuMzffV41f{xgLS3hrDC|=*vJ?{cD2(I`PF2WV=(<`Wn#(Ll<&&MKpLuvJzVV|K3`?snM%q2*<b)7cW0qsE5@*Ae'
            'CVN!bLB{z(**emghcP3K`AIR**xNoryZ%4>+KN9UJqmL2RSrL{?SsTY!Ji$K2qtww$;GvH>M+2q5$7e@;&JMmv|+!c#L*}q0p@LU'
            'w~S7%rIcpKvb-cBdnTGpslJC270Af2QzbuY<V*d#AV)r1;!{Nd6;@T*|I2j!$~-aGdhW<%Y^$DDt5D_Eio2m9+21Q0XmrbF{`CR_'
            'XeqaJPU-pi41h<V@*>Rj%auiJp73;S4T{_}c83JCjtNK|9at{#p)>F(v99WZMvhn!X11Bz=tOnozb6e--U4(niVohdXw5IZd$M7$'
            '`M#1|+Fu7jDi0{!?n9+%O~@0js;RsXsGmn(3(-S~Gp!kibI5LN0!lze4eHoL-U?Xj#zx~fo&yXc54D1MeLH5U+F|*r=0?1fNA-YY'
            'cr`^%)h$bgU3og;9BF?Or*3UB7>(31PgN1vums7i^PFZC(U{dVF+Utu&R53y!lA0DRoXF%))Zd4n|f!PdS_4d=0d2o-=0+Tw$r#;'
            'eXUoVLj~`zJY!7Bg^K1n4;3PLR?`N5s{HBvlqMj@hNuAQ6`&`TwK4lm(WqT9XdURzq(`ni!`D0IFk&52O1qj%<JSC{9?Mmfxfts|'
            '1{2CHhjfQqygr0B6(m-s#*!4p6@S2sk98Hnz0$3&>2_pY*`^u<&|$x3$g0DBT{5)D`($&@Rg~FQw(!w57ENroDeD$BaI1`NvTN;?'
            'SM_I9uO~0QiUK2N$dcXNl3&7KJL1)TotrPiy9#L6tVUPLFk5Z0<ZI<!u0MI3_+xpT&GO`zSl{QZ?f0y!oD=v{*I)>QNzj_k`8x3T'
            'S_H@>OeB$WI95o7O;NpI<>a|vnZi+;n|`@nD?`}H<;8$<*N+19=SD36ty8oPN~Hw!75hu$7yfq^>B?Ctg{x9}X6^o8u&9Ch_H*w-'
            'S0R#BDRQcUViFpzkzi^XgvldVQrT0{*?7wVFra-Aqtyh>fn@zNEZXX>vMlo&B6eBUt*7F63nxQj$+SuS@{JmbsY6QGAd;<MKPKUm'
            '^;8-~sf;}1?-ESflZN9^01uft#q?Fw)TYKX(UOKs4L{+ghHvca2{Dc5)?M;&@2zOD^)cD+lc=#)Bk#pu<N)<w(&<4TXd@p*V-#i~'
            '9G3@Lkje}CX%w$?7gS{F1sSy3%Uko|LTe+j%W_g+yhYiL#2{-LOR`lwB@+O<f|k|p^bY6v&HP(9bq7M?Hks(eB%ZG*yDE*oTQ&Ob'
            '_2b?H4E~6YE(Ul(4&JnwMde}^n-`<0JFRE2)p{oWES&kgp76;9@fYNp7PzJpV@=tnHD#4+%H*1=$vD0G?RPV~@9KvCD}Mh|z(oCQ'
            'VEylh3?VVFAYnx+XhTrsFVovN+tR%838cH)I$la`^GW9LdgH{C_*eI#Cu|X)xUEmkm#r3?Pk!x8ZGG(EX^rivk|uR}@>4}k>NMo1'
            '%9_+^ip72f^MmeYJuiQ)U_i;gX1%yB-BBoWQAv?2+3i`JjMMCQ%9BO=B3VwVWyx<r&YN*!tER~FG*x7ltr2FP5JsEAHrs5<%~*)J'
            'mA9Yrw%yBaG2<t|x>I|{UNf#5LGKg}zUeCd_fe*Lq4H@i_FYFj+xcvo&lZ&ZCb`eo(0{U;Y%wL45-R6onI$*rcmIWKJ}3`ch*+)t'
            'ZLl};LRA$v`(?xa)7`N`9%j|wD%pt(;4YZ^HrZLr3bK|*OLPrys2p_s^iZ*K>b3b&KW(V3nW9Yxi&mr>%bPibWfCty0~VB%C(_2Y'
            'Pk`la*lcMy--O2YrvZFb->^UNjX2`MaO6Ej+N8_?j;RiWk=R1&bM;Ol+Z*jFT-nj=xW$?<8D~{;vxCn=#u6Hv*!Wau5adfF|C5lE'
            '(7L`u8BNKuXS$S2qh6<qak$m+r0B^iEYVqFYOK0N)laAA6Qk99#T6YeWzS3ZH9SE}3dV!M*C)PE$wn9cN9QT)Ho8}N3R?R3*UII^'
            'a^{~Pm)CAgPIPv*B_4YE=7o3Fq@}FVltw8j-LPxOXstY=9H)Y{Q^GGCotvW!MyZ?GX*<%<dUCRKI`JEZVN841MTX~{PK*{u+NqnJ'
            '?-V~I*Er?r*0ekf*jOf4pt6dj;Y61DX2&;}6BhQC_wpm4B`Ftp^eeLEx6*rmNq*u!B~Pn1o~d-YY}%8RJmh+1Y4&28hA<6`c=ie}'
            'BHTN`XR?i9ZnI<#SS<jgSjj#KX5&D7j1x36Oje1|$%%IZG4t0G%my7MZqXGeKY5tYq+Elos0mU!oMNDFvBH~LVyP9Q5w+u=$~s!v'
            'k}hIv926>w16rxE+2x9L@KAv(c`Z_Mz9Jx|_+~=&L4gniAKO-yHf@>k(Qz`RoXh$y)2vBDzA;>(v3&n&TS9p(PAN^J9z1n_+Fr)X'
            '1~)Dy3fbDDMJ2SQ9WH79^i<`JXNn}fcUj$GAC?2tiFRA<=ZOUJUr^;iGa@?_>MA4Ty1Z3kZAuqJxmMVGuQSh64hA(zEAG3U=X#eE'
            'T|T!o#Zt~`1XCxTSuo0f=4-2lOsD#B2~PH(X4wJsMwMXF-;xt2SEE9zVCH-r-nUU=*KoG(N&rR|OmLHcv%yIt81-oqffdWSPenhW'
            'ZzC_!eR1?o=9h5WM%qP=(<vFni^)E0Pd53rLTVhonTL&|*E>dfL1fSL677%`HLSUbOq(zZ)^;wd1Dzp7e%EY4&S6pfPyayl{`lgV'
            '3|bE6lv#xegaG>SwtB?=@G4`B6|?XzAuYb)t5fUYPBCNkEe1nnX?aj88X~iwS#MRawcK#>QKCuG(&QycezAkpjMwmJV$TE6gF8+Q'
            ')r22mWQ?e)uHr`BnehydiXBL-QwVS2J7tR~<vTb`R>0K%_siD!)=yFj?_y+AV>FqolSrat!|0}M@cu;Hto&qRq8jIJsll{;vlfkk'
            'YS^imut(;)Fv9G(RXNPAP}So@<5?3<XC&9Vw<lh@Um*j=IaL(k@}-=!JNQ6~g-+O=iWlPFzDCw=vTW0b70aQ}11eyUa~33?TJXeL'
            '27td9E|7o}m+AyPW(|^B9>KIu_6efAgr$dt!?A3*#Xjl>!i*%`QSuDUw(>$oe^aGLWQEVcpm*|qaPisva4Tj;`mdSr0?6!Y5@w_#'
            '%fcy`5s!`l@;sw_3cl{_B7}E&LhwDaggOU?U|xZFMkxnDeA5*$zEh0E7)nX}yEGJ?yrF~%)vRE^oc#<MUjL|f!DWDluT{=Fm5sU5'
            '4+RTHQR0u)2$q?xxCnD>DH`M3I2==cl65L>VYR6W6#WHtl3^tVbYxu^9SMU}AOo?QlD1+P3?^%)KKwz>azVkRLfX+NM3|#xFbO#p'
            '2mw%YCSJP4FC`SFQX{gup;6Dn%pO<J52g;5#)uZd8i<@6`Ub<^!UOSu!~G$_JKA72$C5AbOqOUhMt2`w$2Ds#su?uYIdc>&?L>B^'
            'soVr(lq)_8xxF+#?Js;7M=nf8V%fh$jw3h~jv-cO8-p|TZ>pSPy*FRyB~~f~0M)8UnT>(2)bHyZ3m$0W(;%>mgj0zS6a-N6*F#|l'
            'kL#AzK-s?X!f>lX6H*0hR#nBOS~v8#5XX@F((1a`)HW|?8^xYlpQjd4<v_Z*ysg%8oYMZAQ~=f0P3M1?@b+={x+3UBf+Y$Ub5`2s'
            'uZ6A~%7hK8AuU2Kqr31ww#$RKRo<(%bhmM9jkCkVN}_DB)asM9<8=i#U9NuHVnXIyqoxpz%C&^(GQb+Xw%(9#XO(j%`&3plA-nK1'
            'ef>&<f%$_(Hau3>sDSD`Wm;tpqpBnE3FUE#<bF!!`C=j6LK&e_FmrwV+`6*6l{Ixgd7e*K-`!&7TNmd2bTC$z@ZDxoHMoZMx8kmE'
            'Xj=B@kTu73!c-H_ta8D>Yu#!}APy8tQxPrN1^_Y`XhX^_vK4AmLeYmYpI5b@=$5Yv$b?&V75<c!^N&Y&=gIhgI{oERS*EH~EpnF%'
            't8rbfgJ;(}6;G9Bgm<%LG2C^pcXldytHo38`t*vo?pPin#<3$f*tNfnwk3Y7e-~S+3ioE+c+HvRuga?PvV1jzf-^4y=QfYWrdsq)'
            'v2&p~{cSiN=^|@Y=q(y}&i5o~(y4943`i*jp}Mb&h3e&)WB(@6EE%f~RurHoJ&^6XvO*o^aA;_iRF%~(rDgq=lG}P%&d|533s>e4'
            ')8!h;)i~F3M}8DZyVJr?G4x%>&~pJusWcn{#-m|S0;+$+BxxCgc(bM~hyiCjZMAeZ0M6K2s1GR-+M$S2&VZ>GYw&8#UCHL&s4Iyl'
            'eaDPINuCPG6d)OolW=5(fz!)Iv0&rV0ihc*sxm^VdQo;g?A-qC3`nhi&`Z7A#k073{rH=4lQnh7AOY<N28DO|hoM1nQH8c$zCqF@'
            'Y!Hq3lCbbS|6ULZiPaK=M`7e$CRCGMnHif=jMurVM%k5#XH|}>x_Wj+RP15CAWWP~mLbX_31YF#y?@9CxpP&)hzjPIRcdm3s5C5j'
            'WPH<~qq3PFvRtpjze)xye8^aH3)@DQ8@ZUK4UUvhwlsWhk5mY1r-D~?TVzl9kM2cfW4Pu5=B3Bh6$6n)KMcr1D@VYV_Ua5orM4CI'
            'iD`L?DN!Q}Jso-cxLtWC*^aNb(kjDHhD^cJMGuDtnQ`XXMcqmWQna0)8vz2;qZBA0@3&><8aCAs=g~N#-gOGQVN6;You@%CL2gW&'
            'Lfy2_wZbGhSUHV)ag4Uvzw;pb$>Z`nF^mWGB=Td@rV0d`qVeliFM#N$<F;3T21%n3Kb8-N#N4i}VZOEMYdn=Cui<bJ5Hz8@KR^4a'
            '|6_m9JHo$yJU%;ocMQ4%mqR%054xxumckP>Y^}9)9*ikHIyI@yJy{CdC^$L|Z%ngl`TkC$JSWE3qV{vT241=FkU?P4Dxk1Nwsfl)'
            '=@x@=Au^D5Z9d-Q>}s7nS)^(I{1KPQ<(Y@P#v$J%;a&}E%B~J|vFJH)MtlPto6Dc!l_{kQ#K%~&a8G3M(p$=ZXYjAaGQKQni}x};'
            'T0EVGV_-!<$?3-Vg}7NkLriaCj+oxYtNDaXl4u(YvsB#;iD#GuPWdk2nkX}~K<Ax6_!C_UL||@plriE(n5imd3$jv@kVBH$J&d2l'
            'X!MK|IUsyaPsyi*dy;@m{RK!AnA-#9jvfWkc)proY@Q{yJRLJ{+9G&ASaic1q{qAzLjgN-yyrew71eCfkwppo)|<TKKnWW;JB<#='
            'rB%w{Q0E30Y8dps5xjI+U{3B~_kADrXaGE*2CyF_So|GKS`@sdw&oWNgY`FngmZLqq5S_`z8r}p8qHNG7m*Se&NrM$6b#Di;Q-^('
            'W6l-|;wBy%L=e&$=>fSGu?^97Fd}Uf8k9k=2jB<v%?HOG7BmX_S8r*7c9<$IZ33IWJiwRB!@=cw?+9sYzB9^hLN%+)Q_`a^!Uf1y'
            '>!oyq^0DRt7n<GU<YKiwd0r03pj*$>5_0-uatAr_x98}12Gf@IgeRw7V<mr3yYQ$C_7_y=B>y@P1K8i$9Pn=)Cv8K{(HjKRk<Y``'
            'gG#f;)lw+Q&EvIN5fRQJG~1m0qd_k%Nr>Mo214$Z9%UCrh7QE7AvJ6dQ-OYHnoj0T_RjRCQ4@5jH0Tf|8~e-7pYTT#-4v=knZ=y$'
            'QK2eUKGJ2Mpxf&_Lp|Q?z!MVrS-j!7v5VP1F#Re_hBEl`?bHw^F*I*=jagS4VyMszT2MhKl(kv%E+86FHqI)uG3;H3{ufAK0!S~@'
            '-xtS(2{Dkwx1#Y}eEY4i;Oe^ge82s@_znka^GKMjT|C!Em&g07FruW@1~lkhp@t1SQTOSj^9HrIt>lc3+Xw=|c#a;WlBM$>-qfd*'
            'x$0p-AFpTdTIEv`!k4?!{c?A@U*}TyX)h<8tF1maojJqY*%vVr*tQz#d43KhG}Ou4Fk;z7SdR9o!69`4B6Z`LaUvUrl<rEN@<RJm'
            '=dNTGrTU&5Whu%HjA9pCVTf`d<(aPLvY4VY=twbfD9=QLk3iEfk|G<RNuY&5@s%M?8I4&ij=_Oh8I@;ou-9T}Y<a`%Z18CumrCeO'
            'hOq5xW+yzO$|A8tE=g7>U3`W@Eu19vw=5MZ%fPeL@gvs9#Uegogbr7DkItpF!<5rS;zf2UGgXcN`><>>U^Q9YRT5;Gdosw-G1jn`'
            'p`}BSsg~%<b0wrjA0&eQ;o$gJanbvaOU_mXf1t>4-UC75^g;<_RLVfeNRM1t>WaC~mA_Gr4XN%puAVptR^%f>&%#%57se~b1yBiA'
            'VT5u&M@lpeuP{zGPp(Rwxsfe-Q$M02MoMyK5iaxR@l9JV4~x!YZa&;QlDrFL3l(Q#PK7Lr`BxK%i>#96`H)9?NDxD@&H<nD*K*OM'
            'l1?fjAyl5e%Cn`jYu0}F<^Hq1m-{cY6G%?%siumwfoV^wd!QWwp0?Lm1!|d^1HIJVBz046>e~dXe7nB0V@V(%^ZO4U^q2~$#HX9Z'
            'TJ#bHPR)w+LX%Kqn(Abv`f#b5Rq%~%%5-0JPkOe)r-ZIzL;1I4@oi<+fe%~^&fZ_NWj<pvSO)Tt?CDb60GZJmgXLxad^|dutQ+ch'
            '7MXM|1C-xoEZ=R)6%kKRADjB4gz6C&Ds#bif+X{z*sIRxD9tE}<DhDyej-ZfjK@1UkJeXP)wf}6UNzi|)hBey86!LNPF`wGIj)tQ'
            '_3~GpN;fk*sg72YI_zhW%Om=iA(W^eO(lQuKQ<?;_Ajb@+N_X1I^{=&@juB;*ofwn@*(m7%j3W1j!dB-UL`REox%-TIkj2EwV3je'
            'FI{vPt<;rAX`tmCZ)r9TQ{OqiWUK0|-d1#1E7u!PaIkTBC`zYD4a#s(JM5wRDZFO8R>k|8G;0nY+6n+0f<yW36ItGjSW-UMqX2WJ'
            'Ki!T?xt5wCACAUrl=v`PU3<Us8j}gprmM>0)>yBlA=jLpcAmGU^tfYPv^|Y6<|Q?~F%8amOs>=_$n*d@>x%4s94ZU&t!ZwCqdmQ}'
            '!nJJS!cUqE=(vS=@%LIheaTvym$^~be5)Q!8impVOmUmyMMbi=lnSDjGT>m5_W2#Ul#`r{HxBZVe4+(tTbDG|NUIhZFE_vNJK?PK'
            '7&QPbe)s+un1fw7$k0fdd(ywS=%2pbW?#1{B>Gi1f^Oa7G$h9a%gYJ`-38gW8-&&F6B$r*ZzkPNd7*p>zmzaf%vHSXV3?H2?X~t&'
            'Df=W=u_aOE%o5Mabhny>I1~{uUwf)@d{Z`ncWUZ%K{cPJ9-Os`Jg9Y^K;<A%J7lOj4789B$&Lnf$v?Fs{9KM)6RVo*nBp7KhX9Zx'
            'eRsHHe_NiGYHo2|m)rJ<UV801uT8;Uh{;RA&|wAV=VEyEYrR9(V`Xe(MZXYPZ@rY^$42<<Cl{G9=1_{{W(zSUxKQ?m0s$=<8Y&{O'
            'S$*i#UN9GHbL&l|_cb{d$!vW=Bb{?YHebtL<q_FRVHO==tA=UJz(Ak;Elc`Ee~IF3i`m0UlwbWPcZzC#gwu724wt9A2vUk9`()y6'
            'pJ6*ta;M%p*|Z}7)zxMm1Zao<eZGvZ>}4I>ZdM-JIA;cC*dh0Pg<Y45^D`*+m!;g*6r3QL6K%QEMVg)eQ9bUXyK&V#Q%buc=en`E'
            'DVm@iD!BKQJE60&V?jv4A6tCL6iyoTV0(M{bMN%H|4x9=bJRc3Mzk!r31P$u5S1)!i?^%0Aex6W3j2W%;l-$*F8moso;$T4m@=65'
            'QdNjT9FdsOKkpx(^p5&yftXNairl2f>ajL&MYVpinZ;}x?{men%JeL`h*ovb^5iSs%fV00S{c;+pCh>WOi8u!|Hq@ms@V*KQ)Xfk'
            '-y|!_&2Z~umH|vVLdI10pp<$<_N+fejy^sq#=T%xQw(4bp!fMREELn;cmz!+$=hJQWL~=~85d}byY8dtuF=gk{GBIJ<Do9|a-Ej~'
            'bN=r?#23BupL^%(7!Ud<y|c>!mq96uvVW<$%d4D9xqGENiT_pI+?TpHS!VMJiQ*^$H_A)f*MQA0{_(C6{=XWQWhJOq=&<up{SWj$'
            '4SN04pS#EXBM&g|{pEn5SD`5VMLavG`BJHEEv)p74JeV@+>cPzT{^zyby%8Lvsw|;Y^0KO({;~IA$OR(FSOG>OToW3p+(907xWT$'
            '=aYs1O2J<iF#SBUy|scuwvX)ufA{~m(WRz(P1$UliwpA4tz`wLjI`LJ^su65>1gBCN9nk4vhHlD5U6I-lz&ZyEl_K&Y-JhtiGBI^'
            'l?^L0E85viEP<wYQ^D+&3=eg9fPC^<<!JV;1}#V$Aki_Y<ehK4D^nhQH{TIgzI0;<qfPjEb<lctt?c#w72?|U^-i5MP`qIwvE;Ri'
            'yr~Ife(<k$X)kxLA3-qk$^N#RBQD5IPCi$)gtDGZR~B(J4rU?a26}yJ%RZXxot>xTTUg%Cb~38t?0gOR>PyH~HS*LaQ=#9lx!B~V'
            '%xnx|WHY8hCgU0k^_djuQ>oEo`AXsUY;%pCRECaud0elCZKFm_oouzRuC-+0%G6InOeWEIFzC~62}5?TL3aV=CRhYk{Gl@rXIUsN'
            'e?7W9zdSza9`)Ofkp%1aAaenUzj!j9+_kGEy-%dsl;YROx9bBtLrLe7i{451bkIM%_$tYH0uEK<x_+FGi~Q9nrp(Buf%TMuq@{+h'
            '*1unzoyrtmdw!DmYx16orxeblg4?7USQ;;=P(~{56Q4as!i>pkoKi%$FNVV*&O%JFVyxR1=V+%+MWrvCs<0?atgt_exLKj^nBfde'
            'm%VZ|Iz^-WyPTv^8pf->*KkI0$`x6q$F>YlDvrjtbyV{fJ<;3jxZOw`SY6|X16-i{!*A)gjrMmxxbO@J8z+l`wio)6ng%E(n4+g`'
            '^dw(4p1IZ+RpBXgKyHM@wL(%{X>L<E)D7nW1HX*FuN;@m_94bgw4S+nmNWQ6JKylY*kDUQ5s)&U(PK1^Z&rAdEuj{$pznX+i6uOf'
            'M?Ew7;2Nj)ofJ&a(;d3~rBkewwQ(B8=V$)Xj~4zSOx51m*V=U8jWVt{lPmrfeE8bBv#!6Z*fg6dPsd*;WEYZR0SFy10xX_GM2+w+'
            'eUqYGJJSYBlg&Izoinm^8UVWSZ7`nUPuEDx#~lsoBD!w|7y?C65qpx-<gcYm=Y{26${uqxdAtY2G|i^7DZM@r!v_bWkjXd3aUO>('
            '`ld4ZoeU9y{;>A>9Fw%o@tlew&wI$#)>FKk#B&o*sbeDob+U<a*Mz*+Bm`Xc2#n`0d<E!GDbG?NVVi<mAvxgY-BM|cc;!)$!vBR$'
            'f%TEmnvD@EYC3Jt!-;y{qeLz%ZC+uwtk4r~Bm+)(w5s@$x;e^3u)?2XLabRDV&pZA`G@4u2rIe6_!O;<Tv3Z_a~|BlNUmWP{H2$i'
            'R*7~Ir%TTi`Efsc&OlMiGe+nYHXgyln6jZ9b3B73B3d*`2vcA0xAMMm^9uY3&Mw}`&d~Kao{QZFobKZ;S%L-dr0R2B$lX%PbpL4|'
            ')dCcJiau|L<V(#{M*0wIm*<>rd7maACJm}{UoTP63ZhjdU6e0{^%HkH%UWRtYWfxk)MUPb1lLWn5M7(dK8DMaNfgDi{jZcj@z#I-'
            '{VN8S&Wx$^4Lp^}5~g-L4@Yetr7^-V>D%DLB)kD-qTz1LbJ?Jnq0u@EQunhGUSj^dB*P*uDVfHqlDBP0sYx;cRZtHas($)T{n`jP'
            '*5WJxVrlUPRkpSO$c^D^h0*v+6h1;uHC*Z1Nk(w6fpFXcIh+S}VgyPufb=Q^wXDGjWDjIZstMnC_5JrRUcrgOZLmoIPeX<8(MWcb'
            '7r!sBPKJWZ`yPp$4dr%=YK%&+@@-RCp<qk$JIpxg`=+DE=1Lh}T~+Z$`kkvLPlkib<_zh|0a>lwzPgT(VsNgDRt~FEeR%0=gwfT&'
            'bnSTbjbNa#Y43Ig1kUr9sIOw45;OQrMrE40lI6!YMKk#&nu6%W0XeKbw^WpS9@@hpIG}o=6<<d<DJ6Q3Fkp}D1bc|uduT#z$y_$l'
            'MGaXxN5*q_52Z53-2AKTHcrBiEI8f*=AxzBH#}xw9+&6GO;+8|&neJcl?goyaTKJJr{!U<EEFB0o0KEg{W%_twYW#0QW?I*QBakh'
            'J4074oGJ-SR}uUFa}@v)VBx)8-Wg2p0v|)G!h{2<J^Iq_YYNnkd1+*UVo{amsshM{<R|DY!)1WEirQ+o<qzYexx+&8ALZsHgEKlR'
            'N&`i!l6Q#C+a%Ab={l_3|7On9Y>^zUHVlMDN@>1vji_#Grkod40#BB-wm`QrN2qP<s-#NB*6fw{MF8{2f<^H0HEacDPXKkPlynLZ'
            'xs~}N5Q#zoI@Ds~i$?9<xG0dJYjw$CTF=M|jxY?5Wl-vu49xm4iE>wdV}>1oV!4dNC?mt7qQl1f{9#>->OhW~n2hq2J~u3*ZY0%g'
            '+X#)iaXv9rqh_pf(~9vbB0DM$P_dD_A@YM1kn$LIGuPc18?vFhiX^CtyeORIO#v9@-xSPVLpADyGJp;E3=`)dFM^8|JgjoG)zv%-'
            'ktF4>o9~+#Y2t2*dMWoWCTZFxCZ;?}O;dM^$eE`H<Cf-FmWU+<Bit-Q_P3g_>CSvoj<l-`Gb(?>62_<woBg0TtWD|78QNcPjj=Y)'
            '(JjP06$CVWu*$Ejqm%@oQ-<iC1l#VxXTmXV?VIyn&l~i6=iX6w(8V%S{llKouR8h>O<CH_X~UkYvNV(?brReJ30-k;ahgH*r<4Ba'
            'QUAQDs?w~3YzCA(l^KpF$eL6{lTwYUrOe!kU9zq#$8N!yTx@o!nYp?q%O7!-9(fn9ANu7mG+list!Q6DCcE!rO|BBPhWczJs>N4O'
            ';NJ&HD`%-FI>r$iG(q@Y06-_HR_L3k$StH|4?TN^KyGklLFI!m<-pl5r)0jG$YM*x<J%ys25XFU#>0{XD67L@<Ss*`B5BI7^n~ni'
            '%GZVhH?Y<>W=~1j*@nE6xK0hq?us?Qa1`IaSk413jQ3S65p>Eu6ggzlA<lS2nq_UWu&wfrZL~_&CLGJt>U64=w{O2?0DH50Ke<Vz'
            '_$mv0c514R@S<F!GPM1^yizwWzlMhKx=a#Qt{yx#I0yckX&~G_(N(7+(N$sisq;`C@A0?jx9EYt{8-GYgyAOD>E&v|sd<IPZ7%$6'
            'Z%qZVlpj@Xd+CWiZjKN7SF<uyk~1RFHnnTg$no7}Ctsn0YDfKON{Ds%ABhFZ^oTd%PZ(cq(`CV>rm-pM7K&~5Sc2?N)uS=nmIaNh'
            'z@3AtARTEkw~mfgf~?w+4ev6IctBNstS%!FCnK`6<EY=w(#ciz%T|#Z={C+5@q#@kP^XsXRwX?AR(9Mbhs-pLfc(RNEgn(uh@!!z'
            'bGpq!+O-afA*+QlMVtrYAiN7wA!7@H5G3%o^lPP?G0cKBiSJXfC(<z>8W0z(+7iQGAXTR7J{viM3LHZPCKin}fC(HQpZ(&UcYpCP'
            'tfK1;&O9yE6ms}g>{bC$FqN6Z>xAEOuPeGSGGXhTJn5zEhl%G=8(PC_iB`)4>)K5zzWf;=;i#$c9TK{HEU)ytYZslyq$KBBxMT1e'
            'r?}mY!F|51MG}U-qGXuW;`d^S+|PVOjU{>F-YqvlUYdUD9>_H<!U(W2y2);>O+e^k(p0$v>Km9IRr?IMSUdLR(=abVZ#$rmR`Z!B'
            'gY{(ozTW}%+wkVLbr;W9i@?m?f@z2p=zlC9&@ciDHCGHjK@Wl?z8Chapj(*DgFVioP1Z{?m1h2pvB6E_9CMDRdEiH@B~VeRV`^Vx'
            '+1{8Y-5>*hxI?eqIQm2sso$Ck7;m>ZcVXfK<p&OyER!Jny}JVs4$u1odT0-du0Z4#j(?`)G9c*VW^NL-fi6{eRZcx5^`@oXm{pIA'
            'vAL<W1edDiZ3h^Z7^uLo^xCnZ!s1Atxme{r`~DLlR+Si17P6)%5+sL;yIi6dl+0K9y#2kAN|N*d031pq6Pc>X7Z)eME=06ii~yFn'
            '2k}y7CQ~!lIqb14uD@m}CRw1GK?!Mn>=TYD-4<7ir^I41ZPUG!xkjsuoVeI8#5d}6(L{cYSA*3kz%!DBV-io~$_IEFQ_A&&=^#Uu'
            'nCOBFHeQ||r+ed&w=N~UM&2U_d|@%eOME#Utw=t_1U1wE8S<M*ogv3Wp%Nlg;*aTQs0a7Bt|Yrls@w0<+lFQKG=0I{x^e%xB0$xQ'
            'E8lKCu8qj-)owkqS}QQX`@29^fY14n^HyQ7pI}7XMfa$8*b}=WEWsPC04?xB>DC{SWn=;f5$@ux@Cq#=IvZ+Yxp!K)Gb4k_U9k2G'
            '2=oy?7ZC<w$Z90k@C%Eu^$t&Ze=j(M+D@IO=VQh0m@Wx;qUXAts&<1+4O}H2k}B`VVGhNs>vlf6;uR`8^&58o@zE)A8e<+bLg*WP'
            '+vkU3_*Qj4QN6_;UykWFhj>%`OE4M=bW2J_W0-%FlitHL?t@W#m4the-Ng$(jD}KjR9Wb+uJbId+JSRS*Nf=)W6&M7l#=h23AtK_'
            '1X+hbLK~Q65Ja+)6c(j&H|LPIoKZIBk=M*wr^YuE$d@<9DHU~7Zw@wB{;_WU<)?IQUh_-59o3}}V26x}VngT(4-175U=#CZ@EU6p'
            'Bw@K)0!|;>!k6$LOlb^YK6tvu52sZNqP9PoG@z?JvfQ){(L*?!1N>%HG!9}PytSs!4=X%>97fY(d->F@jM|YXCZ}0@(lx!@c+#<='
            '%n&{*voEV&L!&YSC;vq5n5HXqGpyoG65Y{3S&>Q*+`<LbkQ`C6!m&&hi75anBj>#jG=@GK<V9=c1o+nL3Z#<jEy{Xqe_(J3{KR^5'
            'ZI7yx!?;=iR5hXla`H!{=c8IwY5(O5g=^Hw+GW{5S%}48aG`{!BliF$q2Bv0Mw?1iTU$q^l#Ua?eIjQg@|jD7G<Lr2o%B!p9*pSy'
            'eAYkgU1*l){5@FSt`jQzuo}<s-@{u!AtBRwhc`s<@0%b>LTYmyOnm(J4EP=X+l>QVe$hDu)CrU1Hl|M9RTfA5vj-~l+@EllzP}7I'
            '{&~3=!HVb9dVUiIBVP@51X~R1DRl-e2#T_76hz^O3Q_o}4}XE0MAP!2#Uu!+Ll=-NiTp&3)tyt|627`X{PSrgZ~bwy`oQa+V~+$B'
            'EZIFw$X(#?z_X(OzquNPYP6p*?e!w`fs<433qYCi?T9~(f6PJK3g{CTtGPe*vBtH#qY^UfEx`B_g0J=JxO@8cvip|aW=U&s?(BSX'
            'Adcekip}6ka7Nr<yWxR&{!+|-`q&iDUbRLcS!sd0p~dC>;fZL#?=hEw3KHOu_>qh5$$Kn$iKx_p?%w|XzWL&p{?Xtkd_xZpf9jr|'
            '_Kp!{o)sp8vmoQ)2W1fA4;-GI4tjrs6`j57opNk6^nUT2J|s==ii!~~d*qX67nsRbfZkX1qG`B3$>aB5sq9Y}p%z!F`@@gD0xvfG'
            '6RE+LOw4Be+!fhJd33BgP|-3kMo^keStuyGH8S_N-5M+AhdS-6m+jH3my>|}9PxQoIwB%@yzSJaN%P)$G7tiBj$$$Z-+TpnCWv<='
            'I?M13MziI3;bpOR@83xjg9isfA(f95)I%eAr*JONl8p`^^R#20ff{c*InaGL$!<HvgTx;G_(jc$e@Xmhs*fn3pfAq*Z{F<v(*3!Y'
            'pH?YY8z)|d2T2Duj5@#pxTv!jwqcb+OnM`Wx{Y81P#Ry({WaeEvZIhpFiqhL!4kCX{Em&CL)qAagd+oS0aT=+iyr{h7)?~DY)j`l'
            'S`(<v(<xO&z|7Mzm5!k~*upfnB|uXKD-Cw>t%fuFG{Kk`gYx`&c!Lu4rj`s5#6Jfi>4@*-I@(JRLmJLs`aqChyrQDt5h4Kv<cu7I'
            '_;is8?2uRLEOh<pr#}3@gA}%jm#AtY76vu*HCAJZv6!Z{09enTT<iNIKGv0s(DSz(hk5zmA-)%5ZkWJoXJNMdgb|;I&a?gJFVV?!'
            '$zDMljzonqNMY(8p7ilv8h!+2Y{gc_y(o@aXj6v<fTv{nl@=ZLlmmPN=u0GjPZ2gGzTp0;1%+7}%%@F>e^C%jsJ6{?$_W6`dcsI4'
            'MO3=5knwDkem%?zh)5aIJMg{imVKvXQ8~0rQ{|Z80|=w&8h~%xVakw04xsM{a_0kmR*}NgWgT(|Iv$du^w}%Y7*u})+@2Bq3op=5'
            'v_6rMqoC|@p!%I2^ikg2!nEeRAY2EzNhQ2sH9GA-WrH+^V;`o<D{XVKM%Dv)7V;+`gQVUPQ;bl1`Qp)Z%x7?4DOg{ZbZVS+np%Dy'
            '^6KC~cE(BbBGF}we^uyQc|}n~uSuM)F}+bdU!@etE}_M&i#8lxHzliZO;BE$KvRhba^9v^q;bS*BK|5~zO>q@{Vg=><R>#Pdt|(Q'
            '`I2xR|GqkSd5st5is0$ze>!;fT)si-W?ZGKkK0&#EnU5HXBH+H<mgsQ>^%BE+-ojshg516NjjwoCQa!1{=th^*SQQp?o0@Pik^Md'
            '>E-^x%a_;g)&8|nm4<dg`Pe)Z5Yn2SiI;i>^(m{$IEzn=fb|@7_E9F=p$RYq{4?4Nh7J7nsys~L+|wu~TkLcyNwQ_pG22tx9K{~h'
            '#|!Zo@`%Hy@(tJ%Tr#MZB{ygldwam4Y#~k`3yXB*QR(V&DXW44L79SKXFW@mlY{91#`U8g3iWMMz+PIoOfpCY|M(o5;S?$m%dc~}'
            'Sf(H{Q(79DQEPP$?cWL1137I6bH7<r+~pB~ZS=>2l_<h^=c?+$>2i))bNRv&I(k`_Hsqf=m|b4sfWEG8Z(f3mRW|)x;Ap8hJ3aoD'
            'jiTyqt}VLi2AHB3rq~Q?{ZR;O7?EQbTkqJVOP0!8EGS4J4OFca!^exXLfEci-B)w^s=%cg^eu@UhlKs5W2~)Ya>X{+I#i6dN7{|p'
            'CfQUMJ<<^C<dr;A6m>B!m%>@w1m{NLG=BI?=c@gOAFkcs((hyxyL(_)LysDAR9DK8YAg!w29(t^<^#!%3)WVvzN>7a@3HNsdI{<~'
            'JEjB7K}};!p~rtbV(DV;`REiuUHG)gC7zt2@;9QE<(tQ9RE<}erv=cvI2`MOS|!^^C8eJG(aj1yp~prot}5F}W$y@6IuN~?go&!s'
            'E$ewX#yKJ`KnpWZROOIGm!7RIOLJvToAy|D!h;vzN0qic6(Y1sW?_1Zb>dVB&c@IJOJ|piM!Q2-`Emh0#R@wFZ(XZX14C$FV15dz'
            'j&f3qrK$w!%cyw}CRn+o40k|8Ebp+1XlStlzYT0sNeqHnGBTW|$oYb$AE?XW?)&%WXFqq34^*&N(cFd88y13&hN$ZLD<A?RtPBN>'
            'cXUY`zHohi-ut=V`vn$Bckv-ty@LkfbS>+VD!@l-MbJ}vB6H~1P>1CZ?bd$ELpTmC_K;HPL}1P6cHuU;LT~QYN)W@X*kgM;TFlu='
            'AdGHdh|DhiL`|`n%WggyyysSQSV#SE_#4vfDON>mVyPnO(kzCyTlJ6?QLzLWSlxi{Xt9sxZEH;G+$ya}{q0(BK}97xv6ul$HTaKI'
            'rH?}~-;>$otO(r=ETKXtkRd`&mZdV9t0JvZUb~}QQ=vjZC`OY|Vica$YstxO*s7$8RaA{qQ+1^_ereFXc;}s;T@HE|SG1h#stcoH'
            'z!HsU7gwpMV<Jn5=sM|7NfMRBk9SE{$p(C!h98iFS5Q&4UnuLK8?M8`%!Y<6O*z}RE*sO#3n(!ylh-Y!n~e<-eNzYh+oMIW1_{WZ'
            'EFjonk^>s|56L44y|a()^2n|u=P-`v97H8=u1H^#cUPb6I4y^2qZ5`{>f5@7<;^VBEZ~dj&Qp%R<!thGp1}5oe<wx1Dg!mT30Eg;'
            'Hp<_|6QHFTCkDH6x|c`&Gv~Tm4RZi60s3Tc72gA~ArrL4^6*HRe{az8XoHO}z_AU`Gd%N3Z}<UvGr*tp$Kzv7*5~r$$4yBrjr$Hz'
            'p_Q$E<84UQ-GR~YV5g=eLcIZ<h#ashX)&L~5%A^7$|+U?QOGHULF1!U<y{yZ#W%1aX;$>PNM#~Zb;-K(8JRhh`AE%6<-i&_QpeJ7'
            'J|T^ZBT0Y)2EfKCt*+t=J731BTsmx3x|K^#_i29Rg+eGg-`yWL^txP!h<BCDgxAFiBX*lSyHZulQno~W>w?C$;zdj`!)QUAxCdcK'
            'n%f$>lWU5cY|E%JbK4S1%`(v*x~XAdt@)o;TOtkI0Dl$I^_gUfEPdt^t8E3pq#hN|5$QGLRe$M<qs)o?r@bt8B2j{X3~(bz7!i*a'
            '!!0L*l~68s-T?Qdl-Bc%v)bCIAueT8kFz<HJo0i@ohMzCBK>Q@xf9XPlO5;^%0XLP00J7$co?Y2S<AvC4fa450Ld+)V10fhWb#uA'
            '+K;ms4skdWH^CTPo<G7q0HrVoTnCe2h7?2!^+tEBAo25g_xPl1JMbxUk4CC1WxcxITqmqpIG{djr8%~GT<bi@@lil+@-z_mhn;Sv'
            'tkQq0ADZb+Dr$z*GUbXGSPp$FYtuS%U28?HF-R-0L{xCKD#TYz=B4RM>CfHyJsP&k>SIIVD5DqM@edi&!@XfijTknyBJKZw-Cb>O'
            '8#faE?q5NuA1oD;6Zeo@YeNN`T2726PGKj9GYmtbv|D+hH&?55qqzR>H*e&UORg;YQXFvk&?YOA%N@QAXP$Y6TciuT#>m`@caLAa'
            '#9zDVQpn0J-w1ViAG9}6|AK~GVdZdbVBBz6BH}DNC1H_mjyZO^_9!wzqTNQIXGRpi?$s`-eY@8!XA*dE-d*=DvJ++0=~llkwi`J&'
            'DE|0vl$->l#o=SM5_73Vs%^>&>sHu7oarX}l4LzR#C-vFWY{%8(c?WPLG-UWJ-awMga7{0d*A!eyEs*EdcVFuz5IAyO^?y|XvssK'
            'M-bwEb^N0vOboGw8F|*;2{}ltgT{$TFVEhCpbyTEZu~o|;8_x>!Kbdz!N4ag6b=T~KZj|2V`jsa*n%6K2D+Uhjhp+%BQJ1j#*X!{'
            'eyo&7iV-`x!)BtzoOZrLgc#oQ73jh95gF~6dkakfs}ZGRhDP^zZwU8WJI&lSVJ}1YI{aF|+|GB^Zdo3UNquTM&81(vYK%@M_S!=)'
            'Za{VptNeHV{%xDutCI%4A~xu8VTg_C>R|u%wb7&M1#K)z-oHKW4v<cg-r1bUeM=i!&6!(qus56WMc^o%)Ytom7XNJgR+Xo2tL}^$'
            'E%H^r$dkUggaj#WZ3%|_hWT;-C~ym3_bH&Wzu$iq776-Xx3=BSa{ItaYgo$w%RRCjSBtFXX$UmPNKYyye6kWcYV*E1X-g6cZzvx&'
            'QaY&KY?vkqQGtnZA3-zEVka;gRPVzZ%yoA5wtsy3p?7w1`tb}7?|;7jF;apuqbd%ehsD!LO?Ns!G<FcV89qmIF9K=?c*YYraB#>N'
            'V)(ZjFH(!FVapIR819tsBts?~c_*j;CQvc&`3*yv+0V0CHq}e!eQF3>pW*T?e^h@dRx1t4b~MGD@JOpx)+?U{#hFS1M>i34s{1+$'
            'LzA2KU&@q-)h1AOeO3CI>OF*BFDq82kIXGUp_tQ%i{5I;F$rzr>VuO%o#*7WVz<<W8M2vD5SKHa<fMdE$O50`(<(n)-DR3)3w5eP'
            'FyOOK%8-LGrqHO39Yd=l+Br<>@+))C$))X}QD7mupE*JheM+r8H3(<+KztgjMaL%2$-RTmyouCtSOYK~u;t-l<_5nKne#0u25EB0'
            '7HcOG!unlkS8A7y^1v&xqoNb&7OpEKadq+Tr~`+Q1S{!3SsS*br!+tbBRI)bKd1~Dc~L4HeFP&w12oON>j)!hE~Lk#1(FYfp89;L'
            '*vbaKOF7VgpZjIJm=&ORz=awa_U8oVZ9ksY{xQNh&*9>H$91b6^|OQxZ{c^5)G+KTso5jxFskkf6wPj`qwfp>FIUzIkESXdn-&w{'
            'BZUuhvd;1+I~<}^AtaEvyI_N>fP32BiFpRh4b83yE!=;w4K(e;)`QDt@(hEpv-NaqB7~g54g%uJ-=1JD_JpNu9)*|@{Vv6f@p&25'
            'WE?xpriAIvYkI&Cr}Cqy0(=cWkB1ZRZ%_0F*bYEvC(zlVV*hr)FWi0$4axo}KJAq?>5D8!{cy2{?eN|<{~$#5>fY%@B;TXoXA!;e'
            'TwF2E?T(iaA+Wfm>sA%xBQ@ze32ZGB{Ru9Gok(^juc-$$&X!5W32E=$w?4=?@N{a_IFbx2I9GbECt@=TYhKKnaj)!(r6>tc<de4Y'
            'XLu*bliV7vvC~csGIaWL(JNTpXQ_PkVO<T*5k`OYim)i~-YH1nE05H$Zl4aDAeugFvZrEBo)TY83k<&WnBPw|YfZaw(ZeXDQq64Y'
            'wi=RRM{DB1)c~~hsNr9*TiULJ;U==&M(`S?*KmO^zG8xe$x72kxtPv0Y?d;`F|Qb{`lakQ$%4RcFz>#o@70>5_@%9Aqx_oPa>uzS'
            '>S%rKl>Wk{DVU}O(+5uES7phelw8|(snMZVl4QWmcGXLk1lR4lN#I1taN>J(=Fs;SsNhlq-%C{!hBRN?rBid6E<?=vynx%*!rQl-'
            'k3T%1=k2Qw0yShj-Mf<>YgF}YBcM3m7DG^bZqnpMNwX~{mT24idoL-c<ch35(i+FEGczbLl_KtT&frd!6rcs8^1P>xQ&^g)aU+ol'
            'H8;-gQYUMs*?nWlOsd}+NOoT&J7gMr%&rqo;!uVQ<oe*HM}Kv*+*!GL%Y=Z+*gTp>DA&+F{sl$Tp|`g{JA7@%5L5ZsTo8RLjeNs!'
            'B8?B?u@+Of_|b5}_q^Qu>4y-Zdu-Z6V6r&+^k})ra|9u;#0F)^8DtO#=0#!78Kg%W;+=&1g!!i=c#NXG*(Rqnp|120rYARO`#V5='
            '3*Rd<gK}i~<7pc4Vz#}vx<gP7Gc5u{t!y!<(t8j0PSt<zI<oaOR_>OK>2FY%2qr>ZYj`#Zo&?k;j&(Ke9&@QRrA3T}Uw4I>YIdMH'
            'vn#i)Yco;APSGE(-EX6}PtlypQ?X`l{P-%I8*i6snylM2GwiYxswZZ)aZDpkMs0Vd8G^R$J31JcnUs`XQo{t_;~oF{k&&!^K`D0P'
            'H%g2Z!G|#MdkDm2URhNx!--&qU#eU+MyLQUvV_>xYKPwWu&c%lm5rs7m67@%hN4O&3c1aLx~=;5%PyhZqPkRbgdr64f{xlph#V-O'
            'FZhil90>-%!!6QoIQ`9^Fm<RM0xy)uPc~e+FL0F#j9kX3Ts=k<a4jYQA_mcRQEu1LO@HxRxpha5Odt5$-6?S`OULf2eR#Wu6SFBN'
            'rz{c1D#~x}viZlW$dUOQmoxSo#b9P6)ar0nCE8_qKcPfTsa|$$7wmy)mW*^ojwJug;Om8yXtg5eOp}ilQv@c4^C@!7uo?LX4w&&$'
            '-{ILVcI1=R0R;ptGs1+!vix{*;+n|nId0KHsmm5E2~^*0FQ5j4ZXr|j_PIoYKa}r$0d32zf$f%Cr#|^6XnXe)J+WGHGPu3K9xun~'
            'vG8O20Q>XLXMN0A`S$$iU#Fig`{%uj-qHCl?~GIi%=I>dnM}-KAaM9pPO?WBc9U~M?nKN4i=`R)ExMZbjNM{B0A1WqZZS#^qEQX-'
            'Q%?-6g|cmO)nU(8OmnYvzEegyyQw?LyFUd5?+BGXTnEtppgNKRH7t^Giq=YM2W5Tq>ErpOgy{-?A<w4Fv<B-aN3ugyJQBU$%>@A`'
            'Qa@eg4z^zq6$5wZ2C#7kppB?*X{K#p!>lB(6@zAMUDTU4jq52wCtg}T-F`QsrAlLGvr#v<7z3l4cY2xYHJpm`)&V7sUFXQsv(yM+'
            'nrZb|A{9*eD_|#tuD8`qHX3G22DO)v_Pr|o(0>PLkBYU5aDOJcz0yc^Zel*=%A*K-JCR{kiYZcZR?X^j3PpRlb)J~5wp+IQ=utLN'
            'I2u7$hQ(@>=0UO;NjI1jOQg}wL5%=ae^{)G39l%%Dkhk^0!KF2lU9%=M%HaN{-KuJej|(c&s}Nef4Im5o%7i8`n_SimimU^1S}<7'
            'Q6Svv_jm94yB;22RWd%%qRzARVtM#~y`Weu?lSroK8MXw3Xeo6)HodiHg(<(n)_`t^*ftqcf=v_41o9MS8QP?b#M*w#Y3i3Hzley'
            'p>9oz9@tnT3CDxUaIMVhujEWOnuHO*XcAPRycEV@>D@Y5x(j>tPSq{Iua!;txOdjOyy(4G+SLIysD8y^h!_5hD7_h~P%zSnEjFo5'
            '+SLu25W=t>7c=JmWMJ_u{b~=|JYB=Cu+A{!ss0*l$n{uUci_t;hz=gNq5y9X?et-twG(Uu&eJ=}^ca2kjR3i6(48S{6X~=JLbFC)'
            '^1+sm0v3Jz;Z=b9^g!oju^=6sKU>R!(RJV@M5`94_GsG`^7m(pIa#J@2h$2M?TO@192a_-m^?8*efzwx#O#e|hYyG!{d|j=$=@It'
            'I!&7A#T>blNt=>W1HtgJ91s@2T=h%rsYY&Ds7C)UpYv4}jbxVl+=6W;dOg<@&Op$hg;O%Q!VTg){k<Uep#?_TCq&FOTHLOe+4u(U'
            'OVmF8O3}R_KSlBbT@0Cx@F7%RN?LWkH6azk?TSLE<(l&zsfpCd$?P^ABkzVr!$;@On+y)iB|P|TeSwJ-j>-B9|DI;U<syfjJVky`'
            '2`pabyz%O}p00C3i@^FjWVA?qmWDQwQ$yL=E#g4%Jue8Ew`1+GLRnxL^(@=$c!6Kec+BXXt2z!9O*ua1JHB^xQ$h&73mvF?XJj8#'
            'F@mK68nBw^)eOW@-I^~y|N6dnhQ!Il?G;U$g(0izvY23`{Y2s34Z2*8B&K@3)Z;+^MumN(L2pO=_BdUplWC@I^kfPbGSDh-vRk#7'
            'CdC*JgH^hQt8lvVwtTV3*r^QX?mU~1n|=ac*#jIc;Pd6t`RT#Nv>Aj7+EefJjHmevhh^Q$$_ZLwv1M}{U_|d`1iE=lSZ@lrx<Fx&'
            '4Oc_KIjz#YW}{>h^d~{D;NLbP=0BPWT9N'
        ),
    },
}

def _load_local_pipeline_modules():
    # 1) Zaten import edilebiliyorsa (yerel .venv, repo Colab'da klonlanmis) dogrudan kullan.
    try:
        import local_pipeline
        modules = {}
        for _name in _LOCAL_PIPELINE_MODULE_NAMES:
            modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
        print('local_pipeline modulleri dogrudan import edildi:', list(modules))
        return modules
    except Exception:
        pass

    # 2) Drive'da veya proje kokunun bir ustunde daha once birakilmis bir kopya var mi?
    search_roots = []
    try:
        search_roots.append(Path(DRIVE_PROJECT_ROOT))
    except NameError:
        pass
    search_roots += [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    for root in search_roots:
        candidate_dir = Path(root) / 'local_pipeline'
        if all((candidate_dir / f'{_n}.py').is_file() for _n in _LOCAL_PIPELINE_MODULE_NAMES):
            sys.path.insert(0, str(candidate_dir.parent))
            try:
                modules = {}
                for _name in _LOCAL_PIPELINE_MODULE_NAMES:
                    modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
                print('local_pipeline modulleri onceden birakilmis kopyadan yuklendi:', candidate_dir)
                return modules
            except Exception:
                pass

    # 3) PROJECT_REPO_URL verilmisse git clone dene.
    if PROJECT_REPO_URL:
        clone_dir = Path('/content/youtube_automation_repo') if os.path.isdir('/content') else Path.cwd() / '_youtube_automation_repo'
        if not (clone_dir / 'local_pipeline' / (_LOCAL_PIPELINE_MODULE_NAMES[0] + '.py')).is_file():
            subprocess.run(['git', 'clone', '--depth', '1', PROJECT_REPO_URL, str(clone_dir)], check=False)
        if all((clone_dir / 'local_pipeline' / f'{_n}.py').is_file() for _n in _LOCAL_PIPELINE_MODULE_NAMES):
            sys.path.insert(0, str(clone_dir))
            try:
                modules = {}
                for _name in _LOCAL_PIPELINE_MODULE_NAMES:
                    modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
                print('local_pipeline modulleri PROJECT_REPO_URL klonundan yuklendi:', clone_dir)
                return modules
            except Exception:
                pass

    # 4) Self-contained fallback: bu hucreye gomulu, build sirasinda
    # local_pipeline/<modul>.py'den birebir uretilmis base64 kopyalari, SHA-256
    # hash'lerini dogrulayarak diske yazip import et.
    fallback_dir = Path(tempfile.mkdtemp(prefix='youtube_automation_bootstrap_'))
    pkg_dir = fallback_dir / 'local_pipeline'
    pkg_dir.mkdir(parents=True, exist_ok=True)
    (pkg_dir / '__init__.py').write_text('', encoding='utf-8')
    for _name, _entry in _LOCAL_PIPELINE_PAYLOADS.items():
        if _entry.get('encoding') != 'zlib+b85':
            raise RuntimeError(f"Desteklenmeyen gomulu payload formati: {_entry.get('encoding')!r}")
        _source_bytes = zlib.decompress(base64.b85decode(_entry['data']))
        _actual_digest = hashlib.sha256(_source_bytes).hexdigest()
        if _actual_digest != _entry['sha256']:
            raise RuntimeError(
                f"Gomulu local_pipeline.{_name} payloadi bozuk/tutarsiz: "
                f"beklenen sha256={_entry['sha256']!r}, hesaplanan={_actual_digest!r}. "
                "Bu notebook yeniden uretilmeli (nb_builder ile)."
            )
        (pkg_dir / f'{_name}.py').write_text(_source_bytes.decode('utf-8'), encoding='utf-8')
    sys.path.insert(0, str(fallback_dir))
    modules = {}
    for _name in _LOCAL_PIPELINE_MODULE_NAMES:
        modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
    print('local_pipeline modulleri: repo/Drive/PROJECT_REPO_URL bulunamadi; gomulu (self-contained, hash dogrulanmis) kopyalar kullanildi.')
    return modules

_local_pipeline_modules = _load_local_pipeline_modules()
video_analysis = _local_pipeline_modules['video_analysis']
render_utils = _local_pipeline_modules['render_utils']
google_ai = _local_pipeline_modules['google_ai']



## 1. Test Videosu

İki seçenek: (A) kendi dosyanı yükle, (B) bir YouTube URL'sinden kısa bir klip indir. Gerçekçi bir karşılaştırma için **3–6 dakikalık** bir örnek yeterli (tüm videoyu değil).

**YouTube "Sign in to confirm you're not a bot" hatası alırsan** (Colab IP'leri sık engellenir), iki yoldan biri:
- **En garantili — SECENEK A (upload):** Videoyu kendi bilgisayarına indir (ör. tarayıcı uzantısıyla ya da yerelde `yt-dlp`), Colab sol paneldeki 📁 **Files**'tan yükle, `SOURCE_MODE = 'upload'` yap ve `UPLOADED_PATH`'i dosya adına eşitle.
- **YouTube'dan indirmeye devam etmek istersen — cookies:** Chrome/Firefox'a "Get cookies.txt LOCALLY" gibi bir uzantı kur, YouTube'da oturum açıkken `cookies.txt` dışa aktar, Colab Files'a yükle ve aşağıda `COOKIES_FILE = 'cookies.txt'` yaz. Bu, yt-dlp'ye "gerçek bir tarayıcı oturumu" gösterip bot kontrolünü çoğu zaman aşar.

In [ ]:
# SECENEK A: Kendi dosyani kullan. Colab ve yerel Jupyter otomatik algilanir.
# SECENEK B: YouTube URL'sinden indir.

SOURCE_MODE = 'upload'  # 'upload' veya 'youtube'
YOUTUBE_URL = ''
VIDEO_FILENAME = 'THE ABSOLUTE BEST OF DAEMON TARGARYEN - ABC CINEMA (1080p).mp4'
COOKIES_FILE = ''
CLIP_START_SEC = 0
CLIP_DURATION_SEC = 240

if IS_COLAB:
    SEARCH_ROOTS = [Path('/content/sample_data'), Path('/content')]
else:
    SEARCH_ROOTS = [
        PROJECT_ROOT / 'sample_data',
        PROJECT_ROOT,
        Path.home() / 'Downloads',
        Path.home() / 'Videos',
    ]

UPLOADED_PATH = str(SEARCH_ROOTS[0] / VIDEO_FILENAME)
RAW_VIDEO = str(RUNTIME_DIR / 'source_raw.mp4')
VIDEO_PATH = str(RUNTIME_DIR / 'source_clip.mp4')
AUDIO_PATH = str(RUNTIME_DIR / 'source_clip.wav')

def resolve_uploaded_path(path):
    """Tam yol yoksa Colab veya yerel arama klasorlerinde kaynak videoyu bulur."""
    requested = Path(path)
    if requested.is_file():
        return str(requested.resolve())

    print(f'UYARI: tam yol bulunamadi -> {requested}')
    generated_names = {'source_clip.mp4', 'input_video.mp4'}
    exact_matches = []
    fallback_matches = []
    for root in SEARCH_ROOTS:
        if not root.exists():
            continue
        exact = root / VIDEO_FILENAME
        if exact.is_file():
            exact_matches.append(exact)
        for extension in ('*.mp4', '*.mkv', '*.webm', '*.mov'):
            fallback_matches.extend(
                item for item in root.rglob(extension)
                if item.name not in generated_names
            )

    candidates = exact_matches or sorted(set(fallback_matches))
    print('Bulunan kaynak videolar:', [str(item) for item in candidates])
    if len(candidates) == 1:
        selected = str(candidates[0].resolve())
        print('Otomatik kullanilan video:', selected)
        return selected
    if len(candidates) > 1:
        raise FileNotFoundError(
            f'Birden fazla video bulundu. VIDEO_FILENAME veya UPLOADED_PATH degerini netlestir: {candidates}'
        )
    raise FileNotFoundError(
        f'Video bulunamadi. Aranan klasorler: {[str(root) for root in SEARCH_ROOTS]}'
    )

def download_youtube(url, out_path, cookies_file=''):
    base_cmd = [
        'yt-dlp',
        '-f', 'bv*[ext=mp4]+ba[ext=m4a]/b[ext=mp4]/b',
        '--merge-output-format', 'mp4',
        '--no-playlist',
        '-o', out_path,
    ]
    attempts = [['--cookies', cookies_file]] if cookies_file else []
    attempts += [[], ['--extractor-args', 'youtube:player_client=android']]
    last_stderr = ''
    for extra in attempts:
        proc = subprocess.run(base_cmd + extra + [url], capture_output=True, text=True)
        if proc.returncode == 0:
            print('Indirme basarili.')
            return
        last_stderr = proc.stderr
    print(last_stderr[-3000:])
    raise RuntimeError('yt-dlp indirme basarisiz oldu; upload modunu veya cookies dosyasini kullan.')

if SOURCE_MODE == 'youtube':
    assert YOUTUBE_URL, 'YOUTUBE_URL bos olamaz'
    download_youtube(YOUTUBE_URL, RAW_VIDEO, cookies_file=COOKIES_FILE)
else:
    RAW_VIDEO = resolve_uploaded_path(UPLOADED_PATH)

file_size_mb = os.path.getsize(RAW_VIDEO) / (1024 ** 2)
print(f'Kaynak video: {RAW_VIDEO}')
print(f'Dosya boyutu: {file_size_mb:.2f} MB')
if file_size_mb < 1:
    raise RuntimeError('Video dosyasi cok kucuk; yukleme veya kopyalama tamamlanmamis olabilir.')

if FFPROBE_BIN:
    probe = subprocess.run(
        [
            FFPROBE_BIN, '-v', 'error',
            '-show_entries', 'stream=index,codec_type,codec_name,width,height,duration',
            '-of', 'json', RAW_VIDEO,
        ],
        capture_output=True,
        text=True,
    )
    if probe.returncode != 0:
        raise RuntimeError(f'FFprobe kaynak videoyu okuyamadi: {probe.stderr[-3000:]}')
    streams = json.loads(probe.stdout).get('streams', [])
    print('Bulunan akislar:', streams)
    if not any(stream.get('codec_type') == 'video' for stream in streams):
        raise RuntimeError('Dosyada video akisi bulunamadi.')
    if not any(stream.get('codec_type') == 'audio' for stream in streams):
        raise RuntimeError('Dosyada ses akisi bulunamadi; transcript uretilemez.')
else:
    print('FFprobe yok; akis kontrolu FFmpeg donusumu sirasinda yapilacak.')

# Kanonik klip+ses cikarimi (master notebook ve verify_local.py ile ayni fonksiyon).
video_analysis.extract_clip_and_audio(
    FFMPEG_BIN, RAW_VIDEO, VIDEO_PATH, AUDIO_PATH, CLIP_START_SEC, CLIP_DURATION_SEC,
)

print('Klip hazir:', VIDEO_PATH)
print('Ses hazir:', AUDIO_PATH)

## 2. Transcript Çıkarımı — Aday Modeller

Adaylar: `faster-whisper` (tiny / small / large-v3-turbo, lokal & ücretsiz) ve `OpenAI gpt-4o-mini-transcribe` (API, ücretli — anahtar varsa).

In [ ]:
import gc

transcripts = {}
transcript_segments = {}

try:
    cuda_check = subprocess.run(['nvidia-smi'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    cuda_available = cuda_check.returncode == 0
except FileNotFoundError:
    cuda_available = False
local_gpu_opt_in = os.environ.get('YOUTUBE_AUTOMATION_USE_GPU') == '1'
cuda_available = cuda_available and (IS_COLAB or local_gpu_opt_in)
TRANSCRIBE_DEVICE = 'cuda' if cuda_available else 'cpu'
TRANSCRIBE_COMPUTE_TYPE = 'float16' if TRANSCRIBE_DEVICE == 'cuda' else 'int8'
# Karar (gercek test sonuclarina gore): large-v3-turbo GPU'da varsayilan
# (~%91 ortalama kelime guveni), small CPU fallback (~%85). tiny (~%72) sadece
# karsilastirma icin tutuluyor, production icin onerilmez.
TRANSCRIPT_MODELS = ['tiny', 'small']
if TRANSCRIBE_DEVICE == 'cuda':
    TRANSCRIPT_MODELS.append('large-v3-turbo')
else:
    print('GPU bulunamadi; large-v3-turbo cok yavas olacagi icin atlandi. CPU fallback: small.')

print('Transkripsiyon cihazi:', TRANSCRIBE_DEVICE, '| modeller:', TRANSCRIPT_MODELS)

# Bu notebook'un katkisi: AYNI kanonik transcribe_audio() fonksiyonunu birden
# fazla model boyutuyla calistirip karsilastirmak.
for size in TRANSCRIPT_MODELS:
    candidate_name = f'faster-whisper-{size}'
    t0 = time.time()
    text, segment_rows = video_analysis.transcribe_audio(
        AUDIO_PATH, model_size=size, device=TRANSCRIBE_DEVICE, compute_type=TRANSCRIBE_COMPUTE_TYPE,
    )
    latency = time.time() - t0
    transcripts[candidate_name] = text
    transcript_segments[candidate_name] = segment_rows
    log_result(
        'transcript', candidate_name, quality_score=None, cost_usd=0.0, latency_sec=latency,
        notes=f'{len(text)} karakter, {len(segment_rows)} segment, VAD+timestamps',
    )
    gc.collect()

if OPENAI_API_KEY:
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)
    t0 = time.time()
    # gpt-4o-mini-transcribe: whisper-1'in yerini alan guncel, daha ucuz model.
    with open(AUDIO_PATH, 'rb') as f:
        resp = client.audio.transcriptions.create(model='gpt-4o-mini-transcribe', file=f)
    latency = time.time() - t0
    transcripts['openai-gpt-4o-mini-transcribe'] = resp.text
    cost = (CLIP_DURATION_SEC / 60) * 0.003  # yaklasik; platform.openai.com/docs/pricing
    log_result('transcript', 'openai-gpt-4o-mini-transcribe', quality_score=None, cost_usd=cost,
               latency_sec=latency, notes=f'{len(resp.text)} karakter')
else:
    print('OpenAI key yok, openai-gpt-4o-mini-transcribe atlandi.')


In [ ]:
# Ciktilari yan yana oku ve kalite puanini (1-5) kendin gir.
for name, text in transcripts.items():
    print('=' * 20, name, '=' * 20)
    print(text[:800], '...\n')

transcript_quality = {
    # ornek: 'faster-whisper-tiny': 2, 'faster-whisper-small': 3, 'faster-whisper-large-v3-turbo': 4
}

for r in results:
    if r['task'] == 'transcript' and r['candidate'] in transcript_quality:
        r['quality_score_1_5'] = transcript_quality[r['candidate']]
print('Kalite puanlarini transcript_quality sozlugune girip bu hucreyi tekrar calistir.')

## 3. Sahne Tespiti — Aday Yöntemler

Adaylar: PySceneDetect `ContentDetector` ve `AdaptiveDetector`. Kalite ölçütü: bulunan sahne sayısının videoyu mantıklı şekilde bölüp bölmediği (elle kontrol).

In [ ]:
scene_results = {}

# Kanonik detect_scenes() iki dedektorle karsilastirmali calistirilir.
for name, detector_key in [('content-detector', 'content'), ('adaptive-detector', 'adaptive')]:
    t0 = time.time()
    scene_list = video_analysis.detect_scenes(VIDEO_PATH, detector=detector_key)
    latency = time.time() - t0
    scene_results[name] = scene_list
    log_result('scene-detection', name, quality_score=None, cost_usd=0.0, latency_sec=latency,
               notes=f'{len(scene_list)} sahne bulundu')


In [ ]:
scene_quality = {
    # ornek: 'content-detector': 4, 'adaptive-detector': 3
}
for r in results:
    if r['task'] == 'scene-detection' and r['candidate'] in scene_quality:
        r['quality_score_1_5'] = scene_quality[r['candidate']]
print('Sahne sayilarini yukarida gordukten sonra scene_quality sozlugunu doldurup bu hucreyi tekrar calistir.')

## 4. Ses Enerjisi Analizi (Destek Sinyali)

Bu adım tek bir sabit yöntemle (librosa RMS enerji tepe noktaları) çalışır — aday model karşılaştırması gerektirmez, sadece hot-moment skorlamasına ek sinyal sağlar. Bu yüzden burada kalite/maliyet karşılaştırması yapılmaz.

In [ ]:
import numpy as np

t0 = time.time()
# Kanonik compute_audio_peaks(): 16kHz yukleme + RMS + min-mesafeli tepe gruplama.
audio_info = video_analysis.compute_audio_peaks(AUDIO_PATH, top_n=10, min_distance_sec=1.5)
times, rms = audio_info['times'], audio_info['rms']
audio_peak_details = audio_info['peaks']
audio_peaks = [item['time_sec'] for item in audio_peak_details]
latency = time.time() - t0
log_result('audio-energy', 'librosa-rms', quality_score='n/a', cost_usd=0.0, latency_sec=latency,
           notes=f'gruplanmis en yuksek enerjili anlar: {[round(t, 1) for t in audio_peaks]}')


## 5. Yerel + Opsiyonel LLM Hot-Moment Skorlama

Önce transcript, sahne ve ses sinyallerinden API gerektirmeyen yerel hot-moment adayları üretilir. Aynı adaylar, API anahtarı varsa LLM sağlayıcılarıyla karşılaştırılır.

In [ ]:
t0 = time.time()
TRANSCRIPT_PREFERENCE = [
    'faster-whisper-large-v3-turbo',
    'faster-whisper-small',
    'faster-whisper-tiny',
]
BEST_TRANSCRIPT_CANDIDATE = next(
    name for name in TRANSCRIPT_PREFERENCE
    if name in transcript_segments and transcripts.get(name, '').strip()
)
print('Kullanilan transcript:', BEST_TRANSCRIPT_CANDIDATE)

LOCAL_WINDOW_SEC = 30.0
LOCAL_STRIDE_SEC = 10.0
TOP_HOT_MOMENTS = 5

analysis_duration = min(float(CLIP_DURATION_SEC), float(times[-1]))
content_scenes = scene_results.get('content-detector', [])
scene_boundaries = [end for _, end in content_scenes[:-1]] if content_scenes else []

# Kanonik skorlama: master notebook da AYNI score_hot_moments() fonksiyonunu cagirir.
hot_moments = video_analysis.score_hot_moments(
    transcript_segments[BEST_TRANSCRIPT_CANDIDATE], times, rms, scene_boundaries,
    analysis_duration, CLIP_START_SEC, top_n=TOP_HOT_MOMENTS,
    window_sec=LOCAL_WINDOW_SEC, stride_sec=LOCAL_STRIDE_SEC,
)

# Kanonik onizleme klibi uretimi (her ana bir 720p klip).
video_analysis.generate_preview_clips(FFMPEG_BIN, VIDEO_PATH, hot_moments, RESULTS_DIR / 'hot_moment_previews')

# Kanonik manifest: status HER ZAMAN 'review_required' (asla otomatik approved).
hot_moments_manifest = video_analysis.build_hot_moments_manifest(
    hot_moments, transcript_candidate=BEST_TRANSCRIPT_CANDIDATE,
    window_sec=LOCAL_WINDOW_SEC, stride_sec=LOCAL_STRIDE_SEC,
    weights=video_analysis.DEFAULT_WEIGHTS,
    raw_video_path=RAW_VIDEO, analysis_clip_path=VIDEO_PATH,
    clip_start_sec=CLIP_START_SEC, clip_duration_sec=CLIP_DURATION_SEC,
)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
hot_moments_path = RESULTS_DIR / 'hot_moments.json'
with open(hot_moments_path, 'w', encoding='utf-8') as hot_file:
    json.dump(hot_moments_manifest, hot_file, ensure_ascii=False, indent=2)

local_latency = time.time() - t0
log_result(
    'hotmoment-scoring', 'local-multisignal-v2', quality_score=None, cost_usd=0.0,
    latency_sec=local_latency,
    notes=str([(item['start_sec'], item['end_sec'], item['score']) for item in hot_moments]),
    status='review_required',
)
print('Yerel hot-moment dosyasi:', hot_moments_path, '(status: review_required)')
display(pd.DataFrame(hot_moments)[['rank', 'aligned_start_sec', 'aligned_end_sec', 'score', 'reason', 'preview_path', 'transcript']])

# Opsiyonel LLM karsilastirmasinda yerel yontemin sectigi ayni pencereleri kullan.
windows = [item['transcript'] for item in hot_moments if item['transcript']]
print(f'{len(windows)} zaman kodlu pencere olusturuldu.')

SCORING_PROMPT = (
    "Asagidaki video transkript parcasini oku. 0-100 arasi bir 'hot moment' (vurucu, "
    "paylasilabilir, dikkat cekici an) skoru ver ve 1 cumlelik gerekce yaz. "
    "SADECE JSON don: {{\"score\": <int>, \"reason\": \"...\"}}\n\nMetin:\n{chunk}"
)


In [ ]:
def parse_json_score(raw):
    try:
        start = raw.find('{'); end = raw.rfind('}') + 1
        return json.loads(raw[start:end])
    except Exception:
        return {'score': None, 'reason': raw[:200]}

llm_outputs = {'local-multisignal-v2': hot_moments}

# --- OpenAI gpt-5.6-luna (gpt-4o-mini'nin yerini alan dusuk maliyetli model) ---
if OPENAI_API_KEY:
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)
    t0 = time.time()
    scored = []
    for chunk in windows:
        resp = client.chat.completions.create(
            model='gpt-5.6-luna',
            messages=[{'role': 'user', 'content': SCORING_PROMPT.format(chunk=chunk)}],
            temperature=0,
        )
        scored.append(parse_json_score(resp.choices[0].message.content))
    latency = time.time() - t0
    llm_outputs['openai-gpt-5.6-luna'] = scored
    approx_tokens = sum(len(c) for c in windows) / 4
    cost = (approx_tokens / 1_000_000) * 1.0  # yaklasik input fiyati, guncelini platform.openai.com/docs/pricing adresinden kontrol et
    log_result('hotmoment-scoring', 'openai-gpt-5.6-luna', quality_score=None, cost_usd=cost, latency_sec=latency,
               notes=str([s.get('score') for s in scored]))
else:
    print('OpenAI key yok, gpt-5.6-luna atlandi.')

# --- Anthropic claude-haiku-4-5 ---
if ANTHROPIC_API_KEY:
    import anthropic
    client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
    t0 = time.time()
    scored = []
    for chunk in windows:
        resp = client.messages.create(
            model='claude-haiku-4-5',
            max_tokens=200,
            messages=[{'role': 'user', 'content': SCORING_PROMPT.format(chunk=chunk)}],
        )
        scored.append(parse_json_score(resp.content[0].text))
    latency = time.time() - t0
    llm_outputs['anthropic-claude-haiku-4-5'] = scored
    approx_tokens = sum(len(c) for c in windows) / 4
    cost = (approx_tokens / 1_000_000) * 1.00  # yaklasik input fiyati, guncelini platform.claude.com/docs/en/pricing adresinden kontrol et
    log_result('hotmoment-scoring', 'anthropic-claude-haiku-4-5', quality_score=None, cost_usd=cost, latency_sec=latency,
               notes=str([s.get('score') for s in scored]))
else:
    print('Anthropic key yok, claude-haiku-4-5 atlandi.')

# --- Google: SABIT MODEL YOK. Her cagri kanonik router uzerinden gider -------
# (local_pipeline/google_ai.py: Rota A = gemini-3.6-flash -> gemini-3.5-flash ->
#  gemini-3.5-flash-lite -> gemma-4-31b-it, ilk basarida durur, kota defteri
#  tutulur, hatalar siniflandirilir ve sanitize edilir.)
#
# Ucretsiz katman veri politikasi: buraya SADECE yerel olarak uretilmis transcript
# pencereleri gonderilir; ham video ASLA yuklenmez.
GOOGLE_CREDENTIAL = google_ai.resolve_secret('google_api', get_secret)
print('Gemini kimlik bilgisi:', google_ai.describe_credential_state(GOOGLE_CREDENTIAL))

if GOOGLE_CREDENTIAL.configured:
    _g_client = google_ai.build_google_client(GOOGLE_CREDENTIAL)
    _g_manifest = google_ai.build_google_capability_manifest(
        _g_client, credential=GOOGLE_CREDENTIAL)
    _g_ledger = google_ai.QuotaLedger()
    _g_selection = google_ai.select_google_route(
        'hotmoment_review', manifest=_g_manifest, ledger=_g_ledger)
    print(google_ai.format_route_line(_g_selection, GOOGLE_CREDENTIAL))

    t0 = time.time()
    scored, _g_models, _g_failed = [], [], None
    for chunk in windows:
        _g_result = google_ai.run_google_hotmoment_review(
            SCORING_PROMPT.format(chunk=chunk), client=_g_client,
            manifest=_g_manifest, ledger=_g_ledger,
            required_fields=(), checks=(),
        )
        if _g_result['status'] != 'ok':
            _g_failed = _g_result['status_reason']
            break
        _g_models.append(_g_result['model'])
        scored.append(parse_json_score(_g_result['output']))
    latency = time.time() - t0

    if _g_failed:
        # Cikti UYDURULMAZ: basarisizlik acikca 'failed' olarak loglanir.
        log_result('hotmoment-scoring', 'google-router', quality_score=None, cost_usd=0.0,
                   latency_sec=latency, notes=f'Google rotasi basarisiz: {_g_failed}',
                   status='failed')
        print('Google hot-moment incelemesi basarisiz:', _g_failed)
    else:
        _g_label = 'google-' + (sorted(set(_g_models))[0] if _g_models else 'router')
        llm_outputs[_g_label] = scored
        # Ucretsiz katman: dolar maliyeti 0; gercek butce RPM/TPM/RPD kotasidir.
        log_result('hotmoment-scoring', _g_label, quality_score=None, cost_usd=0.0,
                   latency_sec=latency,
                   notes=str([s.get('score') for s in scored])
                         + f' | kullanilan modeller: {sorted(set(_g_models))}')
else:
    print('Google key yok (GeminiKey / GOOGLE_API_KEY), Google adayi atlandi.')
    print(google_ai.credential_help_text('google_api'))

In [ ]:
for name, scored in llm_outputs.items():
    print('=' * 20, name, '=' * 20)
    for s in scored:
        print(s)
    print()

hotmoment_quality = {
    # ornek: 'local-multisignal-v2': 4, 'openai-gpt-5.6-luna': 4
}
for r in results:
    if r['task'] == 'hotmoment-scoring' and r['candidate'] in hotmoment_quality:
        r['quality_score_1_5'] = hotmoment_quality[r['candidate']]
print('Skorlarin tutarliligini/isabetini degerlendirip hotmoment_quality sozlugunu doldur, sonra tekrar calistir.')

## 6. Karşılaştırma Tablosu ve Dışa Aktarım

In [ ]:
df = pd.DataFrame(results)
# Not: satir bazli status artik log_result tarafindan ayarlaniyor; eksikleri guvenli varsayilana tamamla.
if 'status' not in df.columns:
    df['status'] = 'experimental'
else:
    df['status'] = df['status'].fillna('experimental')
display(df)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
out_path = RESULTS_DIR / '01_video_analysis_hotmoment_sonuclar.csv'
df.to_csv(out_path, index=False)
print('Kaydedildi:', out_path)
print('\nBu dosyayi indirip experiments/results/ altina koy; roadmap Faz 1deki provider registry karari bu sonuclara gore verilecek.')

## 7. Tanılama Paketini İndir

Bu hücre transcriptleri, sahne zamanlarını, ses tepe noktalarını ve karşılaştırma CSV'sini tek bir ZIP dosyasında toplar.

In [ ]:
import zipfile

scene_report = {}
for detector_name, scenes in globals().get('scene_results', {}).items():
    scene_report[detector_name] = {
        'count': len(scenes),
        # detect_scenes() zaten (start_sec, end_sec) float ikilileri dondurur.
        'scenes': [{'start_sec': round(start, 3), 'end_sec': round(end, 3)} for start, end in scenes],
    }

diagnostic_report = {
    'status': 'experimental',
    'status_reason': 'provider comparison notebook output; not a final production artifact',
    'source': {
        'raw_video': globals().get('RAW_VIDEO'),
        'video_path': globals().get('VIDEO_PATH'),
        'audio_path': globals().get('AUDIO_PATH'),
        'clip_start_sec': globals().get('CLIP_START_SEC'),
        'clip_duration_sec': globals().get('CLIP_DURATION_SEC'),
    },
    'transcripts': {
        name: {
            'character_count': len(text),
            'full_text': text,
            'segments': globals().get('transcript_segments', {}).get(name, []),
        }
        for name, text in globals().get('transcripts', {}).items()
    },
    'scene_detection': scene_report,
    'audio_peaks_sec': globals().get('audio_peaks', []),
    'audio_peak_details': globals().get('audio_peak_details', []),
    'local_hot_moments': globals().get('hot_moments', []),
    'llm_hotmoment_outputs': globals().get('llm_outputs', {}),
    'benchmark_results': globals().get('results', []),
}

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
report_path = RESULTS_DIR / '01_video_analysis_tanilama.json'
zip_path = RESULTS_DIR / '01_video_analysis_paylasim_paketi.zip'

with open(report_path, 'w', encoding='utf-8') as report_file:
    json.dump(diagnostic_report, report_file, ensure_ascii=False, indent=2)

with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    archive.write(report_path, arcname=report_path.name)
    if Path(out_path).exists():
        archive.write(out_path, arcname=Path(out_path).name)
    if Path(globals().get('hot_moments_path', '')).exists():
        archive.write(hot_moments_path, arcname=Path(hot_moments_path).name)
    for preview_file in (RESULTS_DIR / 'hot_moment_previews').glob('*.mp4'):
        archive.write(preview_file, arcname=f'hot_moment_previews/{preview_file.name}')

print('Paylasim paketi hazir:', zip_path)

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception as exc:
    print('Otomatik indirme baslatilamadi. Files panelinden elle indir:', zip_path)
    print('Detay:', exc)


## 8. Karar (elle doldurulacak)

Yukarıdaki tabloyu inceledikten sonra bu hücreyi düzenleyip kararını ve kısa gerekçeni yaz. Bu karar Faz 1'de `packages/providers/vision-analysis` ve `packages/providers/llm` adaptörlerinin varsayılan sağlayıcısını belirleyecek.

- **Transcript:** _(seçilen aday)_ — gerekçe: _...
- **Sahne tespiti:** _(seçilen aday)_ — gerekçe: _...
- **Hot-moment skorlama:** _(seçilen aday)_ — gerekçe: _...
